# DROID terminal-state context diagnostic

Use the same AutoDL GPU as the masking run if available, with at least 22 GiB
VRAM. Upload this notebook beside the existing **DROID_natural_history_frames.zip**
and use **Restart Kernel → Run All**.

The three exact archived terminal images are used in each condition. First
score them alone, then add the four shared recent images, then restore the
three earlier images from the original ten-frame annotated input. Use the
same pinned model, direct scorer and both option orders. Check all 28 archived
ten-frame scores and report pair gains and losses at each context step.

The context lengths and positions differ by design. The result is a capability
diagnostic on seven development pairs, with any confirmation on fresh footage.

Frames are resized from [DROID](https://droid-dataset.github.io/), Khazatsky et al.
and the DROID Dataset Team, under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

After **DROID_TERMINAL_CONTEXT_COMPLETE**, download
**HERMES_droid_terminal_context_evidence.zip** and save the executed notebook.
If replay fails after launch, run the final download cell manually.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_droid_terminal_context')
REUSE = WORK.with_name('hermes_twinpast_refinement')
FALLBACK = WORK.with_name('hermes_causal_pilot')
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-7b-ov-hf'
MODEL_SHA = '0d50680527681998e456c7b78950205bedd8a068'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_DROID_TERMINAL_CONTEXT_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
REPO = WORK/'pinned_source'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'provenance/droid_terminal_runner.py'
print('Plan: 7 DROID pairs, 3 context lengths')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')

FRAME_PACKAGE = Path.cwd() / 'DROID_natural_history_frames.zip'
if not VALIDATE_ONLY:
    if not FRAME_PACKAGE.is_file():
        raise FileNotFoundError(f'Upload the companion frame ZIP alongside this notebook: {FRAME_PACKAGE}')
    with FRAME_PACKAGE.open('rb') as stream:
        if hashlib.file_digest(stream, 'sha256').hexdigest() != '5bcfb613c5eb2cd8a40e306d94a79a3e77258ba162b47ddbb025ffc2a56f8b49':
            raise ValueError(f'Frame ZIP has the wrong SHA-256: {FRAME_PACKAGE}')
    print('DROID_FRAME_PACKAGE_VERIFIED', FRAME_PACKAGE, flush=True)


## 1. Verify runtime and materialize the experiment


In [ ]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    model_folder = 'models--' + MODEL_ID.replace('/', '--')
    candidates = [root/'cache/huggingface' for root in (REUSE, FALLBACK, WORK)]
    snapshots = [root/'hub'/model_folder/'snapshots'/MODEL_SHA for root in candidates]
    cache = next((root for root, snap in zip(candidates, snapshots, strict=True)
                  if sum(p.is_file() for p in snap.glob('model-*-of-00004.safetensors')) == 4), None)
    if cache is None:
        cache = next((root for root, snap in zip(candidates, snapshots, strict=True) if snap.is_dir()),
                     candidates[-1])
    env.update(HF_HOME=str(cache), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    minimum_free = 1 if any((root/'venv/bin/python').is_file() for root in (REUSE, FALLBACK)) else 18
    assert free >= minimum_free, f'Need {minimum_free} GiB free on the data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 22528, 'Use a GPU with at least 22 GiB VRAM'
    # Never print the environment or proxy values.


In [ ]:
SOURCES = {'pinned_source/inference/llavaov_hermes.py': 'import re\nimport time\nimport torch\nimport torch.nn.functional as F\nimport types\nfrom transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor, DynamicCache\nfrom logzero import logger\n\nfrom inference.abstract_hermes import Abstract_Hermes\nfrom inference.reindex_1d import (\n    _get_rotary_module,\n    get_cache_seq_len,\n    contiguous_kv,\n    compute_cos_sin_for_positions,\n    rotary_delta,\n    apply_rotary_delta_to_keys_only,\n    apply_rotary_pos_emb_1d,\n)\n\n\nclass LlavaOneVision_Hermes(LlavaOnevisionForConditionalGeneration, Abstract_Hermes):\n    def __init__(self, config, processor, init_prompt_ids, kv_size, streaming=True):\n        super().__init__(processor, init_prompt_ids, kv_size)\n        self.streaming = streaming\n        \n        num_layers = config.num_hidden_layers if hasattr(config, \'num_hidden_layers\') else 32\n        self.num_layers = num_layers\n        \n        self.short_term_ratio = 0.3\n        self.long_term_ratio = 0.3\n        self.short_term_threshold = int(self.num_layers * self.short_term_ratio)\n        self.long_term_threshold = int(self.num_layers * (1 - self.long_term_ratio))\n        \n        self._position_ids_cache = [None for _ in range(num_layers)]\n        self.token_activity_cache = [None for _ in range(num_layers)]\n        \n        self._layer_position_ids = {}\n        self._hook_handles = []\n        self._register_forward_hooks()\n        \n        self.total_processed_frames = 0\n        \n        self._patch_rotary_embeddings()\n\n    def _patch_rotary_embeddings(self):\n        """Patch rotary embedding to support dynamic resizing based on logical position IDs"""\n        def patched_forward(rotary_self, x, seq_len=None):\n            required_len = getattr(rotary_self, \'_required_seq_len\', 0)\n            if seq_len is not None and required_len > seq_len:\n                seq_len = required_len\n            return rotary_self._original_forward(x, seq_len=seq_len)\n\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    rotary_emb = layer.self_attn.rotary_emb\n                    if not hasattr(rotary_emb, \'_original_forward\'):\n                        rotary_emb._original_forward = rotary_emb.forward\n                        rotary_emb.forward = types.MethodType(patched_forward, rotary_emb)\n                        rotary_emb._required_seq_len = 0\n\n    def _set_rotary_required_len(self, required_len):\n        """Set required sequence length for rotary embeddings"""\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    layer.self_attn.rotary_emb._required_seq_len = required_len\n\n    def _register_forward_hooks(self):\n        if not hasattr(self.language_model, \'model\') or not hasattr(self.language_model.model, \'layers\'):\n            return\n        \n        def create_hook(layer_idx):\n            def hook(module, args, kwargs):\n                if layer_idx in self._layer_position_ids:\n                    kwargs[\'position_ids\'] = self._layer_position_ids[layer_idx]\n                return args, kwargs\n            return hook\n        \n        for layer_idx, layer in enumerate(self.language_model.model.layers):\n            handle = layer.register_forward_pre_hook(create_hook(layer_idx), with_kwargs=True)\n            self._hook_handles.append(handle)\n    \n    def _clear_forward_hooks(self):\n        for handle in self._hook_handles:\n            handle.remove()\n        self._hook_handles = []\n\n    def _prune_activity_cache(self, keep_indices_per_layer):\n        if all(c is None for c in self.token_activity_cache):\n            return\n\n        for layer_idx in range(self.num_layers):\n            if self.token_activity_cache[layer_idx] is not None:\n                keep_indices = keep_indices_per_layer[layer_idx]\n                if not isinstance(keep_indices, torch.Tensor):\n                    keep_indices = torch.tensor(keep_indices, device=self.device, dtype=torch.long)\n                \n                curr_len = self.token_activity_cache[layer_idx].shape[0]\n                keep_indices = keep_indices[(keep_indices >= 0) & (keep_indices < curr_len)]\n                \n                if keep_indices.numel() > 0:\n                    self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][keep_indices]\n                else:\n                    self.token_activity_cache[layer_idx] = torch.zeros(0, device=self.device)\n\n    def _append_position_ids_layer(self, layer_idx: int, start: int, length: int):\n        device = self.device\n        new_pos = torch.arange(start, start + length, device=device, dtype=torch.long)\n        if self._position_ids_cache[layer_idx] is None:\n            self._position_ids_cache[layer_idx] = new_pos\n        else:\n            self._position_ids_cache[layer_idx] = torch.cat(\n                [self._position_ids_cache[layer_idx], new_pos], dim=0\n            )\n    \n    def _append_position_ids(self, start_per_layer, length: int):\n        if isinstance(start_per_layer, int):\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer, length)\n        else:\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer[layer_idx], length)\n\n    def _get_cache_seq_len_per_layer(self) -> list:\n        if self.kv_cache is None:\n            return [0] * self.num_layers\n        \n        lengths = []\n        for layer_idx in range(len(self.kv_cache)):\n            k_layer, v_layer = self.kv_cache[layer_idx]\n            lengths.append(k_layer.shape[2])\n        return lengths\n\n    def _get_next_start_pos_per_layer(self) -> list:\n        next_pos = []\n        for layer_idx in range(self.num_layers):\n            if (layer_idx < len(self._position_ids_cache) and \n                self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > 0):\n                next_pos.append(self._position_ids_cache[layer_idx][-1].item() + 1)\n            else:\n                next_pos.append(0)\n        return next_pos\n    \n    def _sanitize_keep_indices(self, keep_indices_1d: torch.Tensor, seq_len: int) -> torch.Tensor:\n        keep_indices_1d = keep_indices_1d.to(self.device, dtype=torch.long)\n        keep_indices_1d = keep_indices_1d[(keep_indices_1d >= 0) & (keep_indices_1d < seq_len)]\n        if keep_indices_1d.numel() == 0:\n            return torch.tensor([0], device=self.device, dtype=torch.long)\n        keep_indices_1d = torch.unique(keep_indices_1d, sorted=True)\n        return keep_indices_1d\n\n    def _build_position_ids(self, past_len: int, q_len: int, batch: int) -> torch.Tensor:\n        pos_1d = torch.arange(past_len, past_len + q_len, device=self.device, dtype=torch.long)\n        position_ids = pos_1d.view(1, -1).expand(batch, -1)\n        return position_ids\n\n    @torch.inference_mode()\n    def _shrink_positions_and_rerotate_keys(self, keep_indices_per_layer):\n        device = self.device\n        \n        curr_lens = self._get_cache_seq_len_per_layer()\n        \n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is None or \n                self._position_ids_cache[layer_idx].numel() != curr_lens[layer_idx]):\n                self._position_ids_cache[layer_idx] = torch.arange(\n                    curr_lens[layer_idx], device=device, dtype=torch.long\n                )\n        max_pos_limit = getattr(self.language_model.config, "max_position_embeddings", 32768)\n        compact_threshold = max_pos_limit - 1024\n        \n        current_max_pos = 0\n        for cache in self._position_ids_cache:\n             if cache is not None and cache.numel() > 0:\n                 current_max_pos = max(current_max_pos, cache[-1].item())\n        \n        should_compact = (current_max_pos > compact_threshold) if self.streaming else True\n        \n        if should_compact:\n            logger.info(f"[Shrink] Max position {current_max_pos} > {compact_threshold}. Compacting position IDs (Shift Left).")\n        \n        old_position_ids_cache = [cache.clone() for cache in self._position_ids_cache]\n\n        sample_k = self.kv_cache[0][0]\n        dtype = sample_k.dtype\n        \n        new_kv_cache = []\n        \n        for layer_idx, (k_layer, v_layer) in enumerate(self.kv_cache):\n            \n            keep_indices_layer = keep_indices_per_layer[layer_idx]\n            if not isinstance(keep_indices_layer, torch.Tensor):\n                keep_indices_layer = torch.as_tensor(keep_indices_layer, dtype=torch.long, device=device)\n            \n            seq_len_layer = k_layer.shape[2]\n            safe_idx = self._sanitize_keep_indices(keep_indices_layer, seq_len_layer)\n            \n            if safe_idx.numel() == 0:\n                logger.warning(f"Layer {layer_idx}: After sanitization, keep_indices is empty; keeping first token")\n                safe_idx = torch.tensor([0], device=device, dtype=torch.long)\n\n            is_long_term = (layer_idx >= self.long_term_threshold)\n            \n            # === Extract Kept Tokens ===\n            k_kept = torch.index_select(k_layer, dim=2, index=safe_idx)\n            v_kept = torch.index_select(v_layer, dim=2, index=safe_idx)\n            old_pos_kept = old_position_ids_cache[layer_idx].index_select(0, safe_idx)\n            \n            # === Determine New Positions for Kept Tokens ===\n            if should_compact:\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)\n            else:\n                new_pos_kept = old_pos_kept\n\n            # === Handle RoPE for Kept Tokens ===\n            if should_compact:\n                cos_old, sin_old = compute_cos_sin_for_positions(\n                    self.language_model, len(old_pos_kept), old_pos_kept, dtype, device\n                )\n                cos_new, sin_new = compute_cos_sin_for_positions(\n                    self.language_model, len(new_pos_kept), new_pos_kept, dtype, device\n                )\n                cos_delta, sin_delta = rotary_delta(\n                    cos_old, sin_old, cos_new, sin_new\n                )\n                k_kept_final = apply_rotary_delta_to_keys_only(k_kept, cos_delta, sin_delta)\n            else:\n                k_kept_final = k_kept\n            \n            # === Handle Folding (Summary Token) for Long-term Layers ===\n            if is_long_term:\n                mask = torch.ones(seq_len_layer, dtype=torch.bool, device=device)\n                mask[safe_idx] = False\n                prune_indices = torch.nonzero(mask).squeeze(1)\n                \n                if prune_indices.numel() > 0:\n                    k_pruned = torch.index_select(k_layer, dim=2, index=prune_indices)\n                    v_pruned = torch.index_select(v_layer, dim=2, index=prune_indices)\n                    \n                    v_summary = v_pruned.mean(dim=2, keepdim=True)\n                    \n                    old_pos_pruned = old_position_ids_cache[layer_idx].index_select(0, prune_indices)\n                    \n                    if should_compact:\n                        summary_pos_id = len(safe_idx)\n                    else:\n                        summary_pos_id = new_pos_kept[-1].item() + 1\n                    \n                    summary_pos_tensor = torch.tensor([summary_pos_id], device=device, dtype=torch.long)\n                    target_pos_pruned = summary_pos_tensor.repeat(old_pos_pruned.shape[0])\n                    \n                    cos_old, sin_old = compute_cos_sin_for_positions(\n                        self.language_model, len(old_pos_pruned), old_pos_pruned, dtype, device\n                    )\n                    cos_new, sin_new = compute_cos_sin_for_positions(\n                        self.language_model, len(target_pos_pruned), target_pos_pruned, dtype, device\n                    )\n                    \n                    cos_delta, sin_delta = rotary_delta(cos_old, sin_old, cos_new, sin_new)\n                    \n                    k_pruned_aligned = apply_rotary_delta_to_keys_only(k_pruned, cos_delta, sin_delta)\n                    \n                    k_summary_final = k_pruned_aligned.mean(dim=2, keepdim=True)\n                    \n                    k_final = torch.cat([k_kept_final, k_summary_final], dim=2)\n                    v_final = torch.cat([v_kept, v_summary], dim=2)\n                    \n                    new_pos_layer = torch.cat([new_pos_kept, summary_pos_tensor])\n                    \n                else:\n                    k_final = k_kept_final\n                    v_final = v_kept\n                    new_pos_layer = new_pos_kept\n            else:\n                k_final = k_kept_final\n                v_final = v_kept\n                new_pos_layer = new_pos_kept\n\n            new_kv_cache.append((k_final.contiguous(), v_final.contiguous()))\n            \n            self._position_ids_cache[layer_idx] = new_pos_layer.clone()\n        \n        self.kv_cache = new_kv_cache\n        \n        new_lens = self._get_cache_seq_len_per_layer()\n        logger.info(f"Layer-wise shrink completed. Lengths: min={min(new_lens)}, max={max(new_lens)}, "\n                   f"first={new_lens[0]}, last={new_lens[-1]}")\n    \n    @torch.inference_mode()\n    def predict_next_question(self):\n        """\n        Generates queries to drive attention calculation.\n        Includes Semantic Anchors for better coverage.\n        """\n        if hasattr(self, \'conv_history\') and len(self.conv_history) > 0:\n            last_q, last_a, last_options = self.conv_history[-1]\n            \n            option_match = re.match(r\'^\\s*(?:\\()?([A-Z])(?:\\))?\\.?\\s*$\', last_a)\n            if option_match and last_options is not None:\n                option_char = option_match.group(1)\n                option_idx = ord(option_char) - ord(\'A\')\n                if 0 <= option_idx < len(last_options):\n                    last_a = last_options[option_idx]\n\n            last_a = re.sub(r\'[A-Z]\\) \', \'\', last_a)\n\n            last_round_history = f"Question: {last_q} Answer: {last_a}"\n            \n            global_query = f"Context summary: {last_round_history}. Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            local_query = f"Find recent details related to: {last_round_history}. Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            \n        else:\n            global_query = (\n                "Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            )\n            local_query = (\n                "Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            )\n            \n        print(f"Local question: {local_query}")\n        print(f"Global question: {global_query}")\n        return local_query, global_query\n    \n    @torch.inference_mode()\n    def encode_init_prompt(self):\n        super().encode_init_prompt()\n        self.total_processed_frames = 0\n        curr_lens = self._get_cache_seq_len_per_layer()\n        for layer_idx in range(self.num_layers):\n            self._position_ids_cache[layer_idx] = torch.arange(\n                curr_lens[layer_idx], device=self.device, dtype=torch.long\n            )\n    \n    def get_video_features(self, pixel_values_videos):\n            batch_size, frames, channels, height, width = pixel_values_videos.shape\n            pixel_values_videos = pixel_values_videos.view(batch_size * frames, channels, height, width)\n            video_features = self.vision_tower(pixel_values_videos, output_hidden_states=True)\n            selected_video_feature = video_features.hidden_states[self.config.vision_feature_layer]\n            if self.config.vision_feature_select_strategy == "default":\n                selected_video_feature = selected_video_feature[:, 1:]\n            elif self.config.vision_feature_select_strategy == "full":\n                selected_video_feature = selected_video_feature\n            video_features = self.multi_modal_projector(selected_video_feature)\n            video_features = self.apply_pooling(video_features)\n            frames_after_merge = video_features.shape[0]\n            video_features = video_features.reshape(batch_size, frames_after_merge * video_features.shape[1], -1)  # (B, Nv*196, D)\n            return video_features\n    \n    @torch.inference_mode()\n    def encode_video_chunk(self, video_chunk):\n        if video_chunk is None or (hasattr(video_chunk, "shape") and video_chunk.shape[0] == 0):\n            return\n\n        if len(video_chunk.shape) == 4 and video_chunk.shape[-1] == 3:\n            video_chunk = video_chunk.permute(0, 3, 1, 2)\n\n        pixel_values_videos = self.processor.video_processor(\n            video_chunk, return_tensors="pt"\n        ).pixel_values_videos.to(self.device, self.dtype)\n        video_features = self.get_video_features(pixel_values_videos)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = video_features.shape[1]\n        batch = video_features.shape[0]\n        \n        max_start_pos = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start_pos + q_len)\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        default_position_ids = self._build_position_ids(start_pos_per_layer[0], q_len, batch)\n        \n        out = self.language_model(\n            inputs_embeds=video_features,\n            past_key_values=self.kv_cache,\n            use_cache=True,\n            return_dict=True,\n            position_ids=default_position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        self.kv_cache = out.past_key_values\n        self.kv_cache = contiguous_kv(self.kv_cache)\n\n        self._append_position_ids(start_pos_per_layer, q_len)\n        self.last_encoded_frames = video_chunk.shape[0]\n        self.total_processed_frames += video_chunk.shape[0]\n        \n        self._layer_position_ids.clear()\n\n    @torch.inference_mode()\n    def apply_kv_cache_pruning_strict(self, keep_indices_all_layers):\n        if self.kv_cache is None:\n            logger.warning("No KV-Cache to prune")\n            return\n        if not keep_indices_all_layers or len(keep_indices_all_layers[0]) == 0:\n            logger.warning("Empty keep_indices; skip pruning")\n            return\n\n        self._shrink_positions_and_rerotate_keys(keep_indices_all_layers)\n        self._prune_activity_cache(keep_indices_all_layers)\n        \n        logger.info(f"Strict-shrunk KV cache. New length: {get_cache_seq_len(self.kv_cache)}")\n\n    def allocate_budget_by_depth(self, total_budget, num_layers):\n        """Allocate budget uniformly across layers."""\n        budget_per_layer = [total_budget // num_layers] * num_layers\n        diff = total_budget - sum(budget_per_layer)\n        budget_per_layer[-1] += diff\n        return budget_per_layer\n\n    def _compute_attention_scores_manually(self, input_ids, past_key_values):\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = input_ids.shape[1]\n        batch = input_ids.shape[0]\n        \n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        \n        config = self.language_model.config\n        num_layers = config.num_hidden_layers\n        num_heads = config.num_attention_heads\n        num_key_value_heads = config.num_key_value_heads\n        head_dim = config.hidden_size // num_heads\n        \n        attention_weights_list = []\n        \n        hidden_states = inputs_embeds\n        \n        for layer_idx in range(num_layers):\n            layer = self.language_model.model.layers[layer_idx]\n            \n            past_k, past_v = past_key_values[layer_idx]\n            \n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            \n            hidden_states_norm = layer.input_layernorm(hidden_states)\n            \n            attn = layer.self_attn\n            \n            query_states = attn.q_proj(hidden_states_norm)\n            query_states = query_states.view(batch, q_len, num_heads, head_dim).transpose(1, 2)\n            \n            key_states = attn.k_proj(hidden_states_norm)\n            key_states = key_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            value_states = attn.v_proj(hidden_states_norm)\n            value_states = value_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            kv_seq_len = position_ids.max().item() + 1\n            rotary_emb = _get_rotary_module(self.language_model)\n            cos, sin = rotary_emb(value_states, kv_seq_len)\n            cos = cos[position_ids].unsqueeze(1)\n            sin = sin[position_ids].unsqueeze(1)\n            \n            query_states = apply_rotary_pos_emb_1d(query_states, cos, sin)\n            key_states = apply_rotary_pos_emb_1d(key_states, cos, sin)\n            \n            key_states = torch.cat([past_k, key_states], dim=2)\n            value_states = torch.cat([past_v, value_states], dim=2)\n            \n            if num_key_value_heads != num_heads:\n                n_rep = num_heads // num_key_value_heads\n                key_states = torch.repeat_interleave(key_states, n_rep, dim=1)\n                value_states = torch.repeat_interleave(value_states, n_rep, dim=1)\n            \n            attn_weights = torch.matmul(query_states.float(), key_states.float().transpose(-2, -1)) / (head_dim ** 0.5)\n            attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n            attention_weights_list.append(attn_weights)\n        \n        return attention_weights_list\n\n    def prune_kv_cache_by_attention(self, attn_weights_local, attn_weights_global, attn_weights_mixed, num_keep=3000):\n        device = self.device\n        visual_start_idx = self.visual_start_idx\n        num_layers = len(attn_weights_local)\n        \n        curr_cache_lens = self._get_cache_seq_len_per_layer()\n        \n        question_len_local = attn_weights_local[0].shape[2]\n        question_len_global = attn_weights_global[0].shape[2]\n        question_len_mixed = attn_weights_mixed[0].shape[2]\n\n        total_budget = num_keep * num_layers\n        budget_per_layer = self.allocate_budget_by_depth(total_budget, num_layers)\n        \n        keep_indices_all_layers = []\n\n        layer_raw_scores = []\n        layer_configs = []\n\n        for layer_idx in range(len(attn_weights_local)):\n            curr_len = curr_cache_lens[layer_idx]\n            if self.token_activity_cache[layer_idx] is None:\n                self.token_activity_cache[layer_idx] = torch.zeros(curr_len, device=device, dtype=torch.float32)\n            elif self.token_activity_cache[layer_idx].shape[0] < curr_len:\n                diff = curr_len - self.token_activity_cache[layer_idx].shape[0]\n                zeros = torch.zeros(diff, device=device, dtype=torch.float32)\n                self.token_activity_cache[layer_idx] = torch.cat([self.token_activity_cache[layer_idx], zeros])\n            \n            if self.token_activity_cache[layer_idx].shape[0] > curr_len:\n                 self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][:curr_len]\n\n            # Memory hierarchy mode\n            if layer_idx < self.short_term_threshold:\n                layer_type = "short-term"\n                layer_attn_weights = attn_weights_local[layer_idx]\n                question_len = question_len_local\n                layer_recency_alpha = 1\n                k = 20\n                \n            elif layer_idx >= self.long_term_threshold:\n                layer_type = "long-term"\n                layer_attn_weights = attn_weights_global[layer_idx]\n                question_len = question_len_global\n                layer_recency_alpha = 0\n                k = 0.0\n                \n            else:\n                layer_type = "mid-term"\n                layer_attn_weights = attn_weights_mixed[layer_idx]\n                question_len = question_len_mixed\n                progress = (layer_idx - self.short_term_threshold) / (self.long_term_threshold - self.short_term_threshold)\n                layer_recency_alpha = 1.0 - 0.9 * progress\n                k = 20 - 19.5 * progress\n            \n            visual_attn_weights = layer_attn_weights[0].mean(dim=0)[:,visual_start_idx:-1*question_len].mean(dim=0)\n            num_visual_tokens = visual_attn_weights.shape[0]\n            \n            end_idx = visual_start_idx + num_visual_tokens\n            if end_idx <= self.token_activity_cache[layer_idx].shape[0]:\n                self.token_activity_cache[layer_idx][visual_start_idx:end_idx] += visual_attn_weights\n            \n            layer_budget = budget_per_layer[layer_idx]\n            \n            if layer_type == \'long-term\':\n                layer_budget = max(0, layer_budget - 1)\n\n            positions = torch.arange(num_visual_tokens, device=device, dtype=torch.float32)\n            time_distances = (num_visual_tokens - 1 - positions) / max(num_visual_tokens - 1, 1)\n            \n            # Exponential decay\n            recency_weights = torch.exp(-k * time_distances)\n            \n            attn_norm = (visual_attn_weights - visual_attn_weights.min()) / \\\n                        (visual_attn_weights.max() - visual_attn_weights.min() + 1e-6)\n            recency_norm = (recency_weights - recency_weights.min()) / \\\n                        (recency_weights.max() - recency_weights.min() + 1e-6)\n            \n            raw_score = attn_norm * (1 - layer_recency_alpha) + recency_norm * layer_recency_alpha\n            \n            layer_raw_scores.append(raw_score)\n            layer_configs.append({\n                \'budget\': min(layer_budget, num_visual_tokens),\n                \'layer_type\': layer_type,\n                \'visual_start_idx\': visual_start_idx\n            })\n        \n        refined_scores = [s.clone() for s in layer_raw_scores]\n        \n        for i in range(len(refined_scores) - 2, -1, -1):\n            current_type = layer_configs[i][\'layer_type\']\n            \n            # TODO: change hard code here\n            if current_type == \'long-term\':\n                gamma = 0.4  \n            elif current_type == \'mid-term\':\n                gamma = 0.3  \n            else:\n                gamma = 0.1\n            \n            score_current = refined_scores[i]\n            score_next = refined_scores[i+1]\n            \n            if score_current.shape[0] != score_next.shape[0]:\n                score_next_reshaped = score_next.view(1, 1, -1)\n                score_next_interp = F.interpolate(\n                    score_next_reshaped, \n                    size=score_current.shape[0], \n                    mode=\'linear\', \n                    align_corners=False\n                ).view(-1)\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next_interp\n            else:\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next\n\n        for layer_idx, score in enumerate(refined_scores):\n            config = layer_configs[layer_idx]\n            actual_num_keep = config[\'budget\']\n            start_idx = config[\'visual_start_idx\']\n            \n            topk_indices_relative = torch.topk(score, actual_num_keep, sorted=False)[1]\n            topk_indices_absolute = topk_indices_relative + start_idx\n            topk_indices_absolute_sorted = torch.sort(topk_indices_absolute)[0]\n            \n            keep_indices = torch.cat([\n                torch.arange(start_idx, device=device), \n                topk_indices_absolute_sorted\n            ]).tolist()\n            \n            keep_indices_all_layers.append(keep_indices)\n        \n        return keep_indices_all_layers\n    \n    @torch.inference_mode()\n    def pseudo_forward(self, local_question=None, global_question=None):\n        device = self.device\n        \n        if local_question is None:\n            local_question = "What is happening in the video?"\n        if global_question is None:\n            global_question = "What is the main topic of the video?"\n        \n        local_input_ids = self.processor.tokenizer(local_question).input_ids\n        local_input_ids = torch.as_tensor([local_input_ids], device=device, dtype=torch.int)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len_local = local_input_ids.shape[1]\n        batch = local_input_ids.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_local, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_local = self._build_position_ids(start_pos_per_layer[0], q_len_local, batch)\n\n        max_start = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start + q_len_local)\n\n        use_flash_attn = (hasattr(self.language_model.config, \'_attn_implementation\') and \n                        self.language_model.config._attn_implementation in ["flash_attention_2", "sdpa"])\n\n        if use_flash_attn:\n            attn_weights_local = self._compute_attention_scores_manually(local_input_ids, self.kv_cache)\n        else:\n            out_local = self.language_model(\n                input_ids=local_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_local,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_local = out_local.attentions\n        \n        global_input_ids = self.processor.tokenizer(global_question).input_ids\n        global_input_ids = torch.as_tensor([global_input_ids], device=device, dtype=torch.int)\n        \n        q_len_global = global_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_global, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_global = self._build_position_ids(start_pos_per_layer[0], q_len_global, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_global)\n        \n        if use_flash_attn:\n            attn_weights_global = self._compute_attention_scores_manually(global_input_ids, self.kv_cache)\n        else:\n            out_global = self.language_model(\n                input_ids=global_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_global,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_global = out_global.attentions\n        \n        mixed_question = local_question + "; " + global_question\n        mixed_input_ids = self.processor.tokenizer(mixed_question).input_ids\n        mixed_input_ids = torch.as_tensor([mixed_input_ids], device=device, dtype=torch.int)\n        \n        q_len_mixed = mixed_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_mixed, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_mixed = self._build_position_ids(start_pos_per_layer[0], q_len_mixed, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_mixed)\n        \n        if use_flash_attn:\n            attn_weights_mixed = self._compute_attention_scores_manually(mixed_input_ids, self.kv_cache)\n        else:\n            out_mixed = self.language_model(\n                input_ids=mixed_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_mixed,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_mixed = out_mixed.attentions\n        \n        self._layer_position_ids.clear()\n        \n        print(f"GPU memory usage: {self.get_gpu_memory_usage_gb()} GB")\n        current_k_states_len = self.kv_cache[0][0].shape[2]\n        \n        keep_indices_all_layers = self.prune_kv_cache_by_attention(\n            attn_weights_local, attn_weights_global, attn_weights_mixed, \n            num_keep=self.kv_size\n        )\n        \n        if current_k_states_len > self.kv_size:\n            print(f"Applying KV-Cache compression due to k_states > {self.kv_size}")\n            self.apply_kv_cache_pruning_strict(keep_indices_all_layers)\n    \n    @torch.inference_mode()\n    def predict_and_compress(self):\n        local_question, global_question = self.predict_next_question()\n        self.pseudo_forward(local_question, global_question)\n\n    @torch.inference_mode()\n    def question_answering(self, input_text, max_new_tokens=128, temperature=0, repetition_penalty=1.1, pseudo_forward=False):\n        device = self.device\n        stop_token_ids = [self.processor.tokenizer.eos_token_id]\n        output_ids = []\n\n        start_time = time.perf_counter()\n        \n        prompt = input_text[\'prompt\']\n        input_ids = self.processor.tokenizer(prompt).input_ids\n        input_ids = torch.as_tensor([input_ids], device=device)\n\n        past_lens_prefill = self._get_cache_seq_len_per_layer()\n        start_pos_prefill = self._get_next_start_pos_per_layer()\n\n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        q_len_prefill = inputs_embeds.shape[1]\n        batch = inputs_embeds.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_prefill[layer_idx], q_len_prefill, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids = self._build_position_ids(start_pos_prefill[0], q_len_prefill, batch)\n\n        max_start_pos = max(start_pos_prefill) if start_pos_prefill else 0\n        self._set_rotary_required_len(max_start_pos + q_len_prefill)\n\n        out = self.language_model(\n            inputs_embeds=inputs_embeds,\n            use_cache=True,\n            past_key_values=self.kv_cache,\n            position_ids=position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        past_key_values = out.past_key_values\n        logits = out.logits\n        \n        self._append_position_ids(start_pos_prefill, q_len_prefill)\n        self._layer_position_ids.clear()\n\n        for step in range(max_new_tokens):\n            last_token_logits = logits[0, -1, :]\n\n            if repetition_penalty != 1.0 and len(output_ids) > 0:\n                for token_id in set(output_ids):\n                    if last_token_logits[token_id] < 0:\n                        last_token_logits[token_id] *= repetition_penalty\n                    else:\n                        last_token_logits[token_id] /= repetition_penalty\n\n            if temperature == 0.0:\n                _, indices = torch.topk(last_token_logits, 1)\n                token = int(indices[0])\n            else:\n                scaled_logits = last_token_logits / temperature\n                scaled_logits = torch.nan_to_num(\n                    scaled_logits, nan=-float(\'inf\'), posinf=float(\'inf\'), neginf=-float(\'inf\')\n                )\n                probs = F.softmax(scaled_logits, dim=-1)\n                probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)\n                probs_sum = probs.sum()\n                if probs_sum > 0:\n                    probs = probs / probs_sum\n                    token = torch.multinomial(probs, num_samples=1).item()\n                else:\n                    _, indices = torch.topk(last_token_logits, 1)\n                    token = int(indices[0])\n\n            output_ids.append(token)\n            if token in stop_token_ids:\n                break\n\n            curr_start_pos = self._get_next_start_pos_per_layer()\n            \n            max_curr_start = max(curr_start_pos) if curr_start_pos else 0\n            self._set_rotary_required_len(max_curr_start + 1)\n            \n            self._layer_position_ids.clear()\n            for layer_idx in range(self.num_layers):\n                pos_step = torch.tensor([curr_start_pos[layer_idx]], device=device, dtype=torch.long)\n                position_ids_layer = pos_step.view(1, 1)\n                self._layer_position_ids[layer_idx] = position_ids_layer\n            \n            pos_step = torch.tensor([curr_start_pos[0]], device=device, dtype=torch.long)\n            position_ids = pos_step.view(1, 1)\n\n            out = self.language_model(\n                input_ids=torch.as_tensor([[token]], device=device),\n                use_cache=True,\n                past_key_values=past_key_values,\n                position_ids=position_ids,\n                cache_position=None,\n                attention_mask=None,\n            )\n            \n            if (not pseudo_forward) and (step == 0):\n                end_time = time.perf_counter()\n                print(f"TTFT: {end_time - start_time} seconds")\n                \n            logits = out.logits\n            past_key_values = out.past_key_values\n\n            self._append_position_ids(curr_start_pos, 1)\n            self._layer_position_ids.clear()\n\n        output = self.processor.tokenizer.decode(\n            output_ids,\n            skip_special_tokens=True,\n            spaces_between_special_tokens=False,\n            clean_up_tokenization_spaces=True,\n        )\n\n        if not pseudo_forward:\n            current_question = input_text[\'question\']\n            current_options = None\n            formatted_question = input_text.get(\'formatted_question\', None)\n            if formatted_question:\n                option_matches = re.findall(r\'\\([A-Z]\\)\\s*(.+?)(?=\\n\\([A-Z]\\)|\\nThe best answer|\\n*$)\', formatted_question, re.DOTALL)\n                if option_matches:\n                    current_options = [opt.strip() for opt in option_matches]\n            self.conv_history.append((current_question, output, current_options))\n            logger.info(f"Saved conversation to history. Total conversations: {len(self.conv_history)}")\n\n        self._truncate_kv_cache(past_lens_prefill)\n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > past_lens_prefill[layer_idx]):\n                self._position_ids_cache[layer_idx] = self._position_ids_cache[layer_idx][\n                    :past_lens_prefill[layer_idx]\n                ].contiguous()\n\n        new_lens = self._get_cache_seq_len_per_layer()\n        print(f"Answering Cache lengths: min={min(new_lens)}, max={max(new_lens)}")\n        return output\n    \n    def _truncate_kv_cache(self, target_lengths):\n        if self.kv_cache is None:\n            return\n        \n        truncated_cache = []\n        for layer_idx, (k_cache, v_cache) in enumerate(self.kv_cache):\n            if isinstance(target_lengths, int):\n                target_len = target_lengths\n            else:\n                target_len = target_lengths[layer_idx]\n            \n            truncated_k = k_cache[:, :, :target_len, :]\n            truncated_v = v_cache[:, :, :target_len, :]\n            truncated_cache.append((truncated_k, truncated_v))\n        \n        if isinstance(self.kv_cache, DynamicCache):\n            self.kv_cache = DynamicCache.from_legacy_cache(truncated_cache)\n        else:\n            self.kv_cache = truncated_cache\n\n\ndef load_model(model_path=\'llava-onevision-qwen2-7b-ov-hf\',\n               n_init=None, kv_size=None, streaming=True, device="cuda", sample_fps=0.5):\n    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)\n    processor.tokenizer.padding_side = \'left\'\n    \n    system_prompt = \'<|im_start|>system \\nYou are a helpful assistant.<|im_end|><|im_start|>user \'\n    init_prompt_ids = processor.tokenizer(system_prompt, return_tensors="pt").input_ids.to(device)\n    \n    base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(\n        model_path,\n        device_map="auto",\n        torch_dtype=torch.float16,\n        trust_remote_code=True,\n    )\n\n    model = LlavaOneVision_Hermes.__new__(LlavaOneVision_Hermes)\n    model.__dict__ = base_model.__dict__.copy()\n\n    Abstract_Hermes.__init__(\n        model, \n        processor, \n        init_prompt_ids.tolist(), \n        kv_size,\n    )\n    model.streaming = streaming\n    \n    num_layers = base_model.language_model.config.num_hidden_layers\n    model.num_layers = num_layers\n    model._position_ids_cache = [None for _ in range(num_layers)]\n    \n    model.short_term_ratio = 0.1\n    model.long_term_ratio = 0.3\n    model.short_term_threshold = int(model.num_layers * model.short_term_ratio)\n    model.long_term_threshold = int(model.num_layers * (1 - model.long_term_ratio))\n    model.token_activity_cache = [None for _ in range(num_layers)]\n    \n    model.total_processed_frames = 0\n    \n    model._layer_position_ids = {}\n    model._hook_handles = []\n    model._register_forward_hooks()\n    model._patch_rotary_embeddings()\n    \n    logger.info(f\'n_init: {init_prompt_ids.shape[1] if n_init is None else n_init}\')\n    logger.info(f\'kv_size: {kv_size}\')\n\n    model.eval()\n\n    return model, processor\n', 'pinned_source/inference/abstract_hermes.py': 'import torch\n\nclass Abstract_Hermes:\n    kv_cache = None\n\n    def __init__(self, processor, init_prompt_ids, kv_size):\n        self.processor = processor\n        self.init_prompt_ids = init_prompt_ids\n        self.kv_size = kv_size\n        self.last_encoded_frames = 0\n        self.visual_start_idx = 14\n        self.conv_history = []\n        \n    def clear_cache(self):\n        self.kv_cache = None\n        self.conv_history = []\n        torch.cuda.empty_cache()\n        torch.cuda.ipc_collect()\n\n    def encode_init_prompt(self):\n        if not isinstance(self.init_prompt_ids, torch.Tensor):\n            self.init_prompt_ids = torch.as_tensor(self.init_prompt_ids, device=self.device)\n        output = self.language_model(input_ids=self.init_prompt_ids, use_cache=True, return_dict=True)\n        self.kv_cache = output.past_key_values\n        self.visual_start_idx = self.kv_cache[0][0].shape[2]\n\n    def get_prompt(self, query, mc=False):\n        prompt = f"\\n{query}<|im_end|><|im_start|>assistant\\n"\n        \n        if mc:\n            prompt += \'Best option: (\'\n        return prompt\n\n    def get_gpu_memory_usage_gb(self):\n        return torch.cuda.max_memory_allocated() / (1024**3)', 'pinned_source/inference/reindex_1d.py': 'import torch\nfrom typing import Tuple\nfrom transformers import DynamicCache\n\n\ndef get_cache_seq_len(past_key_values) -> int:\n    if past_key_values is None:\n        return 0\n    if isinstance(past_key_values, DynamicCache):\n        return past_key_values.get_seq_length()\n    return past_key_values[0][0].shape[2]\n\n\ndef contiguous_kv(past_key_values):\n    new_legacy = []\n    for (k_layer, v_layer) in past_key_values:\n        new_legacy.append((k_layer.contiguous(), v_layer.contiguous()))\n    return new_legacy\n\n\ndef _get_rotary_module(llm) -> torch.nn.Module:\n    if hasattr(llm, "rotary_emb"):\n        return llm.rotary_emb\n    if hasattr(llm, "model") and hasattr(llm.model, "rotary_emb"):\n        return llm.model.rotary_emb\n\n    if hasattr(llm, "model") and hasattr(llm.model, "layers"):\n        if len(llm.model.layers) > 0 and hasattr(llm.model.layers[0], "self_attn"):\n            if hasattr(llm.model.layers[0].self_attn, "rotary_emb"):\n                return llm.model.layers[0].self_attn.rotary_emb\n    raise AttributeError("Cannot find rotary_emb module on language_model")\n\n\ndef rotate_half(x):\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\ndef apply_rotary_pos_emb_1d(x, cos, sin):\n    if cos.dim() == 3 and x.dim() == 4:\n        cos = cos.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n        sin = sin.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n    \n    x_embed = (x * cos) + (rotate_half(x) * sin)\n    return x_embed\n\n\ndef rotary_delta(cos_old, sin_old, cos_new, sin_new):\n    cos_delta = cos_new * cos_old + sin_new * sin_old\n    sin_delta = sin_new * cos_old - cos_new * sin_old\n    return cos_delta, sin_delta\n\n\n@torch.inference_mode()\ndef compute_cos_sin_for_positions(\n    llm, \n    seq_len: int, \n    position_ids_1d: torch.Tensor, \n    dtype: torch.dtype, \n    device: torch.device\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    rotary_emb = _get_rotary_module(llm)\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    \n    position_ids = position_ids_1d.view(1, -1).to(device)\n    \n    dummy_h = torch.zeros((1, seq_len, hidden_size), device=device, dtype=dtype)\n    \n    \n    seq_len_rope = position_ids.max().item() + 1\n    cos, sin = rotary_emb(dummy_h, seq_len=seq_len_rope)\n    cos = cos[position_ids]\n    sin = sin[position_ids]\n    \n    cos = cos.to(dtype)\n    sin = sin.to(dtype)\n    \n    return cos, sin\n\n@torch.inference_mode()\ndef compute_cos_sin_with_custom_base(\n    llm,\n    seq_len: int,\n    position_ids_1d: torch.Tensor,\n    dtype: torch.dtype,\n    device: torch.device,\n    rope_base: float = 10000.0\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    num_attention_heads = getattr(config, "num_attention_heads", 32)\n    head_dim = hidden_size // num_attention_heads\n    \n    \n    inv_freq = 1.0 / (rope_base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))\n    \n    freqs = torch.outer(position_ids_1d.float(), inv_freq)\n    \n    \n    emb = torch.cat([freqs, freqs], dim=-1)\n    \n    cos = emb.cos().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    sin = emb.sin().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    \n    return cos, sin\n\n\ndef get_layer_rope_base(\n    layer_idx: int,\n    num_layers: int,\n    short_term_ratio: float = 0.3,\n    long_term_ratio: float = 0.3,\n    short_term_base: float = 10000.0,\n    mid_term_base_range: Tuple[float, float] = (10000.0, 50000.0),\n    long_term_base: float = 50000.0,\n) -> float:\n    \n    short_term_threshold = int(num_layers * short_term_ratio)\n    long_term_threshold = int(num_layers * (1 - long_term_ratio))\n    \n    if layer_idx < short_term_threshold:\n        \n        return short_term_base\n    elif layer_idx >= long_term_threshold:\n        \n        return long_term_base\n    else:\n        \n        mid_start = short_term_threshold\n        mid_end = long_term_threshold\n        progress = (layer_idx - mid_start) / max(mid_end - mid_start, 1)\n        base_start, base_end = mid_term_base_range\n        return base_start + (base_end - base_start) * progress\n\n\n@torch.inference_mode()\ndef apply_rotary_delta_to_keys_only(\n    key_states: torch.Tensor, \n    cos_delta: torch.Tensor, \n    sin_delta: torch.Tensor\n) -> torch.Tensor:\n    \n    k_rot = apply_rotary_pos_emb_1d(key_states, cos_delta, sin_delta)\n    return k_rot\n\n', 'provenance/twinpast_base.py': '"""Run the frozen, annotation-guided TwinPast development diagnostic."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport statistics\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nARTIFACT = "HERMES_twinpast_diagnostic_evidence.zip"\nARMS = ("question_only", "recent_four", "history_a", "history_b")\nORDERS = ("canonical", "reversed")\nLETTERS = "ABCDEFGH"\nHOSTS = ("huggingface.co", "hf-mirror.com")\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef atomic(path, data):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef fetch(bundle, relative, dest, expected_sha, size=None, reuse=None):\n    import requests\n\n    if dest.is_file() and digest(dest) == expected_sha:\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temporary = dest.with_suffix(".part")\n    if reuse and reuse.is_file() and digest(reuse) == expected_sha:\n        shutil.copyfile(reuse, temporary)\n        temporary.replace(dest)\n        print(f"MEDIA_REUSED {relative}", flush=True)\n        return\n    cap = size if size is not None else 2 * 1024**2\n    remaining_attempts = [(host, direct) for direct in (False, True) for host in HOSTS]\n    for host, direct in remaining_attempts:\n        url = f"https://{host}/datasets/{bundle[\'manifest\'][\'dataset\']}/resolve/{bundle[\'manifest\'][\'revision\']}/{relative}"\n        try:\n            with requests.Session() as session:\n                session.trust_env = not direct\n                with session.get(url, stream=True, timeout=(8, 15)) as response, temporary.open("wb") as stream:\n                    response.raise_for_status()\n                    start, last_report, count = time.monotonic(), time.monotonic(), 0\n                    for block in response.iter_content(128 * 1024):\n                        count += len(block)\n                        now = time.monotonic()\n                        if count > cap or now - start > 120:\n                            raise ValueError("Download exceeded size or elapsed-time bound")\n                        stream.write(block)\n                        if now - last_report >= 8:\n                            print(f"MEDIA_PROGRESS {relative} {count / 1024**2:.1f} MiB", flush=True)\n                            last_report = now\n            if (size is not None and count != size) or digest(temporary) != expected_sha:\n                raise ValueError("Downloaded size or SHA-256 does not match")\n            temporary.replace(dest)\n            print(f"MEDIA_READY {relative} {count / 1024**2:.1f} MiB", flush=True)\n            return\n        except (requests.RequestException, OSError, ValueError) as exc:\n            print(f"MEDIA_RETRY {relative} host={host} direct={direct} reason={type(exc).__name__}", flush=True)\n    raise RuntimeError(f"Could not acquire {relative}. Verified completed files remain available for rerun.")\n\n\ndef cases(bundle):\n    plans = {row["pair_id"]: row for row in bundle["manifest"]["selected_pairs"]}\n    visual = {row["pair_id"]: row for row in bundle["visual"]}\n    media = {row["pair_id"]: row for row in bundle["media"]}\n    assert len(plans) == len(visual) == len(media) == 12 and plans.keys() == visual.keys() == media.keys()\n    selected = []\n    for pair_id, plan in plans.items():\n        if visual[pair_id]["status"] == "clear_contrast":\n            checked = media[pair_id]\n            assert checked["status"] == "checked" and checked["task"] == plan["task"]\n            assert checked["grid_s"] == sorted(set(checked["grid_s"])) and len(checked["grid_s"]) == 6\n            assert checked["grid_s"][-4:] == plan["latest_four_grid_s"]\n            assert checked["grid_s"][-1] < plan["query_time_s"]\n            for branch in "ab":\n                assert all(checked["decoded"][branch]["early_samples_inside"])\n            assert checked["decoded"]["a"]["decoded_rgb_sha256"][-4:] == checked["decoded"]["b"]["decoded_rgb_sha256"][-4:]\n            selected.append((plan, checked, visual[pair_id]))\n    assert len(selected) == 9\n    return selected\n\n\ndef annotation_rows(bundle, raw):\n    assert sha(raw) == bundle["manifest"]["annotation_sha256"], "Pinned annotations changed"\n    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]\n    result = {row["pair_id"]: row for row in rows}\n    assert len(rows) == len(result) == bundle["manifest"]["dataset_pairs"]\n    for plan, media, visual in cases(bundle):\n        row = result[plan["pair_id"]]\n        assert all(row[field] == plan[field] for field in ("video_a", "video_b", "task"))\n        assert row["t_query"] == plan["query_time_s"] and row["question"].strip()\n        choices = row["options"]\n        assert 2 <= len(choices) <= len(LETTERS) and all(c.strip() for c in choices)\n        assert len({c.strip().casefold() for c in choices}) == len(choices)\n        assert row["answer_a"] != row["answer_b"]\n        for branch in "ab":\n            assert row[f"answer_{branch}"] in LETTERS[:len(choices)]\n            assert row[f"answer_{branch}"] == visual[f"observed_option_{branch}"]\n            seconds = media["decoded"][branch]["frame_start_s"]\n            assert all(t < row["t_query"] for t in seconds)\n            assert all(any(lo - 1e-6 <= t <= hi + 1e-6\n                           for lo, hi in row["evidence_intervals"][branch.upper()]) for t in seconds[:2])\n    return result\n\n\ndef decode(path, recorded):\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    frames = reader.get_batch(recorded["source_frames"]).asnumpy()\n    assert [sha(frame.tobytes()) for frame in frames] == recorded["decoded_rgb_sha256"], (\n        f"Decoded frames differ from the frozen audit for {path.name}")\n    return frames\n\n\ndef features(model, frames):\n    import torch\n\n    chw = torch.from_numpy(frames).permute(0, 3, 1, 2)\n    pixels = model.processor.video_processor(chw, return_tensors="pt").pixel_values_videos\n    output = model.get_video_features(pixels.to(model.device, model.dtype))\n    assert output.shape[0] == 1 and output.shape[1] % len(frames) == 0\n    return output\n\n\ndef score(model, system, prefix, option_ids, visual, cached=False):\n    import torch\n\n    lm = model.language_model\n    embed = lm.get_input_embeddings()\n    question = embed(prefix)\n    left = system if visual is None else torch.cat((system, visual), dim=1)\n    if cached:\n        initial = lm(inputs_embeds=left, use_cache=True, return_dict=True)\n        output = lm(inputs_embeds=question, past_key_values=initial.past_key_values, use_cache=True, return_dict=True)\n    else:\n        output = lm(inputs_embeds=torch.cat((left, question), dim=1), use_cache=False, return_dict=True)\n    result = output.logits[0, -1, option_ids].float().detach().cpu()\n    assert torch.isfinite(result).all()\n    return result, left.shape[1] + question.shape[1]\n\n\ndef make_prompt(model, question, choices, order, with_video):\n    import torch\n\n    perm = list(range(len(choices)))\n    if order == "reversed":\n        perm.reverse()\n    letters = LETTERS[:len(choices)]\n    context = "Answer the question about the video." if with_video else "Answer the question."\n    options = "\\n".join(f"({letter}) {choices[i]}" for letter, i in zip(letters, perm, strict=True))\n    text = model.get_prompt(f"{context}\\nQuestion: {question}\\nOptions:\\n{options}\\n"\n                            "Reply with only the letter of the best option.") + "Answer letter:\\n"\n    tokenizer = model.processor.tokenizer\n    ids = tokenizer.encode(text, add_special_tokens=False)\n    tokens = []\n    for letter in letters:\n        suffix = tokenizer.encode(letter, add_special_tokens=False)\n        assert len(suffix) == 1 and tokenizer.encode(text + letter, add_special_tokens=False) == ids + suffix\n        tokens.extend(suffix)\n    assert len(set(tokens)) == len(tokens)\n    return torch.tensor([ids], device=model.device), tokens, perm, sha(text.encode("utf-8"))\n\n\ndef evaluate_pair(model, plan, media, row, paths, preflight=False):\n    import numpy as np\n    import torch\n\n    frames = {branch: decode(paths[branch], media["decoded"][branch]) for branch in "ab"}\n    assert np.array_equal(frames["a"][2:], frames["b"][2:])\n    recent = features(model, frames["a"][2:])\n    a = features(model, frames["a"][:2])\n    b = features(model, frames["b"][:2])\n    assert a.shape == b.shape and a.shape[1] * 2 == recent.shape[1]\n    visuals = {"question_only": None, "recent_four": recent,\n               "history_a": torch.cat((a, recent), dim=1), "history_b": torch.cat((b, recent), dim=1)}\n    lm = model.language_model\n    system = lm.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    results, checks = [], []\n    for order in ORDERS:\n        prompt = make_prompt(model, row["question"], row["options"], order, True)\n        for arm in ARMS:\n            ids, option_ids, perm, prompt_sha = prompt if arm != "question_only" else make_prompt(\n                model, row["question"], row["options"], order, False)\n            started = time.perf_counter()\n            output, count = score(model, system, ids, option_ids, visuals[arm])\n            elapsed = time.perf_counter() - started\n            if preflight:\n                replay, replay_count = score(model, system, ids, option_ids, visuals[arm])\n                torch.testing.assert_close(output, replay, atol=0, rtol=0)\n                assert replay_count == count\n                if arm == "history_a":\n                    cached, _ = score(model, system, ids, option_ids, visuals[arm], cached=True)\n                    torch.testing.assert_close(output, cached, atol=0.025, rtol=0.005)\n                    assert output.argmax().item() == cached.argmax().item(), "Native-cache top option differs"\n                    checks.append(dict(order=order, native_cache_max_abs=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for idx, value in zip(perm, output.tolist(), strict=True):\n                values[idx] = value\n            # Canonical tie-breaking prevents label order from deciding exact ties.\n            pred = LETTERS[max(range(len(values)), key=values.__getitem__)]\n            recent_sha = media["decoded"]["a"]["decoded_rgb_sha256"][-4:] if arm != "question_only" else []\n            earlier_sha = media["decoded"][arm[-1]]["decoded_rgb_sha256"][:2] if arm.startswith("history") else []\n            results.append(dict(pair_id=plan["pair_id"], task=plan["task"], order=order, arm=arm, prediction=pred,\n                                gold_a=row["answer_a"], gold_b=row["answer_b"], scores=values, permutation=perm,\n                                label_token_ids=option_ids, prompt_sha256=prompt_sha, lm_tokens=count,\n                                earlier_frame_sha256=earlier_sha, recent_frame_sha256=recent_sha,\n                                forward_s=elapsed))\n    return results, checks\n\n\ndef summarize(blocks):\n    pairs = []\n    for block in blocks:\n        records = {(r["order"], r["arm"]): r for r in block}\n        pair = dict(pair_id=block[0]["pair_id"], task=block[0]["task"])\n        for order in ORDERS:\n            a, b = (records[order, arm] for arm in ("history_a", "history_b"))\n            assert a["lm_tokens"] == b["lm_tokens"] and a["prompt_sha256"] == b["prompt_sha256"]\n            assert a["recent_frame_sha256"] == b["recent_frame_sha256"]\n            ai, bi = (LETTERS.index(a[f"gold_{branch}"]) for branch in "ab")\n            ma = a["scores"][ai] - a["scores"][bi]\n            mb = b["scores"][ai] - b["scores"][bi]\n            wins = int(a["prediction"] == a["gold_a"]) + int(b["prediction"] == b["gold_b"])\n            reversal = a["prediction"] != b["prediction"]\n            pair[order] = dict(correct_branches=wins, correct_reversal=wins == 2, unchanged=not reversal,\n                               wrong_reversal=reversal and wins != 2, margin_a=ma, margin_b=mb,\n                               directional_change=ma - mb)\n            for arm in ("question_only", "recent_four"):\n                prediction = records[order, arm]["prediction"]\n                pair[order][f"{arm}_correct_branches"] = int(prediction in (a["gold_a"], a["gold_b"]))\n        pair["both_orders_correct"] = all(pair[order]["correct_reversal"] for order in ORDERS)\n        pair["predictions_order_stable"] = {\n            arm: records["canonical", arm]["prediction"] == records["reversed", arm]["prediction"] for arm in ARMS\n        }\n        pairs.append(pair)\n    orders = {order: dict(\n        both_correct=sum(row[order]["correct_reversal"] for row in pairs),\n        correct_branches=sum(row[order]["correct_branches"] for row in pairs),\n        question_only_correct_branches=sum(row[order]["question_only_correct_branches"] for row in pairs),\n        recent_four_correct_branches=sum(row[order]["recent_four_correct_branches"] for row in pairs),\n        wrong_reversals=sum(row[order]["wrong_reversal"] for row in pairs),\n        unchanged=sum(row[order]["unchanged"] for row in pairs),\n        median_directional_change=statistics.median(row[order]["directional_change"] for row in pairs))\n        for order in ORDERS} if pairs else {}\n    robust = sum(row["both_orders_correct"] for row in pairs)\n    numeric_gate = len(pairs) == 9 and robust >= 5 and all(x["median_directional_change"] > 0 for x in orders.values())\n    return dict(pairs=len(pairs), robust_correct_pairs=robust, order_summary=orders, per_pair=pairs,\n                numeric_development_gate=numeric_gate,\n                advancement="pending source-group verification" if numeric_gate else\n                ("stop on this backbone if diagnostic validity holds" if len(pairs) == 9 and robust == 0\n                 else "review diagnostic outcomes before advancing"),\n                source_groups="The pinned annotations do not identify original source groups. Verify independently.")\n\n\ndef package(work):\n    artifact = work / ARTIFACT\n    temporary = work / (ARTIFACT + ".tmp")\n    files = [path for name in ("exports", "checkpoints", "provenance", "pinned_source", "logs")\n             for path in (work / name).rglob("*") if path.is_file() and path.suffix != ".tmp"]\n    files += [path for path in (work / "launch.json", work / "completion.json") if path.exists()]\n    with ZipFile(temporary, "w", ZIP_DEFLATED) as archive:\n        entries = []\n        for path in sorted(files):\n            raw = path.read_bytes()\n            relative = path.relative_to(work).as_posix()\n            archive.writestr(relative, raw)\n            entries.append(dict(path=relative, bytes=len(raw), sha256=sha(raw)))\n        archive.writestr("evidence_manifest.json", json.dumps(entries, indent=2))\n    temporary.replace(artifact)\n    print(f"EVIDENCE_ZIP={artifact}", flush=True)\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    bundle_path = work / "provenance" / "bundle.json"\n    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))\n    selection = cases(bundle)\n    dataset = work / "media" / "data.jsonl"\n    fetch(bundle, "data.jsonl", dataset, bundle["manifest"]["annotation_sha256"])\n    rows = annotation_rows(bundle, dataset.read_bytes())\n    atomic(work / "provenance" / "selected_annotations.json", [rows[plan["pair_id"]] for plan, _, _ in selection])\n    needed = sum(meta["bytes"] for plan, media, _ in selection\n                 for branch, meta in media["decoded"].items()\n                 if not (work / "media" / plan[f"video_{branch}"]).is_file()\n                 or (work / "media" / plan[f"video_{branch}"]).stat().st_size != meta["bytes"])\n    assert shutil.disk_usage(work).free >= needed + 256 * 1024**2, "Free data-disk space before running"\n    locations = {}\n    print(f"MEDIA_PLAN 18 pinned clips, up to {needed / 1024**2:.1f} MiB not present", flush=True)\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        pending = []\n        for plan, media, _ in selection:\n            locations[plan["pair_id"]] = {}\n            for branch, meta in media["decoded"].items():\n                relative = plan[f"video_{branch}"]\n                dest = work / "media" / relative\n                reuse = args.media_reuse / Path(relative).relative_to("videos")\n                pending.append(pool.submit(fetch, bundle, relative, dest, meta["sha256"], meta["bytes"], reuse))\n                locations[plan["pair_id"]][branch] = dest\n        for download in pending:\n            download.result()\n    settings = dict(bundle_sha256=digest(bundle_path), runner_sha256=digest(Path(__file__)),\n                    model_name=args.model.name, torch_version=torch.__version__,\n                    gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,\n                    driver=torch.cuda.driver_version() if hasattr(torch.cuda, "driver_version") else None)\n    identity = sha(json.dumps(settings, sort_keys=True).encode())\n    checkpoint_dir = work / "checkpoints"\n    checkpoint_dir.mkdir(exist_ok=True)\n    atomic(work / "exports" / "run_config.json", settings)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    blocks = []\n    with torch.inference_mode():\n        for index, (plan, media, _) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            checkpoint = checkpoint_dir / f"{pair_id}.json"\n            if index and checkpoint.is_file():\n                saved = json.loads(checkpoint.read_text(encoding="utf-8"))\n                assert saved["identity"] == identity and saved["pair_id"] == pair_id, "Use a new WORK for changed settings"\n                records = saved["records"]\n                assert len(records) == 8 and {(r["order"], r["arm"]) for r in records} == {\n                    (order, arm) for order in ORDERS for arm in ARMS}\n                assert all(r["pair_id"] == pair_id for r in records)\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n            else:\n                print(f"PAIR_RUNNING {index + 1}/9 {pair_id}", flush=True)\n                records, checks = evaluate_pair(model, plan, media, rows[pair_id], locations[pair_id], preflight=index == 0)\n                if index == 0:\n                    if checkpoint.exists():\n                        assert json.loads(checkpoint.read_text(encoding="utf-8"))["identity"] == identity, (\n                            "Use a new WORK for changed settings")\n                    atomic(work / "exports" / "preflight.json", dict(status="pass", pair_id=pair_id, checks=checks))\n                    print("TWINPAST_PREFLIGHT_PASS", flush=True)\n                atomic(checkpoint, dict(identity=identity, pair_id=pair_id, records=records))\n                print(f"PAIR_DONE {index + 1}/9 {pair_id}", flush=True)\n            blocks.append(records)\n    summary = summarize(blocks)\n    atomic(work / "exports" / "summary.json", summary)\n    fields = ("pair_id", "task", "order", "arm", "prediction", "gold_a", "gold_b", "scores", "permutation",\n              "label_token_ids", "prompt_sha256", "lm_tokens", "earlier_frame_sha256", "recent_frame_sha256", "forward_s")\n    with (work / "exports" / "scores.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value) if isinstance(value, list) else value for key, value in row.items()}\n                         for block in blocks for row in block)\n    atomic(work / "completion.json", dict(status="complete", pairs=len(blocks), scores=sum(map(len, blocks))))\n    print(f"TWINPAST_DIAGNOSTIC_COMPLETE pairs={len(blocks)} scores={sum(map(len, blocks))}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--work", required=True, type=Path)\n    parser.add_argument("--model", required=True, type=Path)\n    parser.add_argument("--media-reuse", type=Path, default=Path("/root/autodl-tmp/twinpast-development"))\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/history_only_runner.py': '"""Test whether earlier evidence alone supports the frozen TwinPast answers."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nfrom pathlib import Path\nimport time\n\nimport twinpast_base as base\n\nARTIFACT = "HERMES_twinpast_history_only_evidence.zip"\nDENSE_COUNT = 6\n\n\ndef dense_indices(seconds, intervals, earlier, cutoff):\n    """Keep the two original frames and add four spread across valid native timestamps."""\n    import numpy as np\n\n    mask = np.zeros(len(seconds), dtype=bool)\n    for start, stop in intervals:\n        mask |= (seconds >= start - 1e-6) & (seconds <= stop + 1e-6)\n    eligible = np.flatnonzero(mask & (seconds < cutoff)).tolist()\n    assert sorted(set(earlier)) == earlier and len(earlier) == 2\n    assert set(earlier) <= set(eligible), "Frozen history frame outside eligible annotated evidence"\n    if len(eligible) < DENSE_COUNT:\n        raise ValueError(f"Only {len(eligible)} distinct annotated frames before recent context")\n    chosen = set(earlier)\n    for k in range(DENSE_COUNT - len(earlier)):\n        target = k * (len(eligible) - 1) / (DENSE_COUNT - len(earlier) - 1)\n        pos = min((i for i, ix in enumerate(eligible) if ix not in chosen), key=lambda i: abs(i - target))\n        chosen.add(eligible[pos])\n    return sorted(chosen)\n\n\ndef prepare_dense(paths, annotated, media):\n    import numpy as np\n    from decord import VideoReader\n\n    result = {}\n    cutoff = min(annotated["t_query"], *(media["decoded"][b]["frame_start_s"][2] for b in "ab"))\n    for branch in "ab":\n        reader = VideoReader(str(paths[branch]), num_threads=1)\n        seconds = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n        seconds -= seconds[0, 0]\n        seconds = seconds[:, 0]\n        indices = dense_indices(seconds, annotated["evidence_intervals"][branch.upper()],\n                                media["decoded"][branch]["source_frames"][:2], cutoff)\n        frames = reader.get_batch(indices).asnumpy()\n        result[branch] = dict(frames=frames, indices=indices, seconds=seconds[indices].tolist(),\n                              cutoff_s=cutoff, hashes=[base.sha(frame.tobytes()) for frame in frames])\n    return result\n\n\ndef contact_sheet(path, pair_id, condition, windows):\n    from PIL import Image, ImageDraw\n\n    cols = len(windows["a"]["frames"])\n    canvas = Image.new("RGB", (320 * cols, 450), "white")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((8, 5), f"{pair_id} | {condition} | Native frames resized to fit", fill="black")\n    for row, branch in enumerate("ab"):\n        for col, (frame, seconds) in enumerate(zip(windows[branch]["frames"], windows[branch]["seconds"], strict=True)):\n            x, y = col * 320, 23 + row * 210\n            picture = Image.fromarray(frame)\n            picture.thumbnail((320, 180))\n            canvas.paste(picture, (x, y))\n            draw.text((x + 4, y + 182), f"{branch.upper()} | {seconds:.3f} s", fill="black")\n    path.parent.mkdir(exist_ok=True)\n    canvas.save(path, quality=90)\n\n\ndef evaluate(model, annotated, windows, reference, preflight=False):\n    import torch\n\n    system = model.language_model.get_input_embeddings()(\n        torch.as_tensor(model.init_prompt_ids, device=model.device))\n    visual = {branch: base.features(model, window["frames"]) for branch, window in windows.items()}\n    assert visual["a"].shape == visual["b"].shape\n    rows, checks = [], []\n    for order in base.ORDERS:\n        ids, option_ids, perm, prompt_sha = base.make_prompt(\n            model, annotated["question"], annotated["options"], order, True)\n        for branch in "ab":\n            old = reference[order, branch]\n            assert old["prompt_sha256"] == prompt_sha and old["label_token_ids"] == option_ids\n            start = time.perf_counter()\n            output, length = base.score(model, system, ids, option_ids, visual[branch])\n            elapsed = time.perf_counter() - start\n            if preflight:\n                repeat, _ = base.score(model, system, ids, option_ids, visual[branch])\n                torch.testing.assert_close(output, repeat, rtol=0, atol=0)\n                if branch == "a":\n                    cached, _ = base.score(model, system, ids, option_ids, visual[branch], cached=True)\n                    torch.testing.assert_close(output, cached, rtol=0.005, atol=0.025)\n                    assert output.argmax().item() == cached.argmax().item()\n                    checks.append(dict(order=order, max_native_cache_diff=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, output.tolist(), strict=True):\n                values[i] = value\n            prediction = base.LETTERS[max(range(len(values)), key=values.__getitem__)]\n            gold = annotated[f"answer_{branch}"]\n            window = windows[branch]\n            rows.append(dict(order=order, branch=branch, prediction=prediction, gold=gold, scores=values,\n                             prompt_sha256=prompt_sha, lm_tokens=length, frame_indices=window["indices"],\n                             frame_seconds=window["seconds"], frame_hashes=window["hashes"],\n                             prior_prediction=old["prediction"], prior_scores=old["scores"], forward_s=elapsed))\n        assert rows[-1]["lm_tokens"] == rows[-2]["lm_tokens"]\n    return rows, checks\n\n\ndef outcome(rows, annotated):\n    wins = {}\n    ai, bi = (base.LETTERS.index(annotated[f"answer_{b}"]) for b in "ab")\n    for order in base.ORDERS:\n        a, b = (next(r for r in rows if r["order"] == order and r["branch"] == branch) for branch in "ab")\n        margin = lambda x: x[ai] - x[bi]\n        wins[order] = dict(branches_correct=int(a["prediction"] == a["gold"]) + int(b["prediction"] == b["gold"]),\n                           directional_change=margin(a["scores"]) - margin(b["scores"]))\n    return dict(orders=wins, robust=all(row["branches_correct"] == 2 for row in wins.values()))\n\n\ndef acquire(args, bundle, selection):\n    paths = {}\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        downloads = []\n        for plan, media, _ in selection:\n            pair_paths = paths[plan["pair_id"]] = {}\n            for branch, check in media["decoded"].items():\n                relative = Path(plan[f"video_{branch}"])\n                reused = args.previous / "media" / relative\n                if reused.is_file() and base.digest(reused) == check["sha256"]:\n                    pair_paths[branch] = reused\n                    print(f"MEDIA_REUSED {relative.as_posix()}", flush=True)\n                else:\n                    dest = args.work / "media" / relative\n                    downloads.append(pool.submit(base.fetch, bundle, relative.as_posix(), dest, check["sha256"],\n                                                 check["bytes"], args.media_reuse / relative.relative_to("videos")))\n                    pair_paths[branch] = dest\n        for download in downloads:\n            download.result()\n    return paths\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    prior = json.loads((provenance / "prior.json").read_bytes())\n    selection = base.cases(bundle)\n    annotations = {r["pair_id"]: r for r in prior["annotations"]}\n    assert annotations.keys() == {plan["pair_id"] for plan, _, _ in selection}\n    paths = acquire(args, bundle, selection)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    config = dict(model_revision=args.model.name, gpu=torch.cuda.get_device_name(0), torch=torch.__version__,\n                  cuda=torch.version.cuda, provenance_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir())})\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    base.atomic(exports / "run_config.json", config)\n    reports = []\n    with torch.inference_mode():\n        for index, (plan, media, screened) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            annotated = annotations[pair_id]\n            assert all(annotated[f"answer_{b}"] == screened[f"observed_option_{b}"] for b in "ab")\n            old = {(r["order"], r["arm"][-1]): r for r in prior["pairs"][pair_id]}\n            checkpoint = checkpoints / f"{pair_id}.json"\n            expected = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if expected is not None:\n                assert expected["identity"] == ident and expected["pair_id"] == pair_id\n            if index and expected is not None:\n                reports.append(expected["summary"])\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n                continue\n            two = {}\n            for branch in "ab":\n                record = media["decoded"][branch]\n                assert all(r["earlier_frame_sha256"] == record["decoded_rgb_sha256"][:2] for (_, b), r in old.items() if b == branch)\n                frames = base.decode(paths[pair_id][branch], record)\n                two[branch] = dict(frames=frames[:2], indices=record["source_frames"][:2],\n                                   seconds=record["frame_start_s"][:2], hashes=record["decoded_rgb_sha256"][:2])\n            rows, checks = evaluate(model, annotated, two, old, preflight=index == 0)\n            if index == 0:\n                base.atomic(exports / "preflight.json", dict(status="pass", checks=checks))\n                print("TWINPAST_HISTORY_ONLY_PREFLIGHT_PASS", flush=True)\n            comparison = outcome(rows, annotated)\n            summary = dict(pair_id=pair_id, task=plan["task"], two_history_only=comparison,\n                           prior_history_with_recent_robust=all(r["prediction"] == r[f"gold_{b}"] for (_, b), r in old.items()))\n            dense_rows = []\n            if plan["task"] == "HSR":\n                contact_sheet(exports / "review" / f"{pair_id}_two.jpg", pair_id, "Two earlier frames", two)\n                if not comparison["robust"]:\n                    try:\n                        dense = prepare_dense(paths[pair_id], annotated, media)\n                    except ValueError as exc:\n                        summary["dense_skip"] = str(exc)\n                    else:\n                        dense_rows, _ = evaluate(model, annotated, dense, old)\n                        summary["dense_history_only"] = outcome(dense_rows, annotated)\n                        contact_sheet(exports / "review" / f"{pair_id}_dense.jpg", pair_id, "Six earlier frames", dense)\n            base.atomic(checkpoint, dict(pair_id=pair_id, identity=ident, summary=summary,\n                                        history_two=rows, history_dense=dense_rows))\n            reports.append(summary)\n            print(f"PAIR_DONE {index + 1}/9 {pair_id} two_order_robust={comparison[\'robust\']}", flush=True)\n    dense = [item["dense_history_only"] for item in reports if "dense_history_only" in item]\n    summary = dict(pairs=len(reports), two_history_only_robust=sum(r["two_history_only"]["robust"] for r in reports),\n                   restored_from_original=sum(r["two_history_only"]["robust"] and not r["prior_history_with_recent_robust"]\n                                              for r in reports),\n                   lost_from_original=sum(not r["two_history_only"]["robust"] and r["prior_history_with_recent_robust"]\n                                          for r in reports),\n                   dense_pairs=len(dense), dense_robust=sum(r["robust"] for r in dense), per_pair=reports)\n    base.atomic(exports / "summary.json", summary)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(reports), dense_pairs=len(dense)))\n    print("TWINPAST_HISTORY_ONLY_COMPLETE", json.dumps({k: v for k, v in summary.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/backbone_runner.py': '"""Replay the frozen history-only windows on the larger same-family backbone."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport history_only_runner as history\n\nbase = history.base\nARTIFACT = "HERMES_twinpast_backbone_evidence.zip"\nMODEL_REVISION = "0d50680527681998e456c7b78950205bedd8a068"\n\n\ndef reproduce_windows(paths, records):\n    windows = {}\n    for branch in "ab":\n        left, right = (next(r for r in records if r["branch"] == branch and r["order"] == order)\n                       for order in base.ORDERS)\n        assert all(left[k] == right[k] for k in ("frame_indices", "frame_seconds", "frame_hashes"))\n        frames = base.decode(paths[branch], dict(source_frames=left["frame_indices"],\n                                                 decoded_rgb_sha256=left["frame_hashes"]))\n        windows[branch] = dict(frames=frames, indices=left["frame_indices"],\n                               seconds=left["frame_seconds"], hashes=left["frame_hashes"])\n    return windows\n\n\ndef run_pair(model, paths, case, smoke=False):\n    windows = reproduce_windows(paths, case["reference_rows"])\n    original = {(r["order"], r["arm"][-1]): r for r in case["original_prompt_rows"]}\n    reference = {}\n    for row in case["reference_rows"]:\n        key = row["order"], row["branch"]\n        assert row["prompt_sha256"] == original[key]["prompt_sha256"]\n        reference[key] = {**row, "label_token_ids": original[key]["label_token_ids"]}\n    rows, checks = history.evaluate(model, case["annotation"], windows, reference, preflight=smoke)\n    prior = history.outcome(case["reference_rows"], case["annotation"])\n    current = history.outcome(rows, case["annotation"])\n    return dict(pair_id=case["annotation"]["pair_id"], cohort=case["cohort"], prior=prior,\n                current=current, preflight=checks, rows=rows)\n\n\ndef summarize(blocks):\n    states = [x for x in blocks if x["cohort"] == "historical_state"]\n    controls = [x for x in blocks if x["cohort"] == "positive_control"]\n    robust_states = sum(x["current"]["robust"] for x in states)\n    robust_controls = sum(x["current"]["robust"] for x in controls)\n    return dict(pairs=len(blocks), scores=sum(len(x["rows"]) for x in blocks),\n                historical_state_robust=robust_states, historical_state_total=len(states),\n                positive_control_robust=robust_controls, positive_control_total=len(controls),\n                development_criterion_met=len(states) == 4 and len(controls) == 3\n                and robust_states >= 2 and robust_controls >= 2,\n                per_pair=[{k: v for k, v in x.items() if k not in {"preflight", "rows"}} for x in blocks])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    frozen = json.loads((provenance / "backbone_cases.json").read_bytes())\n    cases = {c["annotation"]["pair_id"]: c for c in frozen["cases"]}\n    assert len(cases) == 7 and frozen["model_revision"] == MODEL_REVISION == args.model.name\n    selection = [pair for pair in base.cases(bundle) if pair[0]["pair_id"] in cases]\n    assert {p["pair_id"] for p, _, _ in selection} == cases.keys()\n    # Start with the six-frame historical-state case, which exercises the longest input.\n    assert cases[selection[0][0]["pair_id"]]["cohort"] == "historical_state"\n    paths = history.acquire(SimpleNamespace(work=work, previous=args.previous, media_reuse=args.media_reuse),\n                            bundle, selection)\n    print("LOADING_PINNED_7B", args.model.name, flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda" and model.dtype == torch.float16\n    assert {str(p.device) for p in model.parameters()} == {"cuda:0"}, "7B must fit fully on GPU 0"\n    config = dict(model_revision=MODEL_REVISION, gpu=torch.cuda.get_device_name(0),\n                  torch=torch.__version__, cuda=torch.version.cuda, dtype=str(model.dtype),\n                  video_processor=processor.video_processor.to_dict(),\n                  runtime_source_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()},\n                  source_history_only_archive=frozen["history_only_archive_sha256"])\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    base.atomic(work / "exports" / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, (plan, _, _) in enumerate(selection):\n            pair = plan["pair_id"]\n            checkpoint = checkpoints / f"{pair}.json"\n            saved = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if saved is not None:\n                assert saved["identity"] == ident and saved["block"]["pair_id"] == pair\n            if i and saved is not None:\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i + 1}/7 {pair}", flush=True)\n                continue\n            print(f"PAIR_RUNNING {i + 1}/7 {pair}", flush=True)\n            torch.cuda.reset_peak_memory_stats()\n            block = run_pair(model, paths[pair], cases[pair], smoke=i == 0)\n            block["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()\n            if i == 0:\n                base.atomic(work / "exports" / "preflight.json",\n                            dict(status="pass", pair_id=pair, checks=block["preflight"],\n                                 peak_gpu_allocated_bytes=block["peak_gpu_allocated_bytes"]))\n                print("TWINPAST_BACKBONE_PREFLIGHT_PASS", flush=True)\n            base.atomic(checkpoint, dict(identity=ident, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i + 1}/7 {pair} robust={block[\'current\'][\'robust\']}", flush=True)\n    report = summarize(blocks)\n    assert report["pairs"] == 7 and report["scores"] == 28\n    base.atomic(work / "exports" / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=7, scores=28))\n    print("TWINPAST_BACKBONE_COMPLETE", json.dumps({k: v for k, v in report.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/automatic_runner.py': '"""Run a frozen, annotation-blind retrieval and answer-correction screen."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport backbone_runner as backbone\n\nbase = backbone.base\nARTIFACT = "HERMES_twinpast_automatic_evidence.zip"\nMAX_CANDIDATES = 13\nMAX_HISTORY = 6\nMIN_LIFT = 0.5\nMETHODS = ("recent", "latest", "uniform", "relevance", "margin_gate", "history_control_gate")\n\n\ndef spread(count, budget):\n    if count <= budget:\n        return list(range(count))\n    return [i * (count - 1) // (budget - 1) for i in range(budget)]\n\n\ndef sample_prefix(path, query):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    grid = list(range(0, math.ceil(query), 2))\n    assert len(grid) >= 8 and grid[-1] < query and grid[-1] <= times[-1, 1] + 1e-6\n    older_grid, recent_grid = grid[:-4], grid[-4:]\n    picks = [older_grid[i] for i in spread(len(older_grid), MAX_CANDIDATES)] + recent_grid\n    indices = np.searchsorted(times[:, 0], picks, side="right") - 1\n    sampled_s = times[indices, 0]\n    assert indices[0] >= 0 and np.all(np.diff(indices) > 0)\n    assert np.all(sampled_s <= picks) and sampled_s[-1] < query\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=picks, indices=indices.tolist(), timestamps_s=sampled_s.tolist(),\n                        hashes=[base.sha(f.tobytes()) for f in frames])\n\n\ndef encode_frames(model, frames):\n    """Vision encoding is frame-local. Limit the encoder batch to four frames."""\n    bank = []\n    for start in range(0, len(frames), 4):\n        batch = frames[start:start + 4]\n        encoded = base.features(model, batch)\n        bank.extend(encoded.split(encoded.shape[1] // len(batch), dim=1))\n    assert len(bank) == len(frames)\n    return bank\n\n\ndef best(scores):\n    return max(range(len(scores)), key=scores.__getitem__)\n\n\ndef answer_policies(recent, direct, uniform, same_history):\n    initial, alternative = best(recent), best(direct)\n    margin = direct[alternative] - direct[initial]\n    lift = margin - (recent[alternative] - recent[initial])\n    specificity = margin - (uniform[alternative] - uniform[initial])\n    accept = alternative != initial and lift >= MIN_LIFT\n    controlled = accept and not same_history and specificity >= MIN_LIFT\n    return dict(relevance=alternative, margin_gate=alternative if accept else initial,\n                history_control_gate=alternative if controlled else initial,\n                scores=dict(history_margin=margin, lift_from_recent=lift, lift_from_uniform=specificity))\n\n\ndef branch_policy(model, path, query, question, options):\n    """Infer for one branch with no evidence labels, gold, or counterpart data."""\n    import torch\n\n    started = time.perf_counter()\n    frames, sample = sample_prefix(path, query)\n    bank = encode_frames(model, frames)\n    history_count = len(bank) - 4\n    size = min(MAX_HISTORY, history_count)\n    recent = list(range(history_count, len(bank)))\n    uniform = spread(history_count, size)\n    latest = list(range(history_count - size, history_count))\n    embedding = model.language_model.get_input_embeddings()\n    system = embedding(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    calls = []\n    logits_cache = {}\n\n    def score(prompt, window, phase, order):\n        ids, option_ids, perm, digest = prompt\n        key = digest, tuple(window)\n        if key not in logits_cache:\n            visual = torch.cat([bank[i] for i in window], dim=1)\n            tick = time.perf_counter()\n            logits, length = base.score(model, system, ids, option_ids, visual)\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, logits.tolist(), strict=True):\n                values[i] = value\n            logits_cache[key] = values\n            calls.append(dict(phase=phase, order=order, slots=window, forward_s=time.perf_counter()-tick,\n                              visual_tokens=visual.shape[1], lm_tokens=length, prompt_sha256=digest))\n        return logits_cache[key]\n\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {o: base.make_prompt(model, probe, ["Yes", "No"], o, True) for o in base.ORDERS}\n    candidates = []\n    relevance = [-math.inf] * history_count\n    for slot in range(history_count - 1):\n        window = [slot, slot + 1]\n        probe_scores = {o: score(prompts[o], window, "retrieval", o) for o in base.ORDERS}\n        average = sum(v[0] - v[1] for v in probe_scores.values()) / len(probe_scores)\n        candidates.append(dict(slots=window, yes_no=probe_scores, relevance=average))\n        for i in window:\n            relevance[i] = max(relevance[i], average)\n    ranked = sorted(sorted(range(history_count), key=lambda i: (-relevance[i], i))[:size])\n    assert len(set(ranked)) == len(uniform) == len(latest) == size\n    selection = {"recent": recent, "latest": latest + recent,\n                 "uniform": uniform + recent, "relevance": ranked + recent}\n    answers = {}\n    for order in base.ORDERS:\n        prompt = base.make_prompt(model, question, options, order, True)\n        vectors = {arm: score(prompt, slots, arm, order) for arm, slots in selection.items()}\n        decision = answer_policies(vectors["recent"], vectors["relevance"], vectors["uniform"], ranked == uniform)\n        choices = {arm: best(values) for arm, values in vectors.items()}\n        choices.update({k: decision[k] for k in ("margin_gate", "history_control_gate")})\n        answers[order] = dict(logits=vectors, predictions={arm: base.LETTERS[i] for arm, i in choices.items()},\n                              margins=decision["scores"], prompt_sha256=prompt[3])\n    return dict(sample=sample, history_budget=size, candidates=candidates, selection=selection, answers=answers,\n                calls=calls, wall_s=time.perf_counter()-started)\n\n\ndef summarize(blocks, targets):\n    report = {}\n    opened = {arm: 0 for arm in ("margin_gate", "history_control_gate")}\n    gate_wins, gate_losses = 0, 0\n    for arm in METHODS:\n        robust, wins, losses = [], 0, 0\n        for block in blocks:\n            pair_id = block["pair_id"]\n            success = True\n            for order in base.ORDERS:\n                for branch in "ab":\n                    values = block["branches"][branch]["answers"][order]["predictions"]\n                    gold = targets[pair_id][f"answer_{branch}"]\n                    correct = values[arm] == gold\n                    reference = values["recent"] == gold\n                    wins += correct and not reference\n                    losses += reference and not correct\n                    success = success and correct\n                    if arm in opened:\n                        opened[arm] += values[arm] != values["recent"]\n                    if arm == "history_control_gate":\n                        direct_correct = values["relevance"] == gold\n                        gate_wins += correct and not direct_correct\n                        gate_losses += direct_correct and not correct\n            if success:\n                robust.append(pair_id)\n        report[arm] = dict(robust_pairs=robust, recoveries_vs_recent=wins, regressions_vs_recent=losses)\n    return dict(pairs=len(blocks), branch_order_answers=len(blocks)*4, methods=report, overrides=opened,\n                controlled_gate_vs_same_frames_direct=dict(recoveries=gate_wins, regressions=gate_losses),\n                total_forward_calls=sum(len(v["calls"]) for b in blocks for v in b["branches"].values()))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "automatic_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    cases = {c["pair_id"]: c for c in spec["cases"]}\n    assert len(cases) == 7\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, torch=torch.__version__, cuda=torch.version.cuda,\n                  gpu=torch.cuda.get_device_name(0), min_lift=MIN_LIFT, max_candidates=MAX_CANDIDATES,\n                  max_history=MAX_HISTORY, source_archive_sha256=spec["fixed_recent_archive_sha256"],\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    base.atomic(exports / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(spec["cases"]):\n            pair_id = case["pair_id"]\n            checkpoint = checkpoints / f"{pair_id}.json"\n            if checkpoint.is_file():\n                saved = json.loads(checkpoint.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair_id\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i+1}/7 {pair_id}", flush=True)\n                continue\n            torch.cuda.reset_peak_memory_stats()\n            branches = {b: branch_policy(model, paths[pair_id][b], case["query_s"], case["question"], case["options"])\n                        for b in "ab"}\n            left, right = branches["a"], branches["b"]\n            assert left["sample"]["grid_s"] == right["sample"]["grid_s"]\n            assert left["sample"]["hashes"][-4:] == right["sample"]["hashes"][-4:] == case["recent_hashes"]\n            assert left["history_budget"] == right["history_budget"]\n            for order in base.ORDERS:\n                assert left["answers"][order]["predictions"]["recent"] == right["answers"][order]["predictions"]["recent"]\n            block = dict(pair_id=pair_id, branches=branches,\n                         peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n            base.atomic(checkpoint, dict(identity=identity, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i+1}/7 {pair_id}", flush=True)\n    # Ground truth is loaded only after all branch predictions have been written.\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert targets.keys() == cases.keys()\n    report = summarize(blocks, targets)\n    coverage = []\n    for block in blocks:\n        pair_id = block["pair_id"]\n        for branch, trace in block["branches"].items():\n            intervals = targets[pair_id]["evidence_intervals"][branch.upper()]\n            timestamps = trace["sample"]["timestamps_s"]\n            inside = [i for i, t in enumerate(timestamps[:-4]) if any(lo <= t <= hi for lo, hi in intervals)]\n            coverage.append(dict(pair_id=pair_id, branch=branch, candidate_slots_in_evidence=inside,\n                                 selected_slots_in_evidence={name: sorted(set(inside) & set(trace["selection"][name]))\n                                                             for name in ("latest", "uniform", "relevance")}))\n    base.atomic(exports / "postrun_evidence_overlap.json", coverage)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_AUTOMATIC_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/temporal_runner.py': '"""Evaluate endpoint-preserving retrieval on a new TwinPast development screen."""\nimport argparse\nfrom collections import Counter\nfrom itertools import combinations\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport automatic_runner as auto\n\nbase = auto.base\nbackbone = auto.backbone\nARTIFACT = "HERMES_twinpast_temporal_windows_evidence.zip"\nMETHODS = ("recent", "uniform", "individual", "endpoints")\nWINDOW_SPAN = 3\nWINDOW_CAP = 12\n\n\ndef decode_grid(path, grid):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    if grid is None:\n        # Source-screening frames never enter the policy.\n        grid = range(0, math.ceil(times[-1, 1]), 2)\n    grid = np.asarray(list(grid))\n    if grid[-1] > times[-1, 1] + 1e-6:\n        raise ValueError("Clip ends before the frozen grid")\n    indices = np.searchsorted(times[:, 0], grid, side="right") - 1\n    if indices[0] < 0 or not np.all(np.diff(indices) > 0):\n        raise ValueError("Clip does not contain distinct frames on the frozen grid")\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=grid.tolist(), indices=indices.tolist(), timestamps_s=times[indices, 0].tolist(),\n                        hashes=[base.sha(frame.tobytes()) for frame in frames])\n\n\ndef fingerprints(path):\n    import numpy as np\n    from PIL import Image\n\n    frames, trace = decode_grid(path, None)\n    signatures = []\n    for frame, sha in zip(frames, trace["hashes"], strict=True):\n        image = Image.fromarray(frame)\n        gray = np.asarray(image.convert("L").resize((9, 8)))\n        bits = int.from_bytes(np.packbits((gray[:, 1:] > gray[:, :-1]).reshape(-1)).tobytes(), "big")\n        rgb = np.asarray(image.resize((4, 4)), dtype=int).reshape(-1)\n        if gray.std() >= 8:\n            signatures.append((sha, bits, rgb))\n    return signatures\n\n\ndef shared_frames(a, b):\n    """Flag repeat footage on at least three distinct sampled frame matches."""\n    import numpy as np\n\n    matches = set()\n    for sha_a, hash_a, rgb_a in a:\n        for sha_b, hash_b, rgb_b in b:\n            if sha_a == sha_b or ((hash_a ^ hash_b).bit_count() <= 4 and np.abs(rgb_a - rgb_b).mean() <= 6):\n                matches.add(sha_b)\n                break\n    return len(matches)\n\n\ndef screen_pairs(cases, excluded, paths, max_pairs):\n    signature_bank = {pair: {b: fingerprints(paths[pair][b]) for b in "ab"} for pair in excluded}\n    decisions, accepted = [], []\n    for case in cases:\n        pair = case["pair_id"]\n        if len(accepted) == max_pairs:\n            break\n        grid = list(range(math.ceil(case["query_s"])))\n        try:\n            recent = {b: decode_grid(paths[pair][b], grid[-4:])[1]["hashes"] for b in "ab"}\n        except ValueError as exc:\n            decisions.append(dict(pair_id=pair, disposition="invalid_grid", detail=str(exc)))\n            continue\n        if recent["a"] != recent["b"]:\n            decisions.append(dict(pair_id=pair, disposition="recent_context_mismatch"))\n            continue\n        candidate = {b: fingerprints(paths[pair][b]) for b in "ab"}\n        overlaps = []\n        for reference, signature in signature_bank.items():\n            matches = max(shared_frames(candidate[b], signature[c]) for b in "ab" for c in "ab")\n            if matches >= 3:\n                overlaps.append(dict(reference=reference, sampled_matches=matches))\n        if overlaps:\n            decisions.append(dict(pair_id=pair, disposition="possible_media_overlap", matches=overlaps))\n            continue\n        signature_bank[pair] = candidate\n        accepted.append(case)\n        decisions.append(dict(pair_id=pair, disposition="accepted", recent_hashes=recent["a"]))\n    return accepted, decisions\n\n\ndef select_windows(windows):\n    order = sorted(windows, key=lambda row: row["slots"])\n    compatible = (group for group in combinations(order, 3)\n                  if all(left["slots"][-1] < right["slots"][0] for left, right in zip(group, group[1:])))\n    best = max(compatible, key=lambda group: sum(r["relevance"] for r in group))\n    return [i for r in best for i in r["slots"]]\n\n\ndef branch_policy(model, path, case):\n    import torch\n\n    tick = time.perf_counter()\n    frames, sample = decode_grid(path, range(math.ceil(case["query_s"])))\n    assert sample["timestamps_s"][-1] < case["query_s"]\n    bank = auto.encode_frames(model, frames)\n    num_history = len(frames) - 4\n    recent = list(range(num_history, len(frames)))\n    system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    log, cache = [], {}\n\n    def score(prompt, slots, phase, order):\n        ids, option_ids, permutation, prompt_hash = prompt\n        key = (prompt_hash, tuple(slots))\n        if key not in cache:\n            visual = torch.cat([bank[i] for i in slots], dim=1)\n            started = time.perf_counter()\n            values, tokens = base.score(model, system, ids, option_ids, visual)\n            canonical = [0.0] * len(permutation)\n            for i, value in zip(permutation, values.tolist(), strict=True):\n                canonical[i] = value\n            cache[key] = canonical\n            log.append(dict(phase=phase, order=order, slots=slots, forward_s=time.perf_counter()-started,\n                            lm_tokens=tokens, visual_tokens=visual.shape[1]))\n        return cache[key]\n\n    question = case["question"]\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {order: base.make_prompt(model, probe, ["Yes", "No"], order, True) for order in base.ORDERS}\n    windows, individual_scores = [], {}\n    for start in auto.spread(num_history - WINDOW_SPAN, WINDOW_CAP):\n        endpoints = [start, start + WINDOW_SPAN]\n        vectors = {o: score(prompts[o], endpoints, "retrieval", o) for o in base.ORDERS}\n        value = sum(v[0] - v[1] for v in vectors.values()) / len(vectors)\n        windows.append(dict(slots=endpoints, relevance=value, yes_no=vectors))\n        for i in endpoints:\n            individual_scores[i] = max(value, individual_scores.get(i, -math.inf))\n    selection = dict(recent=recent, uniform=auto.spread(num_history, 6) + recent,\n                     individual=sorted(sorted(individual_scores, key=lambda i: (-individual_scores[i], i))[:6]) + recent,\n                     endpoints=select_windows(windows) + recent)\n    assert len(selection["recent"]) == 4 and all(len(selection[m]) == 10 for m in METHODS[1:])\n    assert all(slots == sorted(set(slots)) for slots in selection.values())\n    answers = {}\n    for o in base.ORDERS:\n        prompt = base.make_prompt(model, question, case["options"], o, True)\n        vectors = {m: score(prompt, selection[m], m, o) for m in METHODS}\n        answers[o] = dict(logits=vectors, predictions={m: base.LETTERS[auto.best(v)] for m, v in vectors.items()})\n    return dict(sample=sample, windows=windows, selection=selection, answers=answers,\n                calls=log, wall_s=time.perf_counter()-tick)\n\n\ndef evaluate(blocks, targets):\n    methods = {}\n    for method in METHODS:\n        pairs, correct = [], 0\n        for block in blocks:\n            p = block["pair_id"]\n            wins = [block["branches"][b]["answers"][o]["predictions"][method] == targets[p][f"answer_{b}"]\n                    for b in "ab" for o in base.ORDERS]\n            correct += sum(wins)\n            if all(wins):\n                pairs.append(p)\n        methods[method] = dict(robust_pairs=pairs, correct_branch_orders=correct)\n    endpoints = set(methods["endpoints"]["robust_pairs"])\n    matched = {m: dict(recoveries=sorted(endpoints - set(methods[m]["robust_pairs"])),\n                       regressions=sorted(set(methods[m]["robust_pairs"]) - endpoints))\n               for m in ("uniform", "individual")}\n    calls = Counter(call["phase"] for p in blocks for b in p["branches"].values() for call in b["calls"])\n    return dict(pairs=len(blocks), methods=methods, matched_pair_changes=matched,\n                endpoints_more_robust_than_both_dense_baselines=len(methods["endpoints"]["robust_pairs"]) >\n                max(len(methods[m]["robust_pairs"]) for m in ("uniform", "individual")),\n                forward_calls=sum(calls.values()), forward_calls_by_phase=dict(calls))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "temporal_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    cases, screening = screen_pairs(spec["cases"], spec["excluded"], paths, spec["expected_max_pairs"])\n    exports = work / "exports"\n    base.atomic(exports / "source_screen.json", dict(decisions=screening, source_status=spec["source_status"]))\n    if not cases:\n        raise RuntimeError("No pairs passed the shared-input and media-overlap screen")\n    print(f"SOURCE_SCREEN_PASS {len(cases)}/{spec[\'expected_max_pairs\']} pairs", flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, cuda=torch.version.cuda, torch=torch.__version__,\n                  gpu=torch.cuda.get_device_name(0), accepted=[c["pair_id"] for c in cases],\n                  window_span=WINDOW_SPAN, window_cap=WINDOW_CAP,\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    base.atomic(exports / "run_config.json", config)\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(cases):\n            pair = case["pair_id"]\n            path = checkpoints / f"{pair}.json"\n            if path.is_file():\n                saved = json.loads(path.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair\n                block = saved["block"]\n                print(f"PAIR_REUSED {i+1}/{len(cases)} {pair}", flush=True)\n            else:\n                torch.cuda.reset_peak_memory_stats()\n                branches = {b: branch_policy(model, paths[pair][b], case) for b in "ab"}\n                assert branches["a"]["sample"]["hashes"][-4:] == branches["b"]["sample"]["hashes"][-4:]\n                assert all(branches["a"]["answers"][o]["predictions"]["recent"] ==\n                           branches["b"]["answers"][o]["predictions"]["recent"] for o in base.ORDERS)\n                block = dict(pair_id=pair, branches=branches, peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n                base.atomic(path, dict(identity=identity, block=block))\n                print(f"PAIR_DONE {i+1}/{len(cases)} {pair}", flush=True)\n            blocks.append(block)\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    report = evaluate(blocks, targets)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_TEMPORAL_WINDOWS_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/readout_runner.py': '"""Compare option-blind observation and direct readouts on archived TwinPast frames."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nfrom pathlib import Path\nimport time\n\nimport automatic_runner as auto\n\nbase = auto.base\nARTIFACT = "HERMES_twinpast_readout_evidence.zip"\nFRAME_SETS = ("coarse_uniform", "state_change")\nREPLAY_ATOL = 0.05\nNOTE_TOKENS = 24\n\n\ndef note_prompt(question):\n    return ("Describe only the visible state of the object or event named in the question. "\n            "Use the requested moment in the video. If the state is unclear, write \'unclear\'. "\n            "Write one short observation.\\nQuestion: " + question)\n\n\ndef observe(model, system, visual, question):\n    import torch\n\n    tokenizer = model.processor.tokenizer\n    text = model.get_prompt(note_prompt(question))\n    tokens = tokenizer.encode(text, add_special_tokens=False)\n    ids = torch.tensor([tokens], device=model.device)\n    lm = model.language_model\n    prompt = torch.cat((system, visual, lm.get_input_embeddings()(ids)), dim=1)\n    stops = lm.generation_config.eos_token_id\n    stops = set(stops if isinstance(stops, (list, tuple)) else [stops]) | {tokenizer.eos_token_id}\n    generated, consumed = [], prompt.shape[1]\n    start = time.perf_counter()\n    output = lm(inputs_embeds=prompt, use_cache=True, return_dict=True)\n    forwards = 1\n    for step in range(NOTE_TOKENS):\n        last = output.logits[:, -1, :]\n        if not torch.isfinite(last).all():\n            raise ValueError("Non-finite observation logits")\n        token = last.argmax(dim=-1, keepdim=True)\n        if token.item() in stops:\n            break\n        generated.append(token.item())\n        if step + 1 < NOTE_TOKENS:\n            output = lm(input_ids=token, past_key_values=output.past_key_values, use_cache=True, return_dict=True)\n            consumed += 1\n            forwards += 1\n    note = " ".join(tokenizer.decode(generated, skip_special_tokens=True).split()) or "unclear"\n    return dict(text=note, generated_token_ids=generated, forward_calls=forwards, input_tokens=consumed,\n                prompt_sha256=base.sha(text.encode("utf-8")), forward_s=time.perf_counter()-start)\n\n\ndef choose(model, system, visual, question, options, order):\n    prompt, option_ids, permutation, prompt_hash = base.make_prompt(model, question, options, order, True)\n    start = time.perf_counter()\n    scores, tokens = base.score(model, system, prompt, option_ids, visual)\n    values = [0.0] * len(permutation)\n    for i, score in zip(permutation, scores.tolist(), strict=True):\n        values[i] = score\n    return dict(logits=values, prediction=base.LETTERS[auto.best(values)], lm_tokens=tokens,\n                visual_tokens=visual.shape[1], prompt_sha256=prompt_hash, forward_s=time.perf_counter()-start)\n\n\ndef fixed_inputs(model, path, branch):\n    import torch\n\n    coarse, fine = branch["coarse"], branch["rgb"]\n    frames = base.decode(path, dict(source_frames=coarse["indices"], decoded_rgb_sha256=coarse["hashes"]))\n    rgb = base.decode(path, dict(source_frames=fine["indices"], decoded_rgb_sha256=fine["hashes"]))\n    # Preserve the original vision encoder batch boundaries for numerical replay.\n    bank = auto.encode_frames(model, frames)\n    recent = bank[-4:]\n    history = [bank[i] for i in coarse["slots"]]\n    return dict(coarse_uniform=torch.cat(history+recent, dim=1),\n                state_change=torch.cat(auto.encode_frames(model, rgb)+recent, dim=1))\n\n\ndef acquire(args, spec):\n    def clip(case, side):\n        receipt = case["media"][side]\n        relative = receipt["file"]\n        cached = args.previous / "media" / relative\n        if cached.is_file() and cached.stat().st_size == receipt["bytes"] and base.digest(cached) == receipt["sha256"]:\n            print("MEDIA_REUSED", relative, flush=True)\n            return cached\n        target = args.work / "media" / relative\n        base.fetch(spec["media_source"], relative, target, receipt["sha256"], receipt["bytes"])\n        return target\n\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        jobs = {(case["pair_id"], side): pool.submit(clip, case, side)\n                for case in spec["cases"] for side in "ab"}\n        return {key: job.result() for key, job in jobs.items()}\n\n\ndef replay(case, system, model, paths):\n    pair = case["pair_id"]\n    records, visuals, failures = {}, {}, []\n    for side in "ab":\n        archived = case["branches"][side]\n        for frame_set, visual in fixed_inputs(model, paths[pair, side], archived).items():\n            visuals[side, frame_set] = visual.cpu()\n            for order in base.ORDERS:\n                result = choose(model, system, visual, case["question"], case["options"], order)\n                expected = archived["direct"][frame_set][order]\n                delta = max(abs(a-b) for a, b in zip(result["logits"], expected["logits"], strict=True))\n                result["replay_max_abs"] = delta\n                if delta > REPLAY_ATOL or any(result[k] != expected[k] for k in\n                                               ("prediction", "prompt_sha256", "lm_tokens", "visual_tokens")):\n                    failures.append(dict(pair_id=pair, branch=side, frame_set=frame_set, order=order,\n                                         archived=expected, replay=result))\n                records[side, frame_set, order] = result\n    return records, visuals, failures\n\n\ndef compare(case, model, system, packed):\n    direct, visuals, _ = packed\n    rows = []\n    for (side, frame_set), cpu_visual in visuals.items():\n        visual = cpu_visual.to(model.device)\n        note = observe(model, system, visual, case["question"])\n        with_note = case["question"] + "\\nVisual observation from the first pass (may be incomplete): " + note["text"]\n        results = {order: dict(direct=direct[side, frame_set, order],\n                               observation=choose(model, system, visual, with_note, case["options"], order))\n                   for order in base.ORDERS}\n        rows.append(dict(branch=side, frame_set=frame_set, observation=note, orders=results))\n    return dict(pair_id=case["pair_id"], rows=rows)\n\n\ndef summarize(blocks, targets):\n    summary = {}\n    for frame_set in FRAME_SETS:\n        methods = {m: dict(robust_pairs=[], correct_branch_orders=0, forward_calls=0, input_tokens=0, forward_s=0.0)\n                   for m in ("direct", "observation")}\n        max_replay = 0.0\n        for block in blocks:\n            count = dict.fromkeys(methods, 0)\n            pair = block["pair_id"]\n            for branch in (r for r in block["rows"] if r["frame_set"] == frame_set):\n                note = branch["observation"]\n                for field in ("forward_calls", "input_tokens", "forward_s"):\n                    methods["observation"][field] += note[field]\n                for result in branch["orders"].values():\n                    max_replay = max(max_replay, result["direct"]["replay_max_abs"])\n                    for method, row in result.items():\n                        count[method] += row["prediction"] == targets[pair][f"answer_{branch[\'branch\']}"]\n                        methods[method]["forward_calls"] += 1\n                        methods[method]["input_tokens"] += row["lm_tokens"]\n                        methods[method]["forward_s"] += row["forward_s"]\n            for method, correct in count.items():\n                methods[method]["correct_branch_orders"] += correct\n                if correct == 4:\n                    methods[method]["robust_pairs"].append(pair)\n        direct, observed = (set(methods[m]["robust_pairs"]) for m in methods)\n        summary[frame_set] = dict(methods=methods, recoveries=sorted(observed-direct),\n                                  regressions=sorted(direct-observed), max_replay_abs=max_replay)\n    primary = summary["coarse_uniform"]\n    score_count = sum(len(result) for block in blocks for row in block["rows"] for result in row["orders"].values())\n    return dict(pairs=len(blocks), choice_scores=score_count, frame_sets=summary,\n                development_criterion_met=len(blocks) == 8 and bool(primary["recoveries"]) and not primary["regressions"])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "readout_cases.json").read_bytes())\n    if args.model.name != spec["model_revision"]:\n        raise ValueError("Model revision differs from the archived run")\n    paths = acquire(args, spec)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    exports = work / "exports"\n    base.atomic(exports / "run_config.json", dict(model_revision=args.model.name, torch=torch.__version__,\n                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(), replay_atol=REPLAY_ATOL,\n                observation_max_new_tokens=NOTE_TOKENS, source_archive_sha256=spec["source_archive_sha256"]))\n    with torch.inference_mode():\n        system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n        replays = []\n        print("REPLAY_STARTED", flush=True)\n        for i, case in enumerate(spec["cases"], 1):\n            replays.append(replay(case, system, model, paths))\n            print(f"REPLAY_PAIR {i}/8 {case[\'pair_id\']}", flush=True)\n        failures = [failure for _, _, checks in replays for failure in checks]\n        base.atomic(exports / "replay_gate.json", dict(scores=sum(len(r) for r, _, _ in replays),\n                    max_abs=max(x["replay_max_abs"] for r, _, _ in replays for x in r.values()), failures=failures))\n        if failures:\n            raise RuntimeError("Archived direct replay failed. See exports/replay_gate.json.")\n        print("REPLAY_PASS", flush=True)\n        blocks = []\n        for i, case in enumerate(spec["cases"], 1):\n            block = compare(case, model, system, replays[i-1])\n            base.atomic(work / "checkpoints" / f"{case[\'pair_id\']}.json", block)\n            blocks.append(block)\n            print(f"READOUT_PAIR {i}/8 {case[\'pair_id\']}", flush=True)\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert set(targets) == {b["pair_id"] for b in blocks}\n    report = summarize(blocks, targets)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_READOUT_COMPLETE", json.dumps(report), flush=True)\n\n\ndef main():\n    from filelock import FileLock\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    work = args.work\n    with FileLock(str(work / "run.lock"), timeout=0):\n        for name in ("exports", "checkpoints"):\n            (work / name).mkdir(exist_ok=True)\n        # A rerun must not package scores from an earlier incomplete attempt.\n        for case in json.loads((work / "provenance/readout_cases.json").read_bytes())["cases"]:\n            (work / "checkpoints" / f"{case[\'pair_id\']}.json").unlink(missing_ok=True)\n        for name in ("summary.json", "replay_gate.json", "run_config.json"):\n            (work / "exports" / name).unlink(missing_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/droid_natural_manifest.json': '{\n  "protocol": "DROID natural state history capability v1",\n  "dataset": "DROID raw 1.0.1",\n  "dataset_reference": "https://droid-dataset.github.io/",\n  "dataset_license": "CC BY 4.0",\n  "frame_zip_sha256": "5bcfb613c5eb2cd8a40e306d94a79a3e77258ba162b47ddbb025ffc2a56f8b49",\n  "recent_donor": {\n    "source_episode": "CLVR+236539bc+2023-06-06-01h-09m-11s",\n    "source_site": "CLVR",\n    "camera": "ext2",\n    "source_frames": [\n      15,\n      75,\n      135,\n      195\n    ],\n    "video_source": {\n      "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/CLVR/success/2023-06-06/Tue_Jun__6_01%3A09%3A11_2023/recordings/MP4/20655732.mp4",\n      "generation": "1710757175937043",\n      "bytes": 4077697,\n      "sha256": "cd6862a22e370820d929d8aa18f3999dc4b5d3d65fa287998c38ab78b269b5a0",\n      "frame_count": 298\n    }\n  },\n  "frame_receipts": {\n    "droid_natural_10bd743a70_a_00000.png": {\n      "png_sha256": "c40dcd205dfdc84678b00ae4def3837bc3bf892c2d326518368380ccbf7cc0c7",\n      "rgb_sha256": "9cae7f271288eea988b2d9747f5397c73886e223195482738e355d2dbb2eb455"\n    },\n    "droid_natural_10bd743a70_a_00077.png": {\n      "png_sha256": "9d2d9bf5ab51d5dd7d9ce3387d2e9347ddf78ec0ed4f35b57f1146dd2881acf3",\n      "rgb_sha256": "3c73da3d7090412b873fe9ee48a8d5c29f228cf78561788de577bad3df429cd7"\n    },\n    "droid_natural_10bd743a70_a_00154.png": {\n      "png_sha256": "5890316104b7fcb72e47702fb2a86c114f4d19440e094a9689e2d66fc5656bc6",\n      "rgb_sha256": "88973b702a8acb303b16b57e52600c653b447e8a468b486685ed169812eace43"\n    },\n    "droid_natural_10bd743a70_a_00232.png": {\n      "png_sha256": "784a33a8c6b63eb3d4d91dc246200749a4b504a3cdf32828fe61b70a92669e01",\n      "rgb_sha256": "9da2567884182a616b10f71074974bba7352eb700ac7eb2ec4679c9a1e64374f"\n    },\n    "droid_natural_10bd743a70_a_00309.png": {\n      "png_sha256": "eb1a20d09f180f85bd6de4bd002a04b85883be32c3add4d52d2d7d239ae00352",\n      "rgb_sha256": "c60f28e576699485fbee45de80cce28fc5af64a0cd0ce3a7d42f7dcaa8c38db4"\n    },\n    "droid_natural_10bd743a70_a_00358.png": {\n      "png_sha256": "efff16c14e8aef629c1264b1c5bfd5f5339556502f6c52384901a5e8031f60c0",\n      "rgb_sha256": "009e8977fd1f8b117c7b703a506d1a04dc842fdaf4b8811a5a2b8cce72dbe43a"\n    },\n    "droid_natural_10bd743a70_a_00372.png": {\n      "png_sha256": "75a33ac5bf97476258d10b5c0c4c0acc66d4a9b21cc928930e1de8063dab3b86",\n      "rgb_sha256": "9ac304b5618125fb87cc599f494bba2435fa84986f4619592e887460c29ea624"\n    },\n    "droid_natural_10bd743a70_a_00387.png": {\n      "png_sha256": "19bdd9e864469e353e49b14d43bdfaa38bd2d249feb3fc4683797d1269ef01d4",\n      "rgb_sha256": "1aac2f2260e929006821fe64d3268c3c99b93c969522075b890cc5bd4865b099"\n    },\n    "droid_natural_10bd743a70_b_00000.png": {\n      "png_sha256": "7022063be9422ee64814215568e4b9d5de604de4e8bcb408e7a9863e4aa7d18c",\n      "rgb_sha256": "8766bf61a7a3e2e54d359f0253869ca154c5bed18bdae2740d9931430acef535"\n    },\n    "droid_natural_10bd743a70_b_00055.png": {\n      "png_sha256": "6db741114bfd4a184a51c26e6a4e3762b58e7b790360caba6a8345523b98e591",\n      "rgb_sha256": "bee2f7670e4910b0581132320ba4561136b6090623ee2bdd33891739cb592775"\n    },\n    "droid_natural_10bd743a70_b_00110.png": {\n      "png_sha256": "ebe1f8641f814981d7ffaab3a64312d10a0e253aa58942da7b4825eea24125c9",\n      "rgb_sha256": "587cc06314220796732fcec5456411e2141e1daef1e466d7a81ee8489d7cdfd2"\n    },\n    "droid_natural_10bd743a70_b_00166.png": {\n      "png_sha256": "1844fe98ce7a9fc176509bef3a1afb719d6aea0236f35a40b6e34f821e07e4af",\n      "rgb_sha256": "6c6b11ac7a4a8e3fe6b46740a918581664bbd6ac327cc1923cd31685a7af7fe8"\n    },\n    "droid_natural_10bd743a70_b_00221.png": {\n      "png_sha256": "206e871f6ff30d2098f493828e34cfaf1ecee2850e079e6d8fead5a86360327d",\n      "rgb_sha256": "bba4a33f142259e76852dc02fba72e9c371a925dad61cbe5e14a07a5ca3925f4"\n    },\n    "droid_natural_10bd743a70_b_00262.png": {\n      "png_sha256": "a5bf3458ba10b38b7fea4180793823100eb0134d32ef94972d7493e414d8b5e5",\n      "rgb_sha256": "c9f231b917cdde0017e5ccb3d5093f0d94f15df2fcbb8aaf5d2ab0e656134998"\n    },\n    "droid_natural_10bd743a70_b_00269.png": {\n      "png_sha256": "208bdc06bfa835c41f80f6deb8e49d9777ee33b2b48feee22d2108f5e64d51eb",\n      "rgb_sha256": "83288ad9ec01e4342f1e9c721eb58c126f9d68e53c01a770f677c627ee6818c6"\n    },\n    "droid_natural_10bd743a70_b_00277.png": {\n      "png_sha256": "3a6137ef5b040122124c2d7c0dcb0a4dba7f59cb43f01807addd826b2b116f1b",\n      "rgb_sha256": "7915e7b1ee3138e4eb7dc44456df83fedd18a152089e5e112e70111edfb0bd34"\n    },\n    "droid_natural_19bf4a67ee_a_00000.png": {\n      "png_sha256": "debacb052771f58ea21515c68f72bc67af3b8ce92178ceb6781beed0cc977efd",\n      "rgb_sha256": "03e96cfaf5854212720759e6b3e269af51a2833ad1818bb7da4a26881cbd9b7e"\n    },\n    "droid_natural_19bf4a67ee_a_00036.png": {\n      "png_sha256": "41b550d205ebf14c08561a7d91750bc49b818997b9e306c56be89a9e29d1cc22",\n      "rgb_sha256": "1cc79407e83482563f97a8e8945451159089153b058596b702dd0b50c865a9a3"\n    },\n    "droid_natural_19bf4a67ee_a_00073.png": {\n      "png_sha256": "6f0e2d2e55bd2491a63289a570ea772c3342c3fec8726e6c2d507ed56ee9cc5a",\n      "rgb_sha256": "1a3084049b7c6ee28b6c4bb756fd5a11ff13aa00738dc2661ab547f1dbace6b8"\n    },\n    "droid_natural_19bf4a67ee_a_00110.png": {\n      "png_sha256": "891a2198514c7ddb45abd73a156b2ae1c8f40e01acdd45a83127d19b8542fd56",\n      "rgb_sha256": "2385a7cff31db995056771e4fe15e7e4a6c55016b36dcc177b24eea9b29b565a"\n    },\n    "droid_natural_19bf4a67ee_a_00147.png": {\n      "png_sha256": "d619d0f4f87029c411de358067d24e56c015fb3f187e7e5a5826f6740d2c1278",\n      "rgb_sha256": "fe21833d6f7c4337ae8d209fa8bf4bcb0959f7bec3ee3460709044cf9db9609c"\n    },\n    "droid_natural_19bf4a67ee_a_00162.png": {\n      "png_sha256": "d1e8c26faff85f4a6b3ab04d016102d13fb0da1b0b4d646570806c986e1e02ce",\n      "rgb_sha256": "090a1255bd93e7f5da90f75f228def4d15fc4d966289b29ef0b631c9d89ceb9b"\n    },\n    "droid_natural_19bf4a67ee_a_00173.png": {\n      "png_sha256": "9fc98d98e6afaefcf2d04b9264ebf0b01b9e6a8a10131073557d1131d802a22f",\n      "rgb_sha256": "8e987aa7ad1adf52f8405b3db2cb35af45e3a61ea49ba5523fa6c49d5918c81c"\n    },\n    "droid_natural_19bf4a67ee_a_00184.png": {\n      "png_sha256": "0421d613e6efa7fa05f2232790753de29365c1864ca49ebd9353887ffbe2ce3c",\n      "rgb_sha256": "fc04240b13efe4dc8f93fe327c44015b8bcf290fd42a370378fc3b84d7485cae"\n    },\n    "droid_natural_19bf4a67ee_b_00000.png": {\n      "png_sha256": "b0efeb4463a1a1908e6b0d95f1276e7fe2c558ee9e757c01802ed5d45f7f7896",\n      "rgb_sha256": "f1848d98397bd2d656bb222001b9f2a02e37f8bbf63677cc803f46049f2d1648"\n    },\n    "droid_natural_19bf4a67ee_b_00062.png": {\n      "png_sha256": "6a13adc9415abc3667554d73bc452161beaf566820a29a2877c03cde640cff0d",\n      "rgb_sha256": "6ee37b935ad061df5a237c000fe306b6bf68361618b9cf1edecfaca76f6755ab"\n    },\n    "droid_natural_19bf4a67ee_b_00125.png": {\n      "png_sha256": "6da9f23967d312bc9b4e5b84e3e203272c5c3528320bff24b14c20a084ab3483",\n      "rgb_sha256": "7096ebfb5e1eebfbc71b7d5446921ade4bc2cbd703cff0517895a7cc61e74fea"\n    },\n    "droid_natural_19bf4a67ee_b_00187.png": {\n      "png_sha256": "2983a72e0e95eaa2f16cb2e99608b79346a65561628e146ef5280977cb14d493",\n      "rgb_sha256": "affa452bcdf880c6ee77f4e7396a953417f22bada5431085f4bded12ae546488"\n    },\n    "droid_natural_19bf4a67ee_b_00250.png": {\n      "png_sha256": "3eef60b01149d24d4b6f2ea1ded300850a8727cec60be43e9da877b5d7a3e711",\n      "rgb_sha256": "931348dcbe5d1e58970f01ba8da70fc68024abe858b875eeaa5c836d5ba88512"\n    },\n    "droid_natural_19bf4a67ee_b_00273.png": {\n      "png_sha256": "c2d721dc09522945f103589157a3312d269b500950f3ea92ac76c08a992ee2a5",\n      "rgb_sha256": "5b37d7337dd6ad03f402620875659878360c1336bdb2d3ec3074e7302b9500d1"\n    },\n    "droid_natural_19bf4a67ee_b_00293.png": {\n      "png_sha256": "338236c75313aae9f0dd4f3958a273077129b6de1891b4c26abc6d1e7904cdfb",\n      "rgb_sha256": "f8498a618758fcc0b4e038bcafaf6beaea22695372b123807b215069d38bf8af"\n    },\n    "droid_natural_19bf4a67ee_b_00313.png": {\n      "png_sha256": "474e8673e45b02f833babcc99d85f89a8477d62c38521bfbc1453d0719175409",\n      "rgb_sha256": "557b6ac3ffd004804229a55b6eb0b1a02767ce2dead1e53c1e1096cd879ad1f7"\n    },\n    "droid_natural_29137e19c7_a_00000.png": {\n      "png_sha256": "ca31407621e458a5dd08c7efd5572869bb9f4abee4ad67df71ddacb021cc1647",\n      "rgb_sha256": "b4fb7547434a3bab46eac6a5be94eaf227753f19173cf3ed223e69ed59e1d048"\n    },\n    "droid_natural_29137e19c7_a_00027.png": {\n      "png_sha256": "dfec0b22a96c9a87cc3187b5cff6a5f54e189c4d61c12e7d2945ffd5b9ae2efc",\n      "rgb_sha256": "3f48f342b8a284b81939b539df1fa96c2298a465e4468ba8d89ae7a09ed209e2"\n    },\n    "droid_natural_29137e19c7_a_00054.png": {\n      "png_sha256": "5299b74c8118f2f0dc73d5b6d1b6d787541163a90fad6ced0730efc61476e4ae",\n      "rgb_sha256": "d68e0c97147844e1a261e1b2e81ab75c4e73bd37f34798d380b36ceb25508b85"\n    },\n    "droid_natural_29137e19c7_a_00080.png": {\n      "png_sha256": "1615b37bad86dcbd5f4f5596b69ccc6ce1b289587a93b8ede836c1f2739b2373",\n      "rgb_sha256": "12d4eb8dbec808715c0dab84d9221c664e1c317a93e0c6022028c48c9d178d09"\n    },\n    "droid_natural_29137e19c7_a_00108.png": {\n      "png_sha256": "96bbfd509f43af7d3633a59a56584507627e68fe327f960a12f90e8aec605ebf",\n      "rgb_sha256": "08cf5c5bb858332b68cf96817e1c438278135c8d3b524e56763af0a7cd3b5249"\n    },\n    "droid_natural_29137e19c7_a_00131.png": {\n      "png_sha256": "a46588ca6b9c50a8f7a964dbf468dad50759dde3a0f56729c06a4af179da298a",\n      "rgb_sha256": "bf29320865420438dc3a54e2c4f0ae523ee03c99464fb896329fdb544389dfdb"\n    },\n    "droid_natural_29137e19c7_a_00133.png": {\n      "png_sha256": "c186cada9fcdce37792edbd8d90e1349641764741884734ed830abae0f0cdca7",\n      "rgb_sha256": "1bd8fce85f7157f4e74bed7006e2d8d5f5ce02fc060db51d70ccce6f54ebfeaa"\n    },\n    "droid_natural_29137e19c7_a_00135.png": {\n      "png_sha256": "70d9bafb7a7dd414cdab1572f88ceddad01c9803870ec405de8e7e04163be761",\n      "rgb_sha256": "31ccc8ca1510b9c5cf2ec7a14239fa28a4a1c0d20c6951b992d0738b7e13ced9"\n    },\n    "droid_natural_29137e19c7_b_00000.png": {\n      "png_sha256": "c01a7861a3fa580cdb9e6c8c3a2589cca7cfed92a71e179e9958ffd60e45f37d",\n      "rgb_sha256": "eede46b1066b617b26368eb7fff9ec58f2d5eac7142be9deace2c6b5daa43efa"\n    },\n    "droid_natural_29137e19c7_b_00041.png": {\n      "png_sha256": "b46d08c602aa6b8e1c376f18ae4634a8a854a31b1dd00b7adbaa6352bdf5fa5d",\n      "rgb_sha256": "11b45e94603c7c77748673d155964eeb5344c04462a514a4208a66f6c406c332"\n    },\n    "droid_natural_29137e19c7_b_00082.png": {\n      "png_sha256": "e915422fb15019ba905ecf14f196a235b397920a3bcb81cdb06056bd5e0f6b2f",\n      "rgb_sha256": "50b52e151062cfef792a246c35c2e159b9cf33e7c91eced996485d2439e28ca1"\n    },\n    "droid_natural_29137e19c7_b_00123.png": {\n      "png_sha256": "e8ded427979f5b1a34eec995fe286bbaef8a59140344bbd4dc3a49b8244232cc",\n      "rgb_sha256": "e28c94e20a0861b3ff7e2e661c292adca3eb44f5dadde9e18bc0904f4af3f184"\n    },\n    "droid_natural_29137e19c7_b_00164.png": {\n      "png_sha256": "65d6f2e35dceb2335970211d0c4f4beae28245e4453ad0b7f818be9feda7ede1",\n      "rgb_sha256": "bb37e534aa4efb88a147f5667022ef42800235977a4224829d83abe663b6fbe0"\n    },\n    "droid_natural_29137e19c7_b_00179.png": {\n      "png_sha256": "782339befc2d1c3cfe862df2849b551ac9151ef95d9099902703afa12f4a32e3",\n      "rgb_sha256": "0ddc8a2455eac4a57aa79cb9626dff020f94e56e2dcee32906f1f8d9ae2ec158"\n    },\n    "droid_natural_29137e19c7_b_00192.png": {\n      "png_sha256": "d037b3b1650fa8ed98a47dbcae267d65508a8c15b1a7b7d68cb2db70f911efa9",\n      "rgb_sha256": "bb1f2c704f7212f328945bc52d7e028985a85bd7775f30ba08833e04ef0ca22f"\n    },\n    "droid_natural_29137e19c7_b_00205.png": {\n      "png_sha256": "a164585443a01aa26f239ee529fb48254262d131d9a8e18dbee8976ed9ddd50a",\n      "rgb_sha256": "835fa93fce76e5fd978dc5efc5392a71b07ae2ed12e7f8da1af1f90eda7e6985"\n    },\n    "droid_natural_6d74030dc0_a_00000.png": {\n      "png_sha256": "bb15ff714e5d3b54497be4c24f6371dc36971708106c67e671a603ff617ad0ce",\n      "rgb_sha256": "4a57387d306680c976e0d51fdba04b69c8d4644dabee9f8ca5a97e2384f913e0"\n    },\n    "droid_natural_6d74030dc0_a_00034.png": {\n      "png_sha256": "adc93dbdd788b7cff8e90a8c0131bd74b1816e80677dd456775ee2f445958f21",\n      "rgb_sha256": "dbd5f88e3ea545b4314228c98a57fb355b180d84ff9b865019c6676d130957f3"\n    },\n    "droid_natural_6d74030dc0_a_00068.png": {\n      "png_sha256": "a661332ee34316fc64bcf6d973afc3f6d7b23d42d6e11f716b5df671379294e2",\n      "rgb_sha256": "8609f4fdeee8a51e93f165af3719576bb3d895ff1fd3fae4a0333429dbd2e38d"\n    },\n    "droid_natural_6d74030dc0_a_00102.png": {\n      "png_sha256": "c0c354ee8c93ace7f2034e479b5047b9cf45c9fe5cd81cdd4cfadd18d554b344",\n      "rgb_sha256": "fb5aed1d0d32f45efe5594dd2d47ac5a6078fb6f30601b697bf23806f36c9f59"\n    },\n    "droid_natural_6d74030dc0_a_00136.png": {\n      "png_sha256": "eb143b4a6187ad247a25b202e4899d47c7b080ea702d244ac24e969206a28b23",\n      "rgb_sha256": "652398bc62eaefc57362e4184011e95fd961c21beec075858bebae20c0e56ff1"\n    },\n    "droid_natural_6d74030dc0_a_00149.png": {\n      "png_sha256": "8cdf0806c6073e4d70b140ecb827e27c93cf82f29cbe770a399df24fbdf44183",\n      "rgb_sha256": "536e4ad3c2662ce50663a4c4860815916d45091d9dc78b930ebd7eeff707b70a"\n    },\n    "droid_natural_6d74030dc0_a_00160.png": {\n      "png_sha256": "088e4f4467a484abc80ac2594eba41a9b295c0a0d88f4827b4f03455da4146e5",\n      "rgb_sha256": "6d317bb548c3098314518cd88c74fa078c5cc5aa4c460e7a9657fe5bd89028cc"\n    },\n    "droid_natural_6d74030dc0_a_00171.png": {\n      "png_sha256": "1eecbbec9e9cc18a89468ebe169fed8f08c09a19a453dd463f98da2d41e84c7f",\n      "rgb_sha256": "714d7d2deb3e42ee3c704c80325b1aca3c363736fd93f23fb953c4e4fd9dea8f"\n    },\n    "droid_natural_6d74030dc0_b_00000.png": {\n      "png_sha256": "61983c8134f394fde7d7c5bee766777c8c8c88f5609275211a0919f7ba0eaa35",\n      "rgb_sha256": "f9a08a6a97bd2672583adb67daaff6f32bbc85deb1e39b89e8a823d0b5d9969a"\n    },\n    "droid_natural_6d74030dc0_b_00055.png": {\n      "png_sha256": "2a3a0dcedc2ccf2af785f314cd4c257e8b4f4f0e9967101cfbcf677c9f2853b1",\n      "rgb_sha256": "c10c9e4ede738994869d74607aa7fb096ee9ecfa5568ff4ffb6e8a498db1fa3a"\n    },\n    "droid_natural_6d74030dc0_b_00111.png": {\n      "png_sha256": "d7ad81ea7275446448072d6914bb077f8c9ff34a0e16ee75a5ba4b8a478b177b",\n      "rgb_sha256": "86405e22736d6c681bd9f467d269fee64cc5911aba411056c8a31f13ccf9a913"\n    },\n    "droid_natural_6d74030dc0_b_00167.png": {\n      "png_sha256": "5724dc13865a4a01157d7e4d2f84a2a2c9d044ef84b20bdc8ec0b7a2b60f8280",\n      "rgb_sha256": "5d91b643b745784ed753eec75a87171f3b4257d8a3f7ac97f4fc22ef9687cf0d"\n    },\n    "droid_natural_6d74030dc0_b_00223.png": {\n      "png_sha256": "ab59fb615d70e7ebd2a88d0d7297caed778a313ddd684afd484c40475b5e42a0",\n      "rgb_sha256": "481e05555aaac70cb96c37c51152fdd087ca257242ad147b816b18d8bcdc49b7"\n    },\n    "droid_natural_6d74030dc0_b_00229.png": {\n      "png_sha256": "436dba79a75c7fd5535f6fdf8313130b7ca5f08b0e3799cb2c7bf6064c9d37ba",\n      "rgb_sha256": "feab429de009c541c7cac29a0208a1e9ac01e922a731a88ecd06093c86da1a98"\n    },\n    "droid_natural_6d74030dc0_b_00280.png": {\n      "png_sha256": "12fa04c9ed7ada5c9621ba2503f9a8f03896c95540110218771c2a205aa5e6ca",\n      "rgb_sha256": "3ce83eff61a1a3609b1c0793d8433626ae5d5e2558b220ca0c0092b566e5d7cc"\n    },\n    "droid_natural_87abc314ec_a_00000.png": {\n      "png_sha256": "b80438ecff621713c84b51dd09c48dbcf09bf0d0175482116a39349bfadd8486",\n      "rgb_sha256": "f55789640ab2d5fd3e071f571725e6bd8d322f3190dd21e07fac56ff97ba475a"\n    },\n    "droid_natural_87abc314ec_a_00063.png": {\n      "png_sha256": "dde0ed8a9fadae6f672c3bfd16e6d8c1857a0d0b78f02f7409b8490034ca25a9",\n      "rgb_sha256": "314e2b4d0f0516a12740f15e6acd50203841aec1197703b65c3cad50e0653d12"\n    },\n    "droid_natural_87abc314ec_a_00127.png": {\n      "png_sha256": "c9a1cb1e14a3895c5e5054f6693d19ecd01c978c01017aac1b184d21d49d2029",\n      "rgb_sha256": "d7cd11dadbff2b5546528d616399acf665137b7e455064ea9f48aa12cd72481d"\n    },\n    "droid_natural_87abc314ec_a_00192.png": {\n      "png_sha256": "de7b5d3ccefa43b5cf7205b4bd3e2a9bee44ef144c1359f39ca840f2ad1efbb7",\n      "rgb_sha256": "4a83f6e8eb49d620e2e73c59bd319d90706fd0645e19f873341f3729622a0f44"\n    },\n    "droid_natural_87abc314ec_a_00255.png": {\n      "png_sha256": "b36bc3a87b2e113191d2dc63bf4003b0948e8addb4746bf865fe0dc3f0ad1230",\n      "rgb_sha256": "a1ce2804442a71ac4a922ecf62a87e3e964d512b424cfca18e4d88b270491081"\n    },\n    "droid_natural_87abc314ec_a_00279.png": {\n      "png_sha256": "ec3a2f0286439f91c5a221a32030a178873cf0f889e1ba86f3110ec7b9145871",\n      "rgb_sha256": "1c363c77b881d41a20d26225af05be5049e2a7a247ac8ea84e5acf1ea53924a9"\n    },\n    "droid_natural_87abc314ec_a_00299.png": {\n      "png_sha256": "c3243f8cdd63072bd4b5238be0d97d9b76a51cd7c97352fa769e0d7d6d70b447",\n      "rgb_sha256": "3da10f297a5ddea4ddd35b57ad88444d00e825dcb8de1890226fa2f639814e91"\n    },\n    "droid_natural_87abc314ec_a_00320.png": {\n      "png_sha256": "97638fe9b9c8784144f3ff589bddcb86fc8c737dc935b810030f349d7564e8bb",\n      "rgb_sha256": "3b7753bc7e2fef1b561e67aeea63010a94af0b37fdbd13599f7d282b9a8ce97f"\n    },\n    "droid_natural_87abc314ec_b_00000.png": {\n      "png_sha256": "b901c7d9ac0744bf121520278241d043c98d5dce173de808445078ba78ad8764",\n      "rgb_sha256": "07f653c8066e319d17105cb7a496214d690bf89c3b728a83af9c0f71c245ff74"\n    },\n    "droid_natural_87abc314ec_b_00030.png": {\n      "png_sha256": "68c7f689f41ef0c679d02e8e1b30f0d1c7e39df5e49d54dc4018488b7e0b585c",\n      "rgb_sha256": "17e0dc3f7af20cc0b0b87982dfa57cc912a4b28e2930c6703ece36835683f018"\n    },\n    "droid_natural_87abc314ec_b_00060.png": {\n      "png_sha256": "e1e3cb512c67329d65bacfb04aa489f7dba5261e59a1e239ee1551899f8daddc",\n      "rgb_sha256": "e1058907d3fbe663a54a8f53a6ba98cd93b5b685b37e8fa0f25db2e7f1756695"\n    },\n    "droid_natural_87abc314ec_b_00090.png": {\n      "png_sha256": "525d6142d8619e0cd9d9eb4ffe2beb2c89db5249044f6171e652ff61ebfd42db",\n      "rgb_sha256": "b215a98ae4d33ad75c1dd0ba15bdcacccb7358612322adb66e4996ddddcabaff"\n    },\n    "droid_natural_87abc314ec_b_00120.png": {\n      "png_sha256": "4b1a74048c68bc7f42c475ae6e1f45635e70c2ebe76a7a793cf8fc7e91d1c60b",\n      "rgb_sha256": "553671e325bb730c2986c350e6f57e6c37fe2b8f0b9355b329d3bfae0431d763"\n    },\n    "droid_natural_87abc314ec_b_00131.png": {\n      "png_sha256": "df5e39a020991bfacb60fc84ecf4557b6d4d465594ea2bb7700e7c57d0fb3359",\n      "rgb_sha256": "d840989e0548a7155c6765eb4c855ebd43488ead181f557c41e2fd9610914eaf"\n    },\n    "droid_natural_87abc314ec_b_00151.png": {\n      "png_sha256": "44be1519b81ca0e8ef198942af532fa0aa6118b731b9ec7298df0878e37f451f",\n      "rgb_sha256": "aeed76fd9d7e8f09d5c45a001b278ff75da5b0521416a82875553da29f1b91a9"\n    },\n    "droid_natural_8b01107c59_a_00000.png": {\n      "png_sha256": "17092527011e395d0feb686df9a05d3ecc38dba5cd94e9d436280baaa2ebdb47",\n      "rgb_sha256": "8767a47b9592203e4112d0cc979f5be77ba0ca64b7ffb1ec442e4596187c7d9e"\n    },\n    "droid_natural_8b01107c59_a_00051.png": {\n      "png_sha256": "89094a842cf2c88ef631636087b30ef2023ffd4e192cc077672f5811d9455758",\n      "rgb_sha256": "774194d6b591fab01b63b960c686f8c24109b7da39912644d264d19fcd42077c"\n    },\n    "droid_natural_8b01107c59_a_00102.png": {\n      "png_sha256": "cf252ecc092c4ba54b78406a52ec3cfcf6ed6c54e2e2c7bce783e68a269a008a",\n      "rgb_sha256": "2ca0a173c6eb65c63fd7f67b01e2b35bb60ea3d93df8d615dbd10705ee28e974"\n    },\n    "droid_natural_8b01107c59_a_00153.png": {\n      "png_sha256": "ccf10281877c3c1a02a0b187b591551c4a9836c491afea27524591afa5e32cb0",\n      "rgb_sha256": "71b73f81098ab14d5116cd4451aa6cd27ae9b449ecaead37d5b4ae7f4b62df68"\n    },\n    "droid_natural_8b01107c59_a_00204.png": {\n      "png_sha256": "921c5570ed4fe4bcc5928990548ef2649d3ee71737e0441d53dc4be92fcb8a59",\n      "rgb_sha256": "64955dfa6a305c8741650d00649948d4e787539b2e2e447064532a7ee5f60777"\n    },\n    "droid_natural_8b01107c59_a_00238.png": {\n      "png_sha256": "c1fe4fdd51fab29afda43da148db7cc6205a342907e4737393b13b72b06dc30f",\n      "rgb_sha256": "4411633b2a031a2aa903e37fc4f696abd968709cf7285cd9a9555a2b9fe11276"\n    },\n    "droid_natural_8b01107c59_a_00247.png": {\n      "png_sha256": "207d8ae730ac5acf4e59d2173ec7b639be595406caee095482fc7052aa6cfb50",\n      "rgb_sha256": "d9e04be59aacf03d404a58cab1762d200e0c3aa4137c70fa8c6f8464150a4953"\n    },\n    "droid_natural_8b01107c59_a_00256.png": {\n      "png_sha256": "00fd65554fdd3f8e485a12a0b59a8a0714f2519c067266f60ce8b0752ef02ce3",\n      "rgb_sha256": "e2e0fb7938b39beb6d177d4080b768640631c19d436c71625a5fdda7879e4c43"\n    },\n    "droid_natural_8b01107c59_b_00000.png": {\n      "png_sha256": "78c07fc56b04831ea95b5d0c495184a834db622c8b6092d9c73584a5be49fe47",\n      "rgb_sha256": "563c57dc0ee324dab566e17aba9b3cc8454c7b0711cfa7cb82ed0240fd8db275"\n    },\n    "droid_natural_8b01107c59_b_00035.png": {\n      "png_sha256": "c32b22bfceac6d1849c111f70b2014c8c98c06790af7c0e4034a06be54af8cd3",\n      "rgb_sha256": "25068a9d08a5fbb2af8627b48e7d3f9d05a5d0cf2c6b3ae6f28b6b8121f78094"\n    },\n    "droid_natural_8b01107c59_b_00070.png": {\n      "png_sha256": "3a1e921d084620f3faf8117ad79133eff428d45e2377016d4f67cf537800c947",\n      "rgb_sha256": "b8bb0793248358a78bfaf26df23ddfa4823f3c01dd88fa1febe30be244fd744b"\n    },\n    "droid_natural_8b01107c59_b_00105.png": {\n      "png_sha256": "b356cd0a17f022fdbd56b1ac708462bf44e7eb80caf4dee425f8a6e6a28af688",\n      "rgb_sha256": "003196286608593cbd6426fc5edcb1ff95d37a89336149eaf6f693eacad15328"\n    },\n    "droid_natural_8b01107c59_b_00140.png": {\n      "png_sha256": "80f2d03ef670ea3c0a24c53dbd3649288d0bb89499504c6d31a55a320ace328d",\n      "rgb_sha256": "887d203d9cbebfdf25798584881d53f6bad12020ded2bb049d46e4ffa8d0a89d"\n    },\n    "droid_natural_8b01107c59_b_00167.png": {\n      "png_sha256": "408bd7f29ea155346dfa5b8c9563bde5faac5b24e62e9175b140272d61063b92",\n      "rgb_sha256": "200bdfc090e8600d815055f0ccfce1ae5ff0e81fe5270d6815f09b379c18ec89"\n    },\n    "droid_natural_8b01107c59_b_00171.png": {\n      "png_sha256": "f33603d641e1fef3eed8eaa2bd875cdf154c51554617de63925b2322462c61bf",\n      "rgb_sha256": "d27b7ff452888572826e006ca08dbd15b05efba660a2798170c2107fafca447c"\n    },\n    "droid_natural_8b01107c59_b_00175.png": {\n      "png_sha256": "c43d4f69df774eefb6619ededb515d662527ebd9ec86e6c1d17d32f449990588",\n      "rgb_sha256": "f82aebffa2374e9b6e54e836fb6e4ea53aa9894c6e5640c15343c69691ce2519"\n    },\n    "droid_natural_c00374385e_a_00000.png": {\n      "png_sha256": "a8d7e3752f89f017fc999590ec22f012589ef65b4688ad2d0a75bc0702f6efb9",\n      "rgb_sha256": "25c93b20570a2d8d95836a2cfab1136f95ab1206a610c662b6262c8a987f38d1"\n    },\n    "droid_natural_c00374385e_a_00065.png": {\n      "png_sha256": "5a51251bb40406019e504babe3dda2bc2d47acb03a1e1fc83f8085948341442a",\n      "rgb_sha256": "adf413908710f43157dfdcbe667b0b7fe90590cad06ee946193f649fdf8776b7"\n    },\n    "droid_natural_c00374385e_a_00131.png": {\n      "png_sha256": "1f8ed7681bd83f232333bd66a30f9dc4cf6bf0ae2282abe6beb35f07cabf7e24",\n      "rgb_sha256": "334318dce375e6590833a55867734b85054354ca05fd1f9907253d7477dbdd75"\n    },\n    "droid_natural_c00374385e_a_00198.png": {\n      "png_sha256": "5aeb71da5d006288b405b42f97cd306066590e8edb744552a5d9b12adafc0433",\n      "rgb_sha256": "17b3311b595d9828e9c57c11fcf5d9cd5e74de71a2f473a0e4ccfd471a8c526c"\n    },\n    "droid_natural_c00374385e_a_00263.png": {\n      "png_sha256": "af11d36200aa3fa974d6d03e5c863f690469121617d10e60e18b6024a85adf0a",\n      "rgb_sha256": "d314f3a307fed1722b9c0e128eef304a5ae324c0b97ea3e8085d5d29871f0e73"\n    },\n    "droid_natural_c00374385e_a_00300.png": {\n      "png_sha256": "eed7605e632ff2cf0b6776d47d9337c7ff6a93f8124055d57a7e1c9a7f34b26c",\n      "rgb_sha256": "df78a12045202629bedbe5748aeb75db96637fcb14739ee97dddf5f7d50528f5"\n    },\n    "droid_natural_c00374385e_a_00315.png": {\n      "png_sha256": "c8d1fefe6b1ba65cdbf2f9672fe2c8496432746ea86ae1a7796fc1ec6b0d2d3d",\n      "rgb_sha256": "e1933e82c4783e1a700b295b0bb8877cd01de8b692107218a5fbc9d611ae19a4"\n    },\n    "droid_natural_c00374385e_a_00330.png": {\n      "png_sha256": "7ffef89523e769d9cad5d3587aa858909ea32dc669f263bb42718dfce02a9ff7",\n      "rgb_sha256": "53c3ff287b2a842433b5dd9bf51cd7486d640e33eb1fec53d1e273e055b4a8ed"\n    },\n    "droid_natural_c00374385e_b_00000.png": {\n      "png_sha256": "3d7da22cb2fa8f053661afacdab2a93a3d5468a3d95fcf56b8892acbbc5d9467",\n      "rgb_sha256": "36f3caf5219cec4c584a8b474a91decc7b71bdf45b150e3e7cd14cf6bc0e6e1e"\n    },\n    "droid_natural_c00374385e_b_00044.png": {\n      "png_sha256": "ce5ce93c6f18b45a8c670d19d3108edd8340cf7f0f4fdbbe804405f54484c5fe",\n      "rgb_sha256": "93f3fadedb4712056bc33a97983c7734d916922902ba7a9f56cb157fa63a216a"\n    },\n    "droid_natural_c00374385e_b_00089.png": {\n      "png_sha256": "bd65a846779f56bb92aa21a70c782d544754b7642551fe8a14e76d6132488f36",\n      "rgb_sha256": "5583e6fb1173ca27599bc4c89fcd187062085623b9004a3d713b3d771875e742"\n    },\n    "droid_natural_c00374385e_b_00133.png": {\n      "png_sha256": "4a9ef125787f62c5b0f5f2c80f3a1c328386e219a958b0c4e48be72cd24cef33",\n      "rgb_sha256": "047aa5bb6194d1fff6341f220b927a098ba9b47863d10ec26b9716b95da72f48"\n    },\n    "droid_natural_c00374385e_b_00178.png": {\n      "png_sha256": "32f459885bc88cee62b662853e7a3b40e8f7ee771d8bd7b06f13a441a7c9a316",\n      "rgb_sha256": "f14623a5c21f27bc072ba0d85aa3331a8b58914aea516913c74e17025979a2db"\n    },\n    "droid_natural_c00374385e_b_00194.png": {\n      "png_sha256": "4a8035b472109e037fd5ab39a85b012f8407f49a164dbf9219fcfe4599fbd171",\n      "rgb_sha256": "08d1193138eebdbcd99384cdc0d1d81832fd213c1bd64e512c37d783816d5bac"\n    },\n    "droid_natural_c00374385e_b_00208.png": {\n      "png_sha256": "a899fdf821a8954ec37020d7f9688412d53a0ad087b8f76f842f7dde5aeda2ef",\n      "rgb_sha256": "314fffb38910b2408fa9fee9af11451c1b273671b3453905fbd7dba6cda499e5"\n    },\n    "droid_natural_c00374385e_b_00223.png": {\n      "png_sha256": "02704cc8f932408e6542e7866a29a0c4d1a9c3dccb8fd0ed7c13b3678c35517c",\n      "rgb_sha256": "59443b34d012cee0f8c9d939032036e948ee66e1daa4b214a6499cc96bfe933d"\n    },\n    "droid_shared_recent_00015.png": {\n      "png_sha256": "c73295de2b63a2b7691ce3c4e7b0c25f7a93e06108bc3c850a66ac433aad1d79",\n      "rgb_sha256": "3d3a2dac71d987d075e9bbe321ca72f089a7b2d6fc3a159f64bf546029aefcb8"\n    },\n    "droid_shared_recent_00075.png": {\n      "png_sha256": "1bcb55ebbcd76a9671f4a826a46d38dd41e8514a0966e9b40bcffecc3b8f0ecb",\n      "rgb_sha256": "8ce787896e70698d3f30417a7fdbe3c487deab536488b7e212914481f7ada688"\n    },\n    "droid_shared_recent_00135.png": {\n      "png_sha256": "29563eb3bba0569b23ee90dc52d8dfb9bf4df41b5efcf151f42937bce7493fb6",\n      "rgb_sha256": "0090eadb9ac1f7478ac4313f5d41e127f0bc76b86b2db3b0bd02e69ee23fedbe"\n    },\n    "droid_shared_recent_00195.png": {\n      "png_sha256": "0d64158465ead4163d81607bc0b7edfcf7af3b829b9673777c44b8f9c7a0d8d5",\n      "rgb_sha256": "2fe28128f70b39462570ae0a3fe194314486d9336105fe500f8c428fe72c420f"\n    }\n  },\n  "cases": [\n    {\n      "pair_id": "droid_natural_8b01107c59",\n      "source_site": "IPRL",\n      "scene_id": 1611836543,\n      "camera": "ext2",\n      "question": "In the last view of the wooden drawer to the left of the stove before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "IPRL+7790ec0a+2023-06-30-22h-05m-43s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-06-30/Fri_Jun_30_22%3A05%3A43_2023/recordings/MP4/28221883.mp4",\n            "generation": "1710757816663629",\n            "bytes": 8311236,\n            "sha256": "9ddd63b993cf98aafb39d220197874c8c80ca16422557a48f56aa10c4574180e",\n            "frame_count": 260\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-06-30/Fri_Jun_30_22%3A05%3A43_2023/metadata_IPRL%2B7790ec0a%2B2023-06-30-22h-05m-43s.json",\n            "generation": "1710757816667902",\n            "bytes": 1780,\n            "sha256": "7a30014b88039c94150cd732c867ddae976c6db301bfd2061e5f4654072a1cdf"\n          },\n          "initial_interval": [\n            0,\n            55\n          ],\n          "terminal_interval": [\n            230,\n            256\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              51,\n              102,\n              153,\n              204,\n              256\n            ],\n            "annotated": [\n              0,\n              51,\n              102,\n              238,\n              247,\n              256\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.8500000238418579,\n              1.7000000476837158,\n              2.549999952316284,\n              3.4000000953674316,\n              4.266666889190674\n            ],\n            "annotated": [\n              0.0,\n              0.8500000238418579,\n              1.7000000476837158,\n              3.9666666984558105,\n              4.116666793823242,\n              4.266666889190674\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_8b01107c59_a_00000.png",\n              "droid_natural_8b01107c59_a_00051.png",\n              "droid_natural_8b01107c59_a_00102.png",\n              "droid_natural_8b01107c59_a_00153.png",\n              "droid_natural_8b01107c59_a_00204.png",\n              "droid_natural_8b01107c59_a_00256.png"\n            ],\n            "annotated": [\n              "droid_natural_8b01107c59_a_00000.png",\n              "droid_natural_8b01107c59_a_00051.png",\n              "droid_natural_8b01107c59_a_00102.png",\n              "droid_natural_8b01107c59_a_00238.png",\n              "droid_natural_8b01107c59_a_00247.png",\n              "droid_natural_8b01107c59_a_00256.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "IPRL+7790ec0a+2023-06-30-22h-04m-30s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-06-30/Fri_Jun_30_22%3A04%3A30_2023/recordings/MP4/28221883.mp4",\n            "generation": "1710757816422867",\n            "bytes": 5917508,\n            "sha256": "4e46d10846bde615b357407d8842ee496fa647c11369a5c14394b39989b30bef",\n            "frame_count": 178\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-06-30/Fri_Jun_30_22%3A04%3A30_2023/metadata_IPRL%2B7790ec0a%2B2023-06-30-22h-04m-30s.json",\n            "generation": "1710757816401695",\n            "bytes": 1784,\n            "sha256": "ca2f2753ab7d4c0629a201a57f5e3533b191ab2a8421c323d353e069a7463ccd"\n          },\n          "initial_interval": [\n            0,\n            65\n          ],\n          "terminal_interval": [\n            164,\n            175\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              35,\n              70,\n              105,\n              140,\n              175\n            ],\n            "annotated": [\n              0,\n              35,\n              70,\n              167,\n              171,\n              175\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.5833333134651184,\n              1.1666666269302368,\n              1.75,\n              2.3333332538604736,\n              2.9166667461395264\n            ],\n            "annotated": [\n              0.0,\n              0.5833333134651184,\n              1.1666666269302368,\n              2.7833333015441895,\n              2.8499999046325684,\n              2.9166667461395264\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_8b01107c59_b_00000.png",\n              "droid_natural_8b01107c59_b_00035.png",\n              "droid_natural_8b01107c59_b_00070.png",\n              "droid_natural_8b01107c59_b_00105.png",\n              "droid_natural_8b01107c59_b_00140.png",\n              "droid_natural_8b01107c59_b_00175.png"\n            ],\n            "annotated": [\n              "droid_natural_8b01107c59_b_00000.png",\n              "droid_natural_8b01107c59_b_00035.png",\n              "droid_natural_8b01107c59_b_00070.png",\n              "droid_natural_8b01107c59_b_00167.png",\n              "droid_natural_8b01107c59_b_00171.png",\n              "droid_natural_8b01107c59_b_00175.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_6d74030dc0",\n      "source_site": "IPRL",\n      "scene_id": 4134662280,\n      "camera": "ext1",\n      "question": "In the last view of the wooden cabinet door just right of the range hood before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "IPRL+5085c3ce+2023-10-16-22h-09m-27s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-10-16/Mon_Oct_16_22%3A09%3A27_2023/recordings/MP4/27432424.mp4",\n            "generation": "1710757941602537",\n            "bytes": 5705476,\n            "sha256": "eda54d343755fe54b5f6896fb09105781bb57a835d4e4eb4d63b707213391c70",\n            "frame_count": 174\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-10-16/Mon_Oct_16_22%3A09%3A27_2023/metadata_IPRL%2B5085c3ce%2B2023-10-16-22h-09m-27s.json",\n            "generation": "1710757941611160",\n            "bytes": 1799,\n            "sha256": "b0917641341c070d2c95f68ca75a142027272b37bf412a4db3305ea98f875fdd"\n          },\n          "initial_interval": [\n            0,\n            38\n          ],\n          "terminal_interval": [\n            138,\n            171\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              34,\n              68,\n              102,\n              136,\n              171\n            ],\n            "annotated": [\n              0,\n              34,\n              68,\n              149,\n              160,\n              171\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.5666666626930237,\n              1.1333333253860474,\n              1.7000000476837158,\n              2.2666666507720947,\n              2.8499999046325684\n            ],\n            "annotated": [\n              0.0,\n              0.5666666626930237,\n              1.1333333253860474,\n              2.4833333492279053,\n              2.6666667461395264,\n              2.8499999046325684\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_6d74030dc0_a_00000.png",\n              "droid_natural_6d74030dc0_a_00034.png",\n              "droid_natural_6d74030dc0_a_00068.png",\n              "droid_natural_6d74030dc0_a_00102.png",\n              "droid_natural_6d74030dc0_a_00136.png",\n              "droid_natural_6d74030dc0_a_00171.png"\n            ],\n            "annotated": [\n              "droid_natural_6d74030dc0_a_00000.png",\n              "droid_natural_6d74030dc0_a_00034.png",\n              "droid_natural_6d74030dc0_a_00068.png",\n              "droid_natural_6d74030dc0_a_00149.png",\n              "droid_natural_6d74030dc0_a_00160.png",\n              "droid_natural_6d74030dc0_a_00171.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "IPRL+5085c3ce+2023-10-16-22h-07m-54s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-10-16/Mon_Oct_16_22%3A07%3A54_2023/recordings/MP4/27432424.mp4",\n            "generation": "1710757941296647",\n            "bytes": 9285109,\n            "sha256": "ce1c868e7a2b93e2742d717bd2f46b7a4ba34cc60565f618ae1fa1d0960e7d3b",\n            "frame_count": 284\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IPRL/success/2023-10-16/Mon_Oct_16_22%3A07%3A54_2023/metadata_IPRL%2B5085c3ce%2B2023-10-16-22h-07m-54s.json",\n            "generation": "1710757941280972",\n            "bytes": 1800,\n            "sha256": "ae73f48ea3d22f5c07a06aea068789b4bff379a020a1da71df2d24f9f4368f31"\n          },\n          "initial_interval": [\n            0,\n            85\n          ],\n          "terminal_interval": [\n            178,\n            280\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              55,\n              111,\n              167,\n              223,\n              280\n            ],\n            "annotated": [\n              0,\n              55,\n              111,\n              223,\n              229,\n              280\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.9166666865348816,\n              1.850000023841858,\n              2.7833333015441895,\n              3.7166666984558105,\n              4.666666507720947\n            ],\n            "annotated": [\n              0.0,\n              0.9166666865348816,\n              1.850000023841858,\n              3.7166666984558105,\n              3.816666603088379,\n              4.666666507720947\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_6d74030dc0_b_00000.png",\n              "droid_natural_6d74030dc0_b_00055.png",\n              "droid_natural_6d74030dc0_b_00111.png",\n              "droid_natural_6d74030dc0_b_00167.png",\n              "droid_natural_6d74030dc0_b_00223.png",\n              "droid_natural_6d74030dc0_b_00280.png"\n            ],\n            "annotated": [\n              "droid_natural_6d74030dc0_b_00000.png",\n              "droid_natural_6d74030dc0_b_00055.png",\n              "droid_natural_6d74030dc0_b_00111.png",\n              "droid_natural_6d74030dc0_b_00223.png",\n              "droid_natural_6d74030dc0_b_00229.png",\n              "droid_natural_6d74030dc0_b_00280.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_87abc314ec",\n      "source_site": "IRIS",\n      "scene_id": 8066802860,\n      "camera": "ext2",\n      "question": "In the last view of the wide white drawer below the countertop before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "IRIS+85a5958f+2023-07-05-15h-52m-26s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-07-05/Wed_Jul__5_15%3A52%3A26_2023/recordings/MP4/29838012.mp4",\n            "generation": "1710758376365608",\n            "bytes": 7156442,\n            "sha256": "e6e20fb2b68d3b38af76c732494fce322b79fd50c41c829f5daeb7f832b0d380",\n            "frame_count": 325\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-07-05/Wed_Jul__5_15%3A52%3A26_2023/metadata_IRIS%2B85a5958f%2B2023-07-05-15h-52m-26s.json",\n            "generation": "1710758376337176",\n            "bytes": 1794,\n            "sha256": "01a6dbd2ed52b507c3eef51f0e22a80733ae505f4be568a69c34881b53490555"\n          },\n          "initial_interval": [\n            0,\n            180\n          ],\n          "terminal_interval": [\n            259,\n            320\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              63,\n              127,\n              192,\n              255,\n              320\n            ],\n            "annotated": [\n              0,\n              63,\n              127,\n              279,\n              299,\n              320\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              1.0499999523162842,\n              2.116666555404663,\n              3.200000047683716,\n              4.25,\n              5.333333492279053\n            ],\n            "annotated": [\n              0.0,\n              1.0499999523162842,\n              2.116666555404663,\n              4.650000095367432,\n              4.983333110809326,\n              5.333333492279053\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_87abc314ec_a_00000.png",\n              "droid_natural_87abc314ec_a_00063.png",\n              "droid_natural_87abc314ec_a_00127.png",\n              "droid_natural_87abc314ec_a_00192.png",\n              "droid_natural_87abc314ec_a_00255.png",\n              "droid_natural_87abc314ec_a_00320.png"\n            ],\n            "annotated": [\n              "droid_natural_87abc314ec_a_00000.png",\n              "droid_natural_87abc314ec_a_00063.png",\n              "droid_natural_87abc314ec_a_00127.png",\n              "droid_natural_87abc314ec_a_00279.png",\n              "droid_natural_87abc314ec_a_00299.png",\n              "droid_natural_87abc314ec_a_00320.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "IRIS+85a5958f+2023-07-05-15h-52m-08s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-07-05/Wed_Jul__5_15%3A52%3A08_2023/recordings/MP4/29838012.mp4",\n            "generation": "1710758376295985",\n            "bytes": 3032291,\n            "sha256": "cb1e60e324a1cce7206934beb6587f2a326ddd408c9f43f892b043ce2dd5a251",\n            "frame_count": 154\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-07-05/Wed_Jul__5_15%3A52%3A08_2023/metadata_IRIS%2B85a5958f%2B2023-07-05-15h-52m-08s.json",\n            "generation": "1710758376266027",\n            "bytes": 1783,\n            "sha256": "2b38f21306c7d33f3a4b01a953181f20f987afe392a67815e159a339517cf1ff"\n          },\n          "initial_interval": [\n            0,\n            38\n          ],\n          "terminal_interval": [\n            110,\n            151\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              30,\n              60,\n              90,\n              120,\n              151\n            ],\n            "annotated": [\n              0,\n              30,\n              60,\n              120,\n              131,\n              151\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.5,\n              1.0,\n              1.5,\n              2.0,\n              2.5166666507720947\n            ],\n            "annotated": [\n              0.0,\n              0.5,\n              1.0,\n              2.0,\n              2.183333396911621,\n              2.5166666507720947\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_87abc314ec_b_00000.png",\n              "droid_natural_87abc314ec_b_00030.png",\n              "droid_natural_87abc314ec_b_00060.png",\n              "droid_natural_87abc314ec_b_00090.png",\n              "droid_natural_87abc314ec_b_00120.png",\n              "droid_natural_87abc314ec_b_00151.png"\n            ],\n            "annotated": [\n              "droid_natural_87abc314ec_b_00000.png",\n              "droid_natural_87abc314ec_b_00030.png",\n              "droid_natural_87abc314ec_b_00060.png",\n              "droid_natural_87abc314ec_b_00120.png",\n              "droid_natural_87abc314ec_b_00131.png",\n              "droid_natural_87abc314ec_b_00151.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_29137e19c7",\n      "source_site": "IRIS",\n      "scene_id": 2158993557,\n      "camera": "ext1",\n      "question": "In the last view of the drawer under the white counter beside the black sink before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "IRIS+7dfa2da3+2023-06-01-13h-38m-34s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-06-01/Thu_Jun__1_13%3A38%3A34_2023/recordings/MP4/23404442.mp4",\n            "generation": "1710758322637107",\n            "bytes": 3654602,\n            "sha256": "8037e3eb013ed770746276791fcda236b0b7e6075a3de9e951ab0d71ac25754c",\n            "frame_count": 138\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-06-01/Thu_Jun__1_13%3A38%3A34_2023/metadata_IRIS%2B7dfa2da3%2B2023-06-01-13h-38m-34s.json",\n            "generation": "1710758322596568",\n            "bytes": 1778,\n            "sha256": "e63d53caebfee4f282ba8291dd1cd93c9326b53f46291a4e33bcaee5dcea6d2a"\n          },\n          "initial_interval": [\n            0,\n            44\n          ],\n          "terminal_interval": [\n            130,\n            135\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              27,\n              54,\n              80,\n              108,\n              135\n            ],\n            "annotated": [\n              0,\n              27,\n              54,\n              131,\n              133,\n              135\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.44999998807907104,\n              0.8999999761581421,\n              1.3333333730697632,\n              1.7999999523162842,\n              2.25\n            ],\n            "annotated": [\n              0.0,\n              0.44999998807907104,\n              0.8999999761581421,\n              2.183333396911621,\n              2.2166666984558105,\n              2.25\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_29137e19c7_a_00000.png",\n              "droid_natural_29137e19c7_a_00027.png",\n              "droid_natural_29137e19c7_a_00054.png",\n              "droid_natural_29137e19c7_a_00080.png",\n              "droid_natural_29137e19c7_a_00108.png",\n              "droid_natural_29137e19c7_a_00135.png"\n            ],\n            "annotated": [\n              "droid_natural_29137e19c7_a_00000.png",\n              "droid_natural_29137e19c7_a_00027.png",\n              "droid_natural_29137e19c7_a_00054.png",\n              "droid_natural_29137e19c7_a_00131.png",\n              "droid_natural_29137e19c7_a_00133.png",\n              "droid_natural_29137e19c7_a_00135.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "IRIS+7dfa2da3+2023-06-01-13h-38m-52s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-06-01/Thu_Jun__1_13%3A38%3A52_2023/recordings/MP4/23404442.mp4",\n            "generation": "1710758322698571",\n            "bytes": 5589729,\n            "sha256": "4b9c0c41f0825529c9e98b0f58cc38a2e8c66bcab54a51f30197b04f7936c277",\n            "frame_count": 209\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/IRIS/success/2023-06-01/Thu_Jun__1_13%3A38%3A52_2023/metadata_IRIS%2B7dfa2da3%2B2023-06-01-13h-38m-52s.json",\n            "generation": "1710758322695522",\n            "bytes": 1775,\n            "sha256": "a67182778d57fc82d72c66f29b17872e4fbf29c2a5366189035cefc893382bb6"\n          },\n          "initial_interval": [\n            0,\n            58\n          ],\n          "terminal_interval": [\n            166,\n            205\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              41,\n              82,\n              123,\n              164,\n              205\n            ],\n            "annotated": [\n              0,\n              41,\n              82,\n              179,\n              192,\n              205\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.6833333373069763,\n              1.3666666746139526,\n              2.049999952316284,\n              2.7333333492279053,\n              3.4166667461395264\n            ],\n            "annotated": [\n              0.0,\n              0.6833333373069763,\n              1.3666666746139526,\n              2.9833333492279053,\n              3.200000047683716,\n              3.4166667461395264\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_29137e19c7_b_00000.png",\n              "droid_natural_29137e19c7_b_00041.png",\n              "droid_natural_29137e19c7_b_00082.png",\n              "droid_natural_29137e19c7_b_00123.png",\n              "droid_natural_29137e19c7_b_00164.png",\n              "droid_natural_29137e19c7_b_00205.png"\n            ],\n            "annotated": [\n              "droid_natural_29137e19c7_b_00000.png",\n              "droid_natural_29137e19c7_b_00041.png",\n              "droid_natural_29137e19c7_b_00082.png",\n              "droid_natural_29137e19c7_b_00179.png",\n              "droid_natural_29137e19c7_b_00192.png",\n              "droid_natural_29137e19c7_b_00205.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_c00374385e",\n      "source_site": "PennPAL",\n      "scene_id": 5493550952,\n      "camera": "ext2",\n      "question": "In the last view of the wooden drawer just left of the white refrigerator before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "PennPAL+acda9df3+2023-07-07-21h-18m-34s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-07/Fri_Jul__7_21%3A18%3A34_2023/recordings/MP4/27085680.mp4",\n            "generation": "1710758561397919",\n            "bytes": 9508464,\n            "sha256": "e9e412020855e15ae7f636be3c0881999dbd23d16c5a336f324428975000b09c",\n            "frame_count": 335\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-07/Fri_Jul__7_21%3A18%3A34_2023/metadata_PennPAL%2Bacda9df3%2B2023-07-07-21h-18m-34s.json",\n            "generation": "1710758561353555",\n            "bytes": 1923,\n            "sha256": "4b3413661323da82e8237f923a9ba1e53329e7e3bcb01a8956d4a0eb5b894b26"\n          },\n          "initial_interval": [\n            0,\n            108\n          ],\n          "terminal_interval": [\n            285,\n            330\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              65,\n              131,\n              198,\n              263,\n              330\n            ],\n            "annotated": [\n              0,\n              65,\n              131,\n              300,\n              315,\n              330\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              1.0833333730697632,\n              2.183333396911621,\n              3.299999952316284,\n              4.383333206176758,\n              5.5\n            ],\n            "annotated": [\n              0.0,\n              1.0833333730697632,\n              2.183333396911621,\n              5.0,\n              5.25,\n              5.5\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_c00374385e_a_00000.png",\n              "droid_natural_c00374385e_a_00065.png",\n              "droid_natural_c00374385e_a_00131.png",\n              "droid_natural_c00374385e_a_00198.png",\n              "droid_natural_c00374385e_a_00263.png",\n              "droid_natural_c00374385e_a_00330.png"\n            ],\n            "annotated": [\n              "droid_natural_c00374385e_a_00000.png",\n              "droid_natural_c00374385e_a_00065.png",\n              "droid_natural_c00374385e_a_00131.png",\n              "droid_natural_c00374385e_a_00300.png",\n              "droid_natural_c00374385e_a_00315.png",\n              "droid_natural_c00374385e_a_00330.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "PennPAL+acda9df3+2023-07-07-21h-18m-12s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-07/Fri_Jul__7_21%3A18%3A12_2023/recordings/MP4/27085680.mp4",\n            "generation": "1710758561311076",\n            "bytes": 6155523,\n            "sha256": "d309345236d6628a4d7a4607e1810a8fb6d2186421b73a42fbe823fec2ceaa69",\n            "frame_count": 227\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-07/Fri_Jul__7_21%3A18%3A12_2023/metadata_PennPAL%2Bacda9df3%2B2023-07-07-21h-18m-12s.json",\n            "generation": "1710758561301150",\n            "bytes": 1787,\n            "sha256": "2b2e936cad942ca93db7d27000a0e5e8771bf560ea9deb81c59ec6263d7b7b2c"\n          },\n          "initial_interval": [\n            0,\n            60\n          ],\n          "terminal_interval": [\n            180,\n            223\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              44,\n              89,\n              133,\n              178,\n              223\n            ],\n            "annotated": [\n              0,\n              44,\n              89,\n              194,\n              208,\n              223\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.7333333492279053,\n              1.4833333492279053,\n              2.2166666984558105,\n              2.9666666984558105,\n              3.7166666984558105\n            ],\n            "annotated": [\n              0.0,\n              0.7333333492279053,\n              1.4833333492279053,\n              3.2333333492279053,\n              3.4666666984558105,\n              3.7166666984558105\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_c00374385e_b_00000.png",\n              "droid_natural_c00374385e_b_00044.png",\n              "droid_natural_c00374385e_b_00089.png",\n              "droid_natural_c00374385e_b_00133.png",\n              "droid_natural_c00374385e_b_00178.png",\n              "droid_natural_c00374385e_b_00223.png"\n            ],\n            "annotated": [\n              "droid_natural_c00374385e_b_00000.png",\n              "droid_natural_c00374385e_b_00044.png",\n              "droid_natural_c00374385e_b_00089.png",\n              "droid_natural_c00374385e_b_00194.png",\n              "droid_natural_c00374385e_b_00208.png",\n              "droid_natural_c00374385e_b_00223.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_19bf4a67ee",\n      "source_site": "PennPAL",\n      "scene_id": 9160815698,\n      "camera": "ext1",\n      "question": "In the last view of the glass door of the blue countertop oven before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "PennPAL+acda9df3+2023-07-10-21h-04m-33s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-10/Mon_Jul_10_21%3A04%3A33_2023/recordings/MP4/25455306.mp4",\n            "generation": "1710758567383463",\n            "bytes": 2508718,\n            "sha256": "6f6f27e35653e2fee7907845611c17535b8d611e883b1c1e403949c4714eeb3b",\n            "frame_count": 187\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-10/Mon_Jul_10_21%3A04%3A33_2023/metadata_PennPAL%2Bacda9df3%2B2023-07-10-21h-04m-33s.json",\n            "generation": "1710758567363957",\n            "bytes": 1803,\n            "sha256": "49d2cf2506a4960d24ee85ff6c9c29d5f5fc8292d68d9870e370fbf6f3df37ea"\n          },\n          "initial_interval": [\n            0,\n            69\n          ],\n          "terminal_interval": [\n            151,\n            184\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              36,\n              73,\n              110,\n              147,\n              184\n            ],\n            "annotated": [\n              0,\n              36,\n              73,\n              162,\n              173,\n              184\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.6000000238418579,\n              1.2166666984558105,\n              1.8333333730697632,\n              2.450000047683716,\n              3.066666603088379\n            ],\n            "annotated": [\n              0.0,\n              0.6000000238418579,\n              1.2166666984558105,\n              2.700000047683716,\n              2.883333444595337,\n              3.066666603088379\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_19bf4a67ee_a_00000.png",\n              "droid_natural_19bf4a67ee_a_00036.png",\n              "droid_natural_19bf4a67ee_a_00073.png",\n              "droid_natural_19bf4a67ee_a_00110.png",\n              "droid_natural_19bf4a67ee_a_00147.png",\n              "droid_natural_19bf4a67ee_a_00184.png"\n            ],\n            "annotated": [\n              "droid_natural_19bf4a67ee_a_00000.png",\n              "droid_natural_19bf4a67ee_a_00036.png",\n              "droid_natural_19bf4a67ee_a_00073.png",\n              "droid_natural_19bf4a67ee_a_00162.png",\n              "droid_natural_19bf4a67ee_a_00173.png",\n              "droid_natural_19bf4a67ee_a_00184.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "PennPAL+acda9df3+2023-07-10-21h-05m-18s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-10/Mon_Jul_10_21%3A05%3A18_2023/recordings/MP4/25455306.mp4",\n            "generation": "1710758567451868",\n            "bytes": 4391196,\n            "sha256": "d1838e81cfa032f199efcb795b467f0eeac2a22cdb52d95142176e18c67870a7",\n            "frame_count": 318\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/PennPAL/success/2023-07-10/Mon_Jul_10_21%3A05%3A18_2023/metadata_PennPAL%2Bacda9df3%2B2023-07-10-21h-05m-18s.json",\n            "generation": "1710758567397746",\n            "bytes": 1804,\n            "sha256": "ff06c37ca40faec017ab7833e4d202b36020316c2f324f1ddd4a4a2988ba4b67"\n          },\n          "initial_interval": [\n            0,\n            110\n          ],\n          "terminal_interval": [\n            253,\n            313\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              62,\n              125,\n              187,\n              250,\n              313\n            ],\n            "annotated": [\n              0,\n              62,\n              125,\n              273,\n              293,\n              313\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              1.0333333015441895,\n              2.0833332538604736,\n              3.116666555404663,\n              4.166666507720947,\n              5.2166666984558105\n            ],\n            "annotated": [\n              0.0,\n              1.0333333015441895,\n              2.0833332538604736,\n              4.550000190734863,\n              4.883333206176758,\n              5.2166666984558105\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_19bf4a67ee_b_00000.png",\n              "droid_natural_19bf4a67ee_b_00062.png",\n              "droid_natural_19bf4a67ee_b_00125.png",\n              "droid_natural_19bf4a67ee_b_00187.png",\n              "droid_natural_19bf4a67ee_b_00250.png",\n              "droid_natural_19bf4a67ee_b_00313.png"\n            ],\n            "annotated": [\n              "droid_natural_19bf4a67ee_b_00000.png",\n              "droid_natural_19bf4a67ee_b_00062.png",\n              "droid_natural_19bf4a67ee_b_00125.png",\n              "droid_natural_19bf4a67ee_b_00273.png",\n              "droid_natural_19bf4a67ee_b_00293.png",\n              "droid_natural_19bf4a67ee_b_00313.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    },\n    {\n      "pair_id": "droid_natural_10bd743a70",\n      "source_site": "TRI",\n      "scene_id": 8077138943,\n      "camera": "ext2",\n      "question": "In the last view of the glass-front cabinet door below the fruit basket before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "branches": {\n        "a": {\n          "source_episode": "TRI+52ca9b6a+2023-10-17-14h-09m-01s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/TRI/success/2023-10-17/Tue_Oct_17_14%3A09%3A01_2023/recordings/MP4/28813166.mp4",\n            "generation": "1710760215340966",\n            "bytes": 13204787,\n            "sha256": "dffa11a53fd6b1578448e9f49def98e966a1833ca4f1c1bc6a19b9f3df51aaf1",\n            "frame_count": 392\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/TRI/success/2023-10-17/Tue_Oct_17_14%3A09%3A01_2023/metadata_TRI%2B52ca9b6a%2B2023-10-17-14h-09m-01s.json",\n            "generation": "1710760215307612",\n            "bytes": 1796,\n            "sha256": "d35e395c94c59c1d3025d9e6ffcba3842804a87ea8f22910e5a36b915a0efe36"\n          },\n          "initial_interval": [\n            0,\n            140\n          ],\n          "terminal_interval": [\n            344,\n            387\n          ],\n          "gold": "A",\n          "source_indices": {\n            "uniform": [\n              0,\n              77,\n              154,\n              232,\n              309,\n              387\n            ],\n            "annotated": [\n              0,\n              77,\n              154,\n              358,\n              372,\n              387\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              1.2833333015441895,\n              2.566666603088379,\n              3.866666555404663,\n              5.150000095367432,\n              6.449999809265137\n            ],\n            "annotated": [\n              0.0,\n              1.2833333015441895,\n              2.566666603088379,\n              5.9666666984558105,\n              6.199999809265137,\n              6.449999809265137\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_10bd743a70_a_00000.png",\n              "droid_natural_10bd743a70_a_00077.png",\n              "droid_natural_10bd743a70_a_00154.png",\n              "droid_natural_10bd743a70_a_00232.png",\n              "droid_natural_10bd743a70_a_00309.png",\n              "droid_natural_10bd743a70_a_00387.png"\n            ],\n            "annotated": [\n              "droid_natural_10bd743a70_a_00000.png",\n              "droid_natural_10bd743a70_a_00077.png",\n              "droid_natural_10bd743a70_a_00154.png",\n              "droid_natural_10bd743a70_a_00358.png",\n              "droid_natural_10bd743a70_a_00372.png",\n              "droid_natural_10bd743a70_a_00387.png"\n            ]\n          }\n        },\n        "b": {\n          "source_episode": "TRI+52ca9b6a+2023-10-17-14h-07m-39s",\n          "video_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/TRI/success/2023-10-17/Tue_Oct_17_14%3A07%3A39_2023/recordings/MP4/28813166.mp4",\n            "generation": "1710760215048774",\n            "bytes": 9654241,\n            "sha256": "0e299a82cbd34ad2f1ed72170c564c2e66166f4cfbb1b0dd911d7ced928a2c0f",\n            "frame_count": 281\n          },\n          "metadata_source": {\n            "url": "https://storage.googleapis.com/gresearch/robotics/droid_raw/1.0.1/TRI/success/2023-10-17/Tue_Oct_17_14%3A07%3A39_2023/metadata_TRI%2B52ca9b6a%2B2023-10-17-14h-07m-39s.json",\n            "generation": "1710760215011050",\n            "bytes": 1786,\n            "sha256": "3115aeef488ab0e4e2fea7cf8d9ffbcd29e41bdb2a971175d8f70476e95b1ffe"\n          },\n          "initial_interval": [\n            0,\n            85\n          ],\n          "terminal_interval": [\n            255,\n            277\n          ],\n          "gold": "B",\n          "source_indices": {\n            "uniform": [\n              0,\n              55,\n              110,\n              166,\n              221,\n              277\n            ],\n            "annotated": [\n              0,\n              55,\n              110,\n              262,\n              269,\n              277\n            ]\n          },\n          "source_timestamps_s": {\n            "uniform": [\n              0.0,\n              0.9166666865348816,\n              1.8333333730697632,\n              2.7666666507720947,\n              3.683333396911621,\n              4.616666793823242\n            ],\n            "annotated": [\n              0.0,\n              0.9166666865348816,\n              1.8333333730697632,\n              4.366666793823242,\n              4.483333110809326,\n              4.616666793823242\n            ]\n          },\n          "frames": {\n            "uniform": [\n              "droid_natural_10bd743a70_b_00000.png",\n              "droid_natural_10bd743a70_b_00055.png",\n              "droid_natural_10bd743a70_b_00110.png",\n              "droid_natural_10bd743a70_b_00166.png",\n              "droid_natural_10bd743a70_b_00221.png",\n              "droid_natural_10bd743a70_b_00277.png"\n            ],\n            "annotated": [\n              "droid_natural_10bd743a70_b_00000.png",\n              "droid_natural_10bd743a70_b_00055.png",\n              "droid_natural_10bd743a70_b_00110.png",\n              "droid_natural_10bd743a70_b_00262.png",\n              "droid_natural_10bd743a70_b_00269.png",\n              "droid_natural_10bd743a70_b_00277.png"\n            ]\n          }\n        }\n      },\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ]\n    }\n  ]\n}\n', 'provenance/droid_natural_targets.json': '{\n  "droid_natural_8b01107c59": {\n    "a": "A",\n    "b": "B"\n  },\n  "droid_natural_6d74030dc0": {\n    "a": "B",\n    "b": "A"\n  },\n  "droid_natural_87abc314ec": {\n    "a": "A",\n    "b": "B"\n  },\n  "droid_natural_29137e19c7": {\n    "a": "B",\n    "b": "A"\n  },\n  "droid_natural_c00374385e": {\n    "a": "A",\n    "b": "B"\n  },\n  "droid_natural_19bf4a67ee": {\n    "a": "B",\n    "b": "A"\n  },\n  "droid_natural_10bd743a70": {\n    "a": "A",\n    "b": "B"\n  }\n}\n', 'provenance/natural_history_runner.py': '"""Compare uniform and annotated real histories at a matched ten-frame budget."""\nimport argparse\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport numpy as np\nfrom PIL import Image\n\nfrom readout_runner import auto, base, choose\n\nARTIFACT = "HERMES_droid_natural_history_evidence.zip"\nMETHODS = ("uniform", "annotated")\n\n\ndef load_frames(path, receipts):\n    with ZipFile(path) as zipped:\n        names = zipped.namelist()\n        assert len(names) == len(set(names)) and set(names) == set(receipts)\n        frames = {}\n        for name in names:\n            raw = zipped.read(name)\n            receipt = receipts[name]\n            assert base.sha(raw) == receipt["png_sha256"]\n            with Image.open(BytesIO(raw)) as image:\n                assert image.size == (640, 360) and image.mode == "RGB"\n                rgb = np.array(image)\n            assert base.sha(rgb.tobytes()) == receipt["rgb_sha256"]\n            frames[name] = rgb\n    return frames\n\n\ndef compare(model, system, case, pixels, recent_features):\n    import torch\n\n    branches = {}\n    for side, branch in case["branches"].items():\n        names = sorted(set(branch["frames"]["uniform"]) | set(branch["frames"]["annotated"]))\n        bank = dict(zip(names, auto.encode_frames(model, np.stack([pixels[name] for name in names])), strict=True))\n        visuals = {}\n        for method, path in branch["frames"].items():\n            assert len(path) == len(set(path)) == 6\n            assert set(path).isdisjoint(case["recent_frames"])\n            visuals[method] = torch.cat([bank[name] for name in path] + recent_features, dim=1)\n        assert len({tuple(v.shape) for v in visuals.values()}) == 1\n        results, repeats = {}, {}\n        for order in base.ORDERS:\n            results[order] = {\n                m: choose(model, system, visual, case["question"], case["options"], order)\n                for m, visual in visuals.items()\n            }\n            repeat = choose(model, system, visuals["uniform"], case["question"], case["options"], order)\n            left = results[order]["uniform"]\n            repeats[order] = max(abs(a-b) for a, b in zip(left["logits"], repeat["logits"], strict=True))\n            if repeats[order] != 0.0 or left["prediction"] != repeat["prediction"]:\n                raise RuntimeError(f"Uniform replay differs for {case[\'pair_id\']}/{side}/{order}")\n        branches[side] = dict(answers=results, repeat_max_abs=repeats)\n    return dict(pair_id=case["pair_id"], source_site=case["source_site"], scene_id=case["scene_id"], branches=branches)\n\n\ndef summarize(rows, targets):\n    methods = {m: dict(robust_pairs=[], correct_branch_orders=0) for m in METHODS}\n    count, replay_count = 0, 0\n    for pair in rows:\n        label = pair["pair_id"]\n        for method in METHODS:\n            correct = [r["answers"][o][method]["prediction"] == targets[label][side]\n                       for side, r in pair["branches"].items() for o in base.ORDERS]\n            methods[method]["correct_branch_orders"] += sum(correct)\n            count += len(correct)\n            if all(correct):\n                methods[method]["robust_pairs"].append(label)\n        replay_count += sum(len(r["repeat_max_abs"]) for r in pair["branches"].values())\n    uniform, annotated = (set(methods[m]["robust_pairs"]) for m in METHODS)\n    recovered, lost = sorted(annotated-uniform), sorted(uniform-annotated)\n    return dict(pairs=len(rows), scores=count, uniform_replays=replay_count, methods=methods,\n                recoveries=recovered, regressions=lost,\n                exploratory_criterion_met=len(rows)==7 and len(recovered) >= 2 and not lost)\n\n\ndef main():\n    import torch\n    from filelock import FileLock\n    from inference.llavaov_hermes import load_model\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "frames"):\n        parser.add_argument(f"--{name}", required=True, type=Path)\n    args = parser.parse_args()\n    work = args.work\n    with FileLock(str(work / "run.lock"), timeout=0):\n        export = work / "exports"\n        export.mkdir(exist_ok=True)\n        for name in ("summary.json", "pair_scores.json", "run_config.json"):\n            (export / name).unlink(missing_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(work / "completion.json", dict(status="running"))\n        try:\n            provenance = work / "provenance"\n            spec = json.loads((provenance / "droid_natural_cases.json").read_bytes())\n            assert len(spec["cases"]) == 7 and args.model.name == spec["model_revision"]\n            assert base.digest(args.frames) == spec["frame_zip_sha256"]\n            pixels = load_frames(args.frames, spec["frame_receipts"])\n            print("DROID_NATURAL_INPUTS_VERIFIED", len(pixels), flush=True)\n            model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n            assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n            assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n            base.atomic(export / "run_config.json", dict(model_revision=args.model.name, gpu=torch.cuda.get_device_name(),\n                         torch=torch.__version__, cuda=torch.version.cuda, manifest_sha256=spec["manifest_sha256"],\n                         frame_zip_sha256=spec["frame_zip_sha256"]))\n            rows = []\n            with torch.inference_mode():\n                system = model.language_model.get_input_embeddings()(\n                    torch.as_tensor(model.init_prompt_ids, device=model.device)\n                )\n                recent_names = spec["cases"][0]["recent_frames"]\n                assert len(recent_names) == len(set(recent_names)) == 4\n                recent = auto.encode_frames(model, np.stack([pixels[name] for name in recent_names]))\n                for i, case in enumerate(spec["cases"], start=1):\n                    assert case["recent_frames"] == recent_names\n                    rows.append(compare(model, system, case, pixels, recent))\n                    base.atomic(export / "pair_scores.json", rows)\n                    print(f"DROID_NATURAL_PAIR {i}/7 {case[\'pair_id\']}", flush=True)\n            targets = json.loads((provenance / "droid_natural_targets.json").read_bytes())\n            report = summarize(rows, targets)\n            assert report["scores"] == 56 and report["uniform_replays"] == 28\n            base.atomic(export / "summary.json", report)\n            base.atomic(work / "completion.json", dict(status="complete", pairs=len(rows)))\n            print("DROID_NATURAL_HISTORY_COMPLETE", json.dumps(report), flush=True)\n        except BaseException as exc:\n            base.atomic(work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/droid_terminal_cases.json': '{\n  "model_revision": "0d50680527681998e456c7b78950205bedd8a068",\n  "video_processor": {\n    "_processor_class": "LlavaOnevisionProcessor",\n    "do_pad": true,\n    "image_grid_pinpoints": [\n      [\n        384,\n        384\n      ],\n      [\n        384,\n        768\n      ],\n      [\n        384,\n        1152\n      ],\n      [\n        384,\n        1536\n      ],\n      [\n        384,\n        1920\n      ],\n      [\n        384,\n        2304\n      ],\n      [\n        768,\n        384\n      ],\n      [\n        768,\n        768\n      ],\n      [\n        768,\n        1152\n      ],\n      [\n        768,\n        1536\n      ],\n      [\n        768,\n        1920\n      ],\n      [\n        768,\n        2304\n      ],\n      [\n        1152,\n        384\n      ],\n      [\n        1152,\n        768\n      ],\n      [\n        1152,\n        1152\n      ],\n      [\n        1152,\n        1536\n      ],\n      [\n        1152,\n        1920\n      ],\n      [\n        1152,\n        2304\n      ],\n      [\n        1536,\n        384\n      ],\n      [\n        1536,\n        768\n      ],\n      [\n        1536,\n        1152\n      ],\n      [\n        1536,\n        1536\n      ],\n      [\n        1536,\n        1920\n      ],\n      [\n        1536,\n        2304\n      ],\n      [\n        1920,\n        384\n      ],\n      [\n        1920,\n        768\n      ],\n      [\n        1920,\n        1152\n      ],\n      [\n        1920,\n        1536\n      ],\n      [\n        1920,\n        1920\n      ],\n      [\n        1920,\n        2304\n      ],\n      [\n        2304,\n        384\n      ],\n      [\n        2304,\n        768\n      ],\n      [\n        2304,\n        1152\n      ],\n      [\n        2304,\n        1536\n      ],\n      [\n        2304,\n        1920\n      ],\n      [\n        2304,\n        2304\n      ]\n    ],\n    "image_processor_type": "LlavaOnevisionVideoProcessor",\n    "do_resize": true,\n    "size": {\n      "height": 384,\n      "width": 384\n    },\n    "resample": 3,\n    "do_rescale": true,\n    "rescale_factor": 0.00392156862745098,\n    "do_normalize": true,\n    "image_mean": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "image_std": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "do_convert_rgb": true\n  },\n  "manifest_sha256": "8422fcb786953ddb52ef7219b99cbe8bc21b78fdff4b6c00097fe6e2ebadb12e",\n  "frame_zip_sha256": "5bcfb613c5eb2cd8a40e306d94a79a3e77258ba162b47ddbb025ffc2a56f8b49",\n  "frame_receipts": {\n    "droid_natural_10bd743a70_a_00000.png": {\n      "png_sha256": "c40dcd205dfdc84678b00ae4def3837bc3bf892c2d326518368380ccbf7cc0c7",\n      "rgb_sha256": "9cae7f271288eea988b2d9747f5397c73886e223195482738e355d2dbb2eb455"\n    },\n    "droid_natural_10bd743a70_a_00077.png": {\n      "png_sha256": "9d2d9bf5ab51d5dd7d9ce3387d2e9347ddf78ec0ed4f35b57f1146dd2881acf3",\n      "rgb_sha256": "3c73da3d7090412b873fe9ee48a8d5c29f228cf78561788de577bad3df429cd7"\n    },\n    "droid_natural_10bd743a70_a_00154.png": {\n      "png_sha256": "5890316104b7fcb72e47702fb2a86c114f4d19440e094a9689e2d66fc5656bc6",\n      "rgb_sha256": "88973b702a8acb303b16b57e52600c653b447e8a468b486685ed169812eace43"\n    },\n    "droid_natural_10bd743a70_a_00232.png": {\n      "png_sha256": "784a33a8c6b63eb3d4d91dc246200749a4b504a3cdf32828fe61b70a92669e01",\n      "rgb_sha256": "9da2567884182a616b10f71074974bba7352eb700ac7eb2ec4679c9a1e64374f"\n    },\n    "droid_natural_10bd743a70_a_00309.png": {\n      "png_sha256": "eb1a20d09f180f85bd6de4bd002a04b85883be32c3add4d52d2d7d239ae00352",\n      "rgb_sha256": "c60f28e576699485fbee45de80cce28fc5af64a0cd0ce3a7d42f7dcaa8c38db4"\n    },\n    "droid_natural_10bd743a70_a_00358.png": {\n      "png_sha256": "efff16c14e8aef629c1264b1c5bfd5f5339556502f6c52384901a5e8031f60c0",\n      "rgb_sha256": "009e8977fd1f8b117c7b703a506d1a04dc842fdaf4b8811a5a2b8cce72dbe43a"\n    },\n    "droid_natural_10bd743a70_a_00372.png": {\n      "png_sha256": "75a33ac5bf97476258d10b5c0c4c0acc66d4a9b21cc928930e1de8063dab3b86",\n      "rgb_sha256": "9ac304b5618125fb87cc599f494bba2435fa84986f4619592e887460c29ea624"\n    },\n    "droid_natural_10bd743a70_a_00387.png": {\n      "png_sha256": "19bdd9e864469e353e49b14d43bdfaa38bd2d249feb3fc4683797d1269ef01d4",\n      "rgb_sha256": "1aac2f2260e929006821fe64d3268c3c99b93c969522075b890cc5bd4865b099"\n    },\n    "droid_natural_10bd743a70_b_00000.png": {\n      "png_sha256": "7022063be9422ee64814215568e4b9d5de604de4e8bcb408e7a9863e4aa7d18c",\n      "rgb_sha256": "8766bf61a7a3e2e54d359f0253869ca154c5bed18bdae2740d9931430acef535"\n    },\n    "droid_natural_10bd743a70_b_00055.png": {\n      "png_sha256": "6db741114bfd4a184a51c26e6a4e3762b58e7b790360caba6a8345523b98e591",\n      "rgb_sha256": "bee2f7670e4910b0581132320ba4561136b6090623ee2bdd33891739cb592775"\n    },\n    "droid_natural_10bd743a70_b_00110.png": {\n      "png_sha256": "ebe1f8641f814981d7ffaab3a64312d10a0e253aa58942da7b4825eea24125c9",\n      "rgb_sha256": "587cc06314220796732fcec5456411e2141e1daef1e466d7a81ee8489d7cdfd2"\n    },\n    "droid_natural_10bd743a70_b_00166.png": {\n      "png_sha256": "1844fe98ce7a9fc176509bef3a1afb719d6aea0236f35a40b6e34f821e07e4af",\n      "rgb_sha256": "6c6b11ac7a4a8e3fe6b46740a918581664bbd6ac327cc1923cd31685a7af7fe8"\n    },\n    "droid_natural_10bd743a70_b_00221.png": {\n      "png_sha256": "206e871f6ff30d2098f493828e34cfaf1ecee2850e079e6d8fead5a86360327d",\n      "rgb_sha256": "bba4a33f142259e76852dc02fba72e9c371a925dad61cbe5e14a07a5ca3925f4"\n    },\n    "droid_natural_10bd743a70_b_00262.png": {\n      "png_sha256": "a5bf3458ba10b38b7fea4180793823100eb0134d32ef94972d7493e414d8b5e5",\n      "rgb_sha256": "c9f231b917cdde0017e5ccb3d5093f0d94f15df2fcbb8aaf5d2ab0e656134998"\n    },\n    "droid_natural_10bd743a70_b_00269.png": {\n      "png_sha256": "208bdc06bfa835c41f80f6deb8e49d9777ee33b2b48feee22d2108f5e64d51eb",\n      "rgb_sha256": "83288ad9ec01e4342f1e9c721eb58c126f9d68e53c01a770f677c627ee6818c6"\n    },\n    "droid_natural_10bd743a70_b_00277.png": {\n      "png_sha256": "3a6137ef5b040122124c2d7c0dcb0a4dba7f59cb43f01807addd826b2b116f1b",\n      "rgb_sha256": "7915e7b1ee3138e4eb7dc44456df83fedd18a152089e5e112e70111edfb0bd34"\n    },\n    "droid_natural_19bf4a67ee_a_00000.png": {\n      "png_sha256": "debacb052771f58ea21515c68f72bc67af3b8ce92178ceb6781beed0cc977efd",\n      "rgb_sha256": "03e96cfaf5854212720759e6b3e269af51a2833ad1818bb7da4a26881cbd9b7e"\n    },\n    "droid_natural_19bf4a67ee_a_00036.png": {\n      "png_sha256": "41b550d205ebf14c08561a7d91750bc49b818997b9e306c56be89a9e29d1cc22",\n      "rgb_sha256": "1cc79407e83482563f97a8e8945451159089153b058596b702dd0b50c865a9a3"\n    },\n    "droid_natural_19bf4a67ee_a_00073.png": {\n      "png_sha256": "6f0e2d2e55bd2491a63289a570ea772c3342c3fec8726e6c2d507ed56ee9cc5a",\n      "rgb_sha256": "1a3084049b7c6ee28b6c4bb756fd5a11ff13aa00738dc2661ab547f1dbace6b8"\n    },\n    "droid_natural_19bf4a67ee_a_00110.png": {\n      "png_sha256": "891a2198514c7ddb45abd73a156b2ae1c8f40e01acdd45a83127d19b8542fd56",\n      "rgb_sha256": "2385a7cff31db995056771e4fe15e7e4a6c55016b36dcc177b24eea9b29b565a"\n    },\n    "droid_natural_19bf4a67ee_a_00147.png": {\n      "png_sha256": "d619d0f4f87029c411de358067d24e56c015fb3f187e7e5a5826f6740d2c1278",\n      "rgb_sha256": "fe21833d6f7c4337ae8d209fa8bf4bcb0959f7bec3ee3460709044cf9db9609c"\n    },\n    "droid_natural_19bf4a67ee_a_00162.png": {\n      "png_sha256": "d1e8c26faff85f4a6b3ab04d016102d13fb0da1b0b4d646570806c986e1e02ce",\n      "rgb_sha256": "090a1255bd93e7f5da90f75f228def4d15fc4d966289b29ef0b631c9d89ceb9b"\n    },\n    "droid_natural_19bf4a67ee_a_00173.png": {\n      "png_sha256": "9fc98d98e6afaefcf2d04b9264ebf0b01b9e6a8a10131073557d1131d802a22f",\n      "rgb_sha256": "8e987aa7ad1adf52f8405b3db2cb35af45e3a61ea49ba5523fa6c49d5918c81c"\n    },\n    "droid_natural_19bf4a67ee_a_00184.png": {\n      "png_sha256": "0421d613e6efa7fa05f2232790753de29365c1864ca49ebd9353887ffbe2ce3c",\n      "rgb_sha256": "fc04240b13efe4dc8f93fe327c44015b8bcf290fd42a370378fc3b84d7485cae"\n    },\n    "droid_natural_19bf4a67ee_b_00000.png": {\n      "png_sha256": "b0efeb4463a1a1908e6b0d95f1276e7fe2c558ee9e757c01802ed5d45f7f7896",\n      "rgb_sha256": "f1848d98397bd2d656bb222001b9f2a02e37f8bbf63677cc803f46049f2d1648"\n    },\n    "droid_natural_19bf4a67ee_b_00062.png": {\n      "png_sha256": "6a13adc9415abc3667554d73bc452161beaf566820a29a2877c03cde640cff0d",\n      "rgb_sha256": "6ee37b935ad061df5a237c000fe306b6bf68361618b9cf1edecfaca76f6755ab"\n    },\n    "droid_natural_19bf4a67ee_b_00125.png": {\n      "png_sha256": "6da9f23967d312bc9b4e5b84e3e203272c5c3528320bff24b14c20a084ab3483",\n      "rgb_sha256": "7096ebfb5e1eebfbc71b7d5446921ade4bc2cbd703cff0517895a7cc61e74fea"\n    },\n    "droid_natural_19bf4a67ee_b_00187.png": {\n      "png_sha256": "2983a72e0e95eaa2f16cb2e99608b79346a65561628e146ef5280977cb14d493",\n      "rgb_sha256": "affa452bcdf880c6ee77f4e7396a953417f22bada5431085f4bded12ae546488"\n    },\n    "droid_natural_19bf4a67ee_b_00250.png": {\n      "png_sha256": "3eef60b01149d24d4b6f2ea1ded300850a8727cec60be43e9da877b5d7a3e711",\n      "rgb_sha256": "931348dcbe5d1e58970f01ba8da70fc68024abe858b875eeaa5c836d5ba88512"\n    },\n    "droid_natural_19bf4a67ee_b_00273.png": {\n      "png_sha256": "c2d721dc09522945f103589157a3312d269b500950f3ea92ac76c08a992ee2a5",\n      "rgb_sha256": "5b37d7337dd6ad03f402620875659878360c1336bdb2d3ec3074e7302b9500d1"\n    },\n    "droid_natural_19bf4a67ee_b_00293.png": {\n      "png_sha256": "338236c75313aae9f0dd4f3958a273077129b6de1891b4c26abc6d1e7904cdfb",\n      "rgb_sha256": "f8498a618758fcc0b4e038bcafaf6beaea22695372b123807b215069d38bf8af"\n    },\n    "droid_natural_19bf4a67ee_b_00313.png": {\n      "png_sha256": "474e8673e45b02f833babcc99d85f89a8477d62c38521bfbc1453d0719175409",\n      "rgb_sha256": "557b6ac3ffd004804229a55b6eb0b1a02767ce2dead1e53c1e1096cd879ad1f7"\n    },\n    "droid_natural_29137e19c7_a_00000.png": {\n      "png_sha256": "ca31407621e458a5dd08c7efd5572869bb9f4abee4ad67df71ddacb021cc1647",\n      "rgb_sha256": "b4fb7547434a3bab46eac6a5be94eaf227753f19173cf3ed223e69ed59e1d048"\n    },\n    "droid_natural_29137e19c7_a_00027.png": {\n      "png_sha256": "dfec0b22a96c9a87cc3187b5cff6a5f54e189c4d61c12e7d2945ffd5b9ae2efc",\n      "rgb_sha256": "3f48f342b8a284b81939b539df1fa96c2298a465e4468ba8d89ae7a09ed209e2"\n    },\n    "droid_natural_29137e19c7_a_00054.png": {\n      "png_sha256": "5299b74c8118f2f0dc73d5b6d1b6d787541163a90fad6ced0730efc61476e4ae",\n      "rgb_sha256": "d68e0c97147844e1a261e1b2e81ab75c4e73bd37f34798d380b36ceb25508b85"\n    },\n    "droid_natural_29137e19c7_a_00080.png": {\n      "png_sha256": "1615b37bad86dcbd5f4f5596b69ccc6ce1b289587a93b8ede836c1f2739b2373",\n      "rgb_sha256": "12d4eb8dbec808715c0dab84d9221c664e1c317a93e0c6022028c48c9d178d09"\n    },\n    "droid_natural_29137e19c7_a_00108.png": {\n      "png_sha256": "96bbfd509f43af7d3633a59a56584507627e68fe327f960a12f90e8aec605ebf",\n      "rgb_sha256": "08cf5c5bb858332b68cf96817e1c438278135c8d3b524e56763af0a7cd3b5249"\n    },\n    "droid_natural_29137e19c7_a_00131.png": {\n      "png_sha256": "a46588ca6b9c50a8f7a964dbf468dad50759dde3a0f56729c06a4af179da298a",\n      "rgb_sha256": "bf29320865420438dc3a54e2c4f0ae523ee03c99464fb896329fdb544389dfdb"\n    },\n    "droid_natural_29137e19c7_a_00133.png": {\n      "png_sha256": "c186cada9fcdce37792edbd8d90e1349641764741884734ed830abae0f0cdca7",\n      "rgb_sha256": "1bd8fce85f7157f4e74bed7006e2d8d5f5ce02fc060db51d70ccce6f54ebfeaa"\n    },\n    "droid_natural_29137e19c7_a_00135.png": {\n      "png_sha256": "70d9bafb7a7dd414cdab1572f88ceddad01c9803870ec405de8e7e04163be761",\n      "rgb_sha256": "31ccc8ca1510b9c5cf2ec7a14239fa28a4a1c0d20c6951b992d0738b7e13ced9"\n    },\n    "droid_natural_29137e19c7_b_00000.png": {\n      "png_sha256": "c01a7861a3fa580cdb9e6c8c3a2589cca7cfed92a71e179e9958ffd60e45f37d",\n      "rgb_sha256": "eede46b1066b617b26368eb7fff9ec58f2d5eac7142be9deace2c6b5daa43efa"\n    },\n    "droid_natural_29137e19c7_b_00041.png": {\n      "png_sha256": "b46d08c602aa6b8e1c376f18ae4634a8a854a31b1dd00b7adbaa6352bdf5fa5d",\n      "rgb_sha256": "11b45e94603c7c77748673d155964eeb5344c04462a514a4208a66f6c406c332"\n    },\n    "droid_natural_29137e19c7_b_00082.png": {\n      "png_sha256": "e915422fb15019ba905ecf14f196a235b397920a3bcb81cdb06056bd5e0f6b2f",\n      "rgb_sha256": "50b52e151062cfef792a246c35c2e159b9cf33e7c91eced996485d2439e28ca1"\n    },\n    "droid_natural_29137e19c7_b_00123.png": {\n      "png_sha256": "e8ded427979f5b1a34eec995fe286bbaef8a59140344bbd4dc3a49b8244232cc",\n      "rgb_sha256": "e28c94e20a0861b3ff7e2e661c292adca3eb44f5dadde9e18bc0904f4af3f184"\n    },\n    "droid_natural_29137e19c7_b_00164.png": {\n      "png_sha256": "65d6f2e35dceb2335970211d0c4f4beae28245e4453ad0b7f818be9feda7ede1",\n      "rgb_sha256": "bb37e534aa4efb88a147f5667022ef42800235977a4224829d83abe663b6fbe0"\n    },\n    "droid_natural_29137e19c7_b_00179.png": {\n      "png_sha256": "782339befc2d1c3cfe862df2849b551ac9151ef95d9099902703afa12f4a32e3",\n      "rgb_sha256": "0ddc8a2455eac4a57aa79cb9626dff020f94e56e2dcee32906f1f8d9ae2ec158"\n    },\n    "droid_natural_29137e19c7_b_00192.png": {\n      "png_sha256": "d037b3b1650fa8ed98a47dbcae267d65508a8c15b1a7b7d68cb2db70f911efa9",\n      "rgb_sha256": "bb1f2c704f7212f328945bc52d7e028985a85bd7775f30ba08833e04ef0ca22f"\n    },\n    "droid_natural_29137e19c7_b_00205.png": {\n      "png_sha256": "a164585443a01aa26f239ee529fb48254262d131d9a8e18dbee8976ed9ddd50a",\n      "rgb_sha256": "835fa93fce76e5fd978dc5efc5392a71b07ae2ed12e7f8da1af1f90eda7e6985"\n    },\n    "droid_natural_6d74030dc0_a_00000.png": {\n      "png_sha256": "bb15ff714e5d3b54497be4c24f6371dc36971708106c67e671a603ff617ad0ce",\n      "rgb_sha256": "4a57387d306680c976e0d51fdba04b69c8d4644dabee9f8ca5a97e2384f913e0"\n    },\n    "droid_natural_6d74030dc0_a_00034.png": {\n      "png_sha256": "adc93dbdd788b7cff8e90a8c0131bd74b1816e80677dd456775ee2f445958f21",\n      "rgb_sha256": "dbd5f88e3ea545b4314228c98a57fb355b180d84ff9b865019c6676d130957f3"\n    },\n    "droid_natural_6d74030dc0_a_00068.png": {\n      "png_sha256": "a661332ee34316fc64bcf6d973afc3f6d7b23d42d6e11f716b5df671379294e2",\n      "rgb_sha256": "8609f4fdeee8a51e93f165af3719576bb3d895ff1fd3fae4a0333429dbd2e38d"\n    },\n    "droid_natural_6d74030dc0_a_00102.png": {\n      "png_sha256": "c0c354ee8c93ace7f2034e479b5047b9cf45c9fe5cd81cdd4cfadd18d554b344",\n      "rgb_sha256": "fb5aed1d0d32f45efe5594dd2d47ac5a6078fb6f30601b697bf23806f36c9f59"\n    },\n    "droid_natural_6d74030dc0_a_00136.png": {\n      "png_sha256": "eb143b4a6187ad247a25b202e4899d47c7b080ea702d244ac24e969206a28b23",\n      "rgb_sha256": "652398bc62eaefc57362e4184011e95fd961c21beec075858bebae20c0e56ff1"\n    },\n    "droid_natural_6d74030dc0_a_00149.png": {\n      "png_sha256": "8cdf0806c6073e4d70b140ecb827e27c93cf82f29cbe770a399df24fbdf44183",\n      "rgb_sha256": "536e4ad3c2662ce50663a4c4860815916d45091d9dc78b930ebd7eeff707b70a"\n    },\n    "droid_natural_6d74030dc0_a_00160.png": {\n      "png_sha256": "088e4f4467a484abc80ac2594eba41a9b295c0a0d88f4827b4f03455da4146e5",\n      "rgb_sha256": "6d317bb548c3098314518cd88c74fa078c5cc5aa4c460e7a9657fe5bd89028cc"\n    },\n    "droid_natural_6d74030dc0_a_00171.png": {\n      "png_sha256": "1eecbbec9e9cc18a89468ebe169fed8f08c09a19a453dd463f98da2d41e84c7f",\n      "rgb_sha256": "714d7d2deb3e42ee3c704c80325b1aca3c363736fd93f23fb953c4e4fd9dea8f"\n    },\n    "droid_natural_6d74030dc0_b_00000.png": {\n      "png_sha256": "61983c8134f394fde7d7c5bee766777c8c8c88f5609275211a0919f7ba0eaa35",\n      "rgb_sha256": "f9a08a6a97bd2672583adb67daaff6f32bbc85deb1e39b89e8a823d0b5d9969a"\n    },\n    "droid_natural_6d74030dc0_b_00055.png": {\n      "png_sha256": "2a3a0dcedc2ccf2af785f314cd4c257e8b4f4f0e9967101cfbcf677c9f2853b1",\n      "rgb_sha256": "c10c9e4ede738994869d74607aa7fb096ee9ecfa5568ff4ffb6e8a498db1fa3a"\n    },\n    "droid_natural_6d74030dc0_b_00111.png": {\n      "png_sha256": "d7ad81ea7275446448072d6914bb077f8c9ff34a0e16ee75a5ba4b8a478b177b",\n      "rgb_sha256": "86405e22736d6c681bd9f467d269fee64cc5911aba411056c8a31f13ccf9a913"\n    },\n    "droid_natural_6d74030dc0_b_00167.png": {\n      "png_sha256": "5724dc13865a4a01157d7e4d2f84a2a2c9d044ef84b20bdc8ec0b7a2b60f8280",\n      "rgb_sha256": "5d91b643b745784ed753eec75a87171f3b4257d8a3f7ac97f4fc22ef9687cf0d"\n    },\n    "droid_natural_6d74030dc0_b_00223.png": {\n      "png_sha256": "ab59fb615d70e7ebd2a88d0d7297caed778a313ddd684afd484c40475b5e42a0",\n      "rgb_sha256": "481e05555aaac70cb96c37c51152fdd087ca257242ad147b816b18d8bcdc49b7"\n    },\n    "droid_natural_6d74030dc0_b_00229.png": {\n      "png_sha256": "436dba79a75c7fd5535f6fdf8313130b7ca5f08b0e3799cb2c7bf6064c9d37ba",\n      "rgb_sha256": "feab429de009c541c7cac29a0208a1e9ac01e922a731a88ecd06093c86da1a98"\n    },\n    "droid_natural_6d74030dc0_b_00280.png": {\n      "png_sha256": "12fa04c9ed7ada5c9621ba2503f9a8f03896c95540110218771c2a205aa5e6ca",\n      "rgb_sha256": "3ce83eff61a1a3609b1c0793d8433626ae5d5e2558b220ca0c0092b566e5d7cc"\n    },\n    "droid_natural_87abc314ec_a_00000.png": {\n      "png_sha256": "b80438ecff621713c84b51dd09c48dbcf09bf0d0175482116a39349bfadd8486",\n      "rgb_sha256": "f55789640ab2d5fd3e071f571725e6bd8d322f3190dd21e07fac56ff97ba475a"\n    },\n    "droid_natural_87abc314ec_a_00063.png": {\n      "png_sha256": "dde0ed8a9fadae6f672c3bfd16e6d8c1857a0d0b78f02f7409b8490034ca25a9",\n      "rgb_sha256": "314e2b4d0f0516a12740f15e6acd50203841aec1197703b65c3cad50e0653d12"\n    },\n    "droid_natural_87abc314ec_a_00127.png": {\n      "png_sha256": "c9a1cb1e14a3895c5e5054f6693d19ecd01c978c01017aac1b184d21d49d2029",\n      "rgb_sha256": "d7cd11dadbff2b5546528d616399acf665137b7e455064ea9f48aa12cd72481d"\n    },\n    "droid_natural_87abc314ec_a_00192.png": {\n      "png_sha256": "de7b5d3ccefa43b5cf7205b4bd3e2a9bee44ef144c1359f39ca840f2ad1efbb7",\n      "rgb_sha256": "4a83f6e8eb49d620e2e73c59bd319d90706fd0645e19f873341f3729622a0f44"\n    },\n    "droid_natural_87abc314ec_a_00255.png": {\n      "png_sha256": "b36bc3a87b2e113191d2dc63bf4003b0948e8addb4746bf865fe0dc3f0ad1230",\n      "rgb_sha256": "a1ce2804442a71ac4a922ecf62a87e3e964d512b424cfca18e4d88b270491081"\n    },\n    "droid_natural_87abc314ec_a_00279.png": {\n      "png_sha256": "ec3a2f0286439f91c5a221a32030a178873cf0f889e1ba86f3110ec7b9145871",\n      "rgb_sha256": "1c363c77b881d41a20d26225af05be5049e2a7a247ac8ea84e5acf1ea53924a9"\n    },\n    "droid_natural_87abc314ec_a_00299.png": {\n      "png_sha256": "c3243f8cdd63072bd4b5238be0d97d9b76a51cd7c97352fa769e0d7d6d70b447",\n      "rgb_sha256": "3da10f297a5ddea4ddd35b57ad88444d00e825dcb8de1890226fa2f639814e91"\n    },\n    "droid_natural_87abc314ec_a_00320.png": {\n      "png_sha256": "97638fe9b9c8784144f3ff589bddcb86fc8c737dc935b810030f349d7564e8bb",\n      "rgb_sha256": "3b7753bc7e2fef1b561e67aeea63010a94af0b37fdbd13599f7d282b9a8ce97f"\n    },\n    "droid_natural_87abc314ec_b_00000.png": {\n      "png_sha256": "b901c7d9ac0744bf121520278241d043c98d5dce173de808445078ba78ad8764",\n      "rgb_sha256": "07f653c8066e319d17105cb7a496214d690bf89c3b728a83af9c0f71c245ff74"\n    },\n    "droid_natural_87abc314ec_b_00030.png": {\n      "png_sha256": "68c7f689f41ef0c679d02e8e1b30f0d1c7e39df5e49d54dc4018488b7e0b585c",\n      "rgb_sha256": "17e0dc3f7af20cc0b0b87982dfa57cc912a4b28e2930c6703ece36835683f018"\n    },\n    "droid_natural_87abc314ec_b_00060.png": {\n      "png_sha256": "e1e3cb512c67329d65bacfb04aa489f7dba5261e59a1e239ee1551899f8daddc",\n      "rgb_sha256": "e1058907d3fbe663a54a8f53a6ba98cd93b5b685b37e8fa0f25db2e7f1756695"\n    },\n    "droid_natural_87abc314ec_b_00090.png": {\n      "png_sha256": "525d6142d8619e0cd9d9eb4ffe2beb2c89db5249044f6171e652ff61ebfd42db",\n      "rgb_sha256": "b215a98ae4d33ad75c1dd0ba15bdcacccb7358612322adb66e4996ddddcabaff"\n    },\n    "droid_natural_87abc314ec_b_00120.png": {\n      "png_sha256": "4b1a74048c68bc7f42c475ae6e1f45635e70c2ebe76a7a793cf8fc7e91d1c60b",\n      "rgb_sha256": "553671e325bb730c2986c350e6f57e6c37fe2b8f0b9355b329d3bfae0431d763"\n    },\n    "droid_natural_87abc314ec_b_00131.png": {\n      "png_sha256": "df5e39a020991bfacb60fc84ecf4557b6d4d465594ea2bb7700e7c57d0fb3359",\n      "rgb_sha256": "d840989e0548a7155c6765eb4c855ebd43488ead181f557c41e2fd9610914eaf"\n    },\n    "droid_natural_87abc314ec_b_00151.png": {\n      "png_sha256": "44be1519b81ca0e8ef198942af532fa0aa6118b731b9ec7298df0878e37f451f",\n      "rgb_sha256": "aeed76fd9d7e8f09d5c45a001b278ff75da5b0521416a82875553da29f1b91a9"\n    },\n    "droid_natural_8b01107c59_a_00000.png": {\n      "png_sha256": "17092527011e395d0feb686df9a05d3ecc38dba5cd94e9d436280baaa2ebdb47",\n      "rgb_sha256": "8767a47b9592203e4112d0cc979f5be77ba0ca64b7ffb1ec442e4596187c7d9e"\n    },\n    "droid_natural_8b01107c59_a_00051.png": {\n      "png_sha256": "89094a842cf2c88ef631636087b30ef2023ffd4e192cc077672f5811d9455758",\n      "rgb_sha256": "774194d6b591fab01b63b960c686f8c24109b7da39912644d264d19fcd42077c"\n    },\n    "droid_natural_8b01107c59_a_00102.png": {\n      "png_sha256": "cf252ecc092c4ba54b78406a52ec3cfcf6ed6c54e2e2c7bce783e68a269a008a",\n      "rgb_sha256": "2ca0a173c6eb65c63fd7f67b01e2b35bb60ea3d93df8d615dbd10705ee28e974"\n    },\n    "droid_natural_8b01107c59_a_00153.png": {\n      "png_sha256": "ccf10281877c3c1a02a0b187b591551c4a9836c491afea27524591afa5e32cb0",\n      "rgb_sha256": "71b73f81098ab14d5116cd4451aa6cd27ae9b449ecaead37d5b4ae7f4b62df68"\n    },\n    "droid_natural_8b01107c59_a_00204.png": {\n      "png_sha256": "921c5570ed4fe4bcc5928990548ef2649d3ee71737e0441d53dc4be92fcb8a59",\n      "rgb_sha256": "64955dfa6a305c8741650d00649948d4e787539b2e2e447064532a7ee5f60777"\n    },\n    "droid_natural_8b01107c59_a_00238.png": {\n      "png_sha256": "c1fe4fdd51fab29afda43da148db7cc6205a342907e4737393b13b72b06dc30f",\n      "rgb_sha256": "4411633b2a031a2aa903e37fc4f696abd968709cf7285cd9a9555a2b9fe11276"\n    },\n    "droid_natural_8b01107c59_a_00247.png": {\n      "png_sha256": "207d8ae730ac5acf4e59d2173ec7b639be595406caee095482fc7052aa6cfb50",\n      "rgb_sha256": "d9e04be59aacf03d404a58cab1762d200e0c3aa4137c70fa8c6f8464150a4953"\n    },\n    "droid_natural_8b01107c59_a_00256.png": {\n      "png_sha256": "00fd65554fdd3f8e485a12a0b59a8a0714f2519c067266f60ce8b0752ef02ce3",\n      "rgb_sha256": "e2e0fb7938b39beb6d177d4080b768640631c19d436c71625a5fdda7879e4c43"\n    },\n    "droid_natural_8b01107c59_b_00000.png": {\n      "png_sha256": "78c07fc56b04831ea95b5d0c495184a834db622c8b6092d9c73584a5be49fe47",\n      "rgb_sha256": "563c57dc0ee324dab566e17aba9b3cc8454c7b0711cfa7cb82ed0240fd8db275"\n    },\n    "droid_natural_8b01107c59_b_00035.png": {\n      "png_sha256": "c32b22bfceac6d1849c111f70b2014c8c98c06790af7c0e4034a06be54af8cd3",\n      "rgb_sha256": "25068a9d08a5fbb2af8627b48e7d3f9d05a5d0cf2c6b3ae6f28b6b8121f78094"\n    },\n    "droid_natural_8b01107c59_b_00070.png": {\n      "png_sha256": "3a1e921d084620f3faf8117ad79133eff428d45e2377016d4f67cf537800c947",\n      "rgb_sha256": "b8bb0793248358a78bfaf26df23ddfa4823f3c01dd88fa1febe30be244fd744b"\n    },\n    "droid_natural_8b01107c59_b_00105.png": {\n      "png_sha256": "b356cd0a17f022fdbd56b1ac708462bf44e7eb80caf4dee425f8a6e6a28af688",\n      "rgb_sha256": "003196286608593cbd6426fc5edcb1ff95d37a89336149eaf6f693eacad15328"\n    },\n    "droid_natural_8b01107c59_b_00140.png": {\n      "png_sha256": "80f2d03ef670ea3c0a24c53dbd3649288d0bb89499504c6d31a55a320ace328d",\n      "rgb_sha256": "887d203d9cbebfdf25798584881d53f6bad12020ded2bb049d46e4ffa8d0a89d"\n    },\n    "droid_natural_8b01107c59_b_00167.png": {\n      "png_sha256": "408bd7f29ea155346dfa5b8c9563bde5faac5b24e62e9175b140272d61063b92",\n      "rgb_sha256": "200bdfc090e8600d815055f0ccfce1ae5ff0e81fe5270d6815f09b379c18ec89"\n    },\n    "droid_natural_8b01107c59_b_00171.png": {\n      "png_sha256": "f33603d641e1fef3eed8eaa2bd875cdf154c51554617de63925b2322462c61bf",\n      "rgb_sha256": "d27b7ff452888572826e006ca08dbd15b05efba660a2798170c2107fafca447c"\n    },\n    "droid_natural_8b01107c59_b_00175.png": {\n      "png_sha256": "c43d4f69df774eefb6619ededb515d662527ebd9ec86e6c1d17d32f449990588",\n      "rgb_sha256": "f82aebffa2374e9b6e54e836fb6e4ea53aa9894c6e5640c15343c69691ce2519"\n    },\n    "droid_natural_c00374385e_a_00000.png": {\n      "png_sha256": "a8d7e3752f89f017fc999590ec22f012589ef65b4688ad2d0a75bc0702f6efb9",\n      "rgb_sha256": "25c93b20570a2d8d95836a2cfab1136f95ab1206a610c662b6262c8a987f38d1"\n    },\n    "droid_natural_c00374385e_a_00065.png": {\n      "png_sha256": "5a51251bb40406019e504babe3dda2bc2d47acb03a1e1fc83f8085948341442a",\n      "rgb_sha256": "adf413908710f43157dfdcbe667b0b7fe90590cad06ee946193f649fdf8776b7"\n    },\n    "droid_natural_c00374385e_a_00131.png": {\n      "png_sha256": "1f8ed7681bd83f232333bd66a30f9dc4cf6bf0ae2282abe6beb35f07cabf7e24",\n      "rgb_sha256": "334318dce375e6590833a55867734b85054354ca05fd1f9907253d7477dbdd75"\n    },\n    "droid_natural_c00374385e_a_00198.png": {\n      "png_sha256": "5aeb71da5d006288b405b42f97cd306066590e8edb744552a5d9b12adafc0433",\n      "rgb_sha256": "17b3311b595d9828e9c57c11fcf5d9cd5e74de71a2f473a0e4ccfd471a8c526c"\n    },\n    "droid_natural_c00374385e_a_00263.png": {\n      "png_sha256": "af11d36200aa3fa974d6d03e5c863f690469121617d10e60e18b6024a85adf0a",\n      "rgb_sha256": "d314f3a307fed1722b9c0e128eef304a5ae324c0b97ea3e8085d5d29871f0e73"\n    },\n    "droid_natural_c00374385e_a_00300.png": {\n      "png_sha256": "eed7605e632ff2cf0b6776d47d9337c7ff6a93f8124055d57a7e1c9a7f34b26c",\n      "rgb_sha256": "df78a12045202629bedbe5748aeb75db96637fcb14739ee97dddf5f7d50528f5"\n    },\n    "droid_natural_c00374385e_a_00315.png": {\n      "png_sha256": "c8d1fefe6b1ba65cdbf2f9672fe2c8496432746ea86ae1a7796fc1ec6b0d2d3d",\n      "rgb_sha256": "e1933e82c4783e1a700b295b0bb8877cd01de8b692107218a5fbc9d611ae19a4"\n    },\n    "droid_natural_c00374385e_a_00330.png": {\n      "png_sha256": "7ffef89523e769d9cad5d3587aa858909ea32dc669f263bb42718dfce02a9ff7",\n      "rgb_sha256": "53c3ff287b2a842433b5dd9bf51cd7486d640e33eb1fec53d1e273e055b4a8ed"\n    },\n    "droid_natural_c00374385e_b_00000.png": {\n      "png_sha256": "3d7da22cb2fa8f053661afacdab2a93a3d5468a3d95fcf56b8892acbbc5d9467",\n      "rgb_sha256": "36f3caf5219cec4c584a8b474a91decc7b71bdf45b150e3e7cd14cf6bc0e6e1e"\n    },\n    "droid_natural_c00374385e_b_00044.png": {\n      "png_sha256": "ce5ce93c6f18b45a8c670d19d3108edd8340cf7f0f4fdbbe804405f54484c5fe",\n      "rgb_sha256": "93f3fadedb4712056bc33a97983c7734d916922902ba7a9f56cb157fa63a216a"\n    },\n    "droid_natural_c00374385e_b_00089.png": {\n      "png_sha256": "bd65a846779f56bb92aa21a70c782d544754b7642551fe8a14e76d6132488f36",\n      "rgb_sha256": "5583e6fb1173ca27599bc4c89fcd187062085623b9004a3d713b3d771875e742"\n    },\n    "droid_natural_c00374385e_b_00133.png": {\n      "png_sha256": "4a9ef125787f62c5b0f5f2c80f3a1c328386e219a958b0c4e48be72cd24cef33",\n      "rgb_sha256": "047aa5bb6194d1fff6341f220b927a098ba9b47863d10ec26b9716b95da72f48"\n    },\n    "droid_natural_c00374385e_b_00178.png": {\n      "png_sha256": "32f459885bc88cee62b662853e7a3b40e8f7ee771d8bd7b06f13a441a7c9a316",\n      "rgb_sha256": "f14623a5c21f27bc072ba0d85aa3331a8b58914aea516913c74e17025979a2db"\n    },\n    "droid_natural_c00374385e_b_00194.png": {\n      "png_sha256": "4a8035b472109e037fd5ab39a85b012f8407f49a164dbf9219fcfe4599fbd171",\n      "rgb_sha256": "08d1193138eebdbcd99384cdc0d1d81832fd213c1bd64e512c37d783816d5bac"\n    },\n    "droid_natural_c00374385e_b_00208.png": {\n      "png_sha256": "a899fdf821a8954ec37020d7f9688412d53a0ad087b8f76f842f7dde5aeda2ef",\n      "rgb_sha256": "314fffb38910b2408fa9fee9af11451c1b273671b3453905fbd7dba6cda499e5"\n    },\n    "droid_natural_c00374385e_b_00223.png": {\n      "png_sha256": "02704cc8f932408e6542e7866a29a0c4d1a9c3dccb8fd0ed7c13b3678c35517c",\n      "rgb_sha256": "59443b34d012cee0f8c9d939032036e948ee66e1daa4b214a6499cc96bfe933d"\n    },\n    "droid_shared_recent_00015.png": {\n      "png_sha256": "c73295de2b63a2b7691ce3c4e7b0c25f7a93e06108bc3c850a66ac433aad1d79",\n      "rgb_sha256": "3d3a2dac71d987d075e9bbe321ca72f089a7b2d6fc3a159f64bf546029aefcb8"\n    },\n    "droid_shared_recent_00075.png": {\n      "png_sha256": "1bcb55ebbcd76a9671f4a826a46d38dd41e8514a0966e9b40bcffecc3b8f0ecb",\n      "rgb_sha256": "8ce787896e70698d3f30417a7fdbe3c487deab536488b7e212914481f7ada688"\n    },\n    "droid_shared_recent_00135.png": {\n      "png_sha256": "29563eb3bba0569b23ee90dc52d8dfb9bf4df41b5efcf151f42937bce7493fb6",\n      "rgb_sha256": "0090eadb9ac1f7478ac4313f5d41e127f0bc76b86b2db3b0bd02e69ee23fedbe"\n    },\n    "droid_shared_recent_00195.png": {\n      "png_sha256": "0d64158465ead4163d81607bc0b7edfcf7af3b829b9673777c44b8f9c7a0d8d5",\n      "rgb_sha256": "2fe28128f70b39462570ae0a3fe194314486d9336105fe500f8c428fe72c420f"\n    }\n  },\n  "cases": [\n    {\n      "pair_id": "droid_natural_8b01107c59",\n      "source_site": "IPRL",\n      "scene_id": 1611836543,\n      "question": "In the last view of the wooden drawer to the left of the stove before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_8b01107c59_a_00000.png",\n              "droid_natural_8b01107c59_a_00051.png",\n              "droid_natural_8b01107c59_a_00102.png",\n              "droid_natural_8b01107c59_a_00153.png",\n              "droid_natural_8b01107c59_a_00204.png",\n              "droid_natural_8b01107c59_a_00256.png"\n            ],\n            "annotated": [\n              "droid_natural_8b01107c59_a_00000.png",\n              "droid_natural_8b01107c59_a_00051.png",\n              "droid_natural_8b01107c59_a_00102.png",\n              "droid_natural_8b01107c59_a_00238.png",\n              "droid_natural_8b01107c59_a_00247.png",\n              "droid_natural_8b01107c59_a_00256.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.34375,\n                21.796875\n              ],\n              "prediction": "A",\n              "prompt_sha256": "81267a78cc998dcd581e71fd488663fc635d879268bcbb19ae9cfdf7a5c7af11",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.359375,\n                21.140625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "724aa539f2aace597cac1b08a569e40eed3aa838958c90db791afe0d0b56b09c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_8b01107c59_b_00000.png",\n              "droid_natural_8b01107c59_b_00035.png",\n              "droid_natural_8b01107c59_b_00070.png",\n              "droid_natural_8b01107c59_b_00105.png",\n              "droid_natural_8b01107c59_b_00140.png",\n              "droid_natural_8b01107c59_b_00175.png"\n            ],\n            "annotated": [\n              "droid_natural_8b01107c59_b_00000.png",\n              "droid_natural_8b01107c59_b_00035.png",\n              "droid_natural_8b01107c59_b_00070.png",\n              "droid_natural_8b01107c59_b_00167.png",\n              "droid_natural_8b01107c59_b_00171.png",\n              "droid_natural_8b01107c59_b_00175.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.28125,\n                20.703125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "81267a78cc998dcd581e71fd488663fc635d879268bcbb19ae9cfdf7a5c7af11",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.109375,\n                19.8125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "724aa539f2aace597cac1b08a569e40eed3aa838958c90db791afe0d0b56b09c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_6d74030dc0",\n      "source_site": "IPRL",\n      "scene_id": 4134662280,\n      "question": "In the last view of the wooden cabinet door just right of the range hood before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_6d74030dc0_a_00000.png",\n              "droid_natural_6d74030dc0_a_00034.png",\n              "droid_natural_6d74030dc0_a_00068.png",\n              "droid_natural_6d74030dc0_a_00102.png",\n              "droid_natural_6d74030dc0_a_00136.png",\n              "droid_natural_6d74030dc0_a_00171.png"\n            ],\n            "annotated": [\n              "droid_natural_6d74030dc0_a_00000.png",\n              "droid_natural_6d74030dc0_a_00034.png",\n              "droid_natural_6d74030dc0_a_00068.png",\n              "droid_natural_6d74030dc0_a_00149.png",\n              "droid_natural_6d74030dc0_a_00160.png",\n              "droid_natural_6d74030dc0_a_00171.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.359375,\n                21.984375\n              ],\n              "prediction": "A",\n              "prompt_sha256": "869f1202ba9d56642a97f305022e873b31efaacc7f8815ac7c5117cd36daaa1e",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            },\n            "reversed": {\n              "logits": [\n                22.34375,\n                21.6875\n              ],\n              "prediction": "A",\n              "prompt_sha256": "51f05daed082bb09ffa633d7a0ed536f4fd922abfaa963c4edac2f8fe0f11438",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_6d74030dc0_b_00000.png",\n              "droid_natural_6d74030dc0_b_00055.png",\n              "droid_natural_6d74030dc0_b_00111.png",\n              "droid_natural_6d74030dc0_b_00167.png",\n              "droid_natural_6d74030dc0_b_00223.png",\n              "droid_natural_6d74030dc0_b_00280.png"\n            ],\n            "annotated": [\n              "droid_natural_6d74030dc0_b_00000.png",\n              "droid_natural_6d74030dc0_b_00055.png",\n              "droid_natural_6d74030dc0_b_00111.png",\n              "droid_natural_6d74030dc0_b_00223.png",\n              "droid_natural_6d74030dc0_b_00229.png",\n              "droid_natural_6d74030dc0_b_00280.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                21.84375,\n                22.75\n              ],\n              "prediction": "B",\n              "prompt_sha256": "869f1202ba9d56642a97f305022e873b31efaacc7f8815ac7c5117cd36daaa1e",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            },\n            "reversed": {\n              "logits": [\n                22.140625,\n                22.390625\n              ],\n              "prediction": "B",\n              "prompt_sha256": "51f05daed082bb09ffa633d7a0ed536f4fd922abfaa963c4edac2f8fe0f11438",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_87abc314ec",\n      "source_site": "IRIS",\n      "scene_id": 8066802860,\n      "question": "In the last view of the wide white drawer below the countertop before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_87abc314ec_a_00000.png",\n              "droid_natural_87abc314ec_a_00063.png",\n              "droid_natural_87abc314ec_a_00127.png",\n              "droid_natural_87abc314ec_a_00192.png",\n              "droid_natural_87abc314ec_a_00255.png",\n              "droid_natural_87abc314ec_a_00320.png"\n            ],\n            "annotated": [\n              "droid_natural_87abc314ec_a_00000.png",\n              "droid_natural_87abc314ec_a_00063.png",\n              "droid_natural_87abc314ec_a_00127.png",\n              "droid_natural_87abc314ec_a_00279.png",\n              "droid_natural_87abc314ec_a_00299.png",\n              "droid_natural_87abc314ec_a_00320.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                23.21875,\n                21.734375\n              ],\n              "prediction": "A",\n              "prompt_sha256": "f98815a530bde678f71645134899dea2c9d1803713df1486537f8d60ecb75713",\n              "visual_tokens": 1960,\n              "lm_tokens": 2034\n            },\n            "reversed": {\n              "logits": [\n                23.0,\n                21.046875\n              ],\n              "prediction": "A",\n              "prompt_sha256": "f61e4cb6607abd024053275777dc80e1a3397db458e3e5bc9039dbb2aaf7c8ea",\n              "visual_tokens": 1960,\n              "lm_tokens": 2034\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_87abc314ec_b_00000.png",\n              "droid_natural_87abc314ec_b_00030.png",\n              "droid_natural_87abc314ec_b_00060.png",\n              "droid_natural_87abc314ec_b_00090.png",\n              "droid_natural_87abc314ec_b_00120.png",\n              "droid_natural_87abc314ec_b_00151.png"\n            ],\n            "annotated": [\n              "droid_natural_87abc314ec_b_00000.png",\n              "droid_natural_87abc314ec_b_00030.png",\n              "droid_natural_87abc314ec_b_00060.png",\n              "droid_natural_87abc314ec_b_00120.png",\n              "droid_natural_87abc314ec_b_00131.png",\n              "droid_natural_87abc314ec_b_00151.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                21.953125,\n                22.125\n              ],\n              "prediction": "B",\n              "prompt_sha256": "f98815a530bde678f71645134899dea2c9d1803713df1486537f8d60ecb75713",\n              "visual_tokens": 1960,\n              "lm_tokens": 2034\n            },\n            "reversed": {\n              "logits": [\n                21.890625,\n                21.984375\n              ],\n              "prediction": "B",\n              "prompt_sha256": "f61e4cb6607abd024053275777dc80e1a3397db458e3e5bc9039dbb2aaf7c8ea",\n              "visual_tokens": 1960,\n              "lm_tokens": 2034\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_29137e19c7",\n      "source_site": "IRIS",\n      "scene_id": 2158993557,\n      "question": "In the last view of the drawer under the white counter beside the black sink before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_29137e19c7_a_00000.png",\n              "droid_natural_29137e19c7_a_00027.png",\n              "droid_natural_29137e19c7_a_00054.png",\n              "droid_natural_29137e19c7_a_00080.png",\n              "droid_natural_29137e19c7_a_00108.png",\n              "droid_natural_29137e19c7_a_00135.png"\n            ],\n            "annotated": [\n              "droid_natural_29137e19c7_a_00000.png",\n              "droid_natural_29137e19c7_a_00027.png",\n              "droid_natural_29137e19c7_a_00054.png",\n              "droid_natural_29137e19c7_a_00131.png",\n              "droid_natural_29137e19c7_a_00133.png",\n              "droid_natural_29137e19c7_a_00135.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.453125,\n                21.390625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "2a7517e3c9fc9fd617822ee266b40cb3003e737b9c7e6dc20b25903e93ca88af",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            },\n            "reversed": {\n              "logits": [\n                22.34375,\n                20.84375\n              ],\n              "prediction": "A",\n              "prompt_sha256": "3f74e25ea3280e9f669e9585e7985be99966cccb0ba0f8a18b341bc06daaef0f",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_29137e19c7_b_00000.png",\n              "droid_natural_29137e19c7_b_00041.png",\n              "droid_natural_29137e19c7_b_00082.png",\n              "droid_natural_29137e19c7_b_00123.png",\n              "droid_natural_29137e19c7_b_00164.png",\n              "droid_natural_29137e19c7_b_00205.png"\n            ],\n            "annotated": [\n              "droid_natural_29137e19c7_b_00000.png",\n              "droid_natural_29137e19c7_b_00041.png",\n              "droid_natural_29137e19c7_b_00082.png",\n              "droid_natural_29137e19c7_b_00179.png",\n              "droid_natural_29137e19c7_b_00192.png",\n              "droid_natural_29137e19c7_b_00205.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.65625,\n                21.75\n              ],\n              "prediction": "A",\n              "prompt_sha256": "2a7517e3c9fc9fd617822ee266b40cb3003e737b9c7e6dc20b25903e93ca88af",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            },\n            "reversed": {\n              "logits": [\n                22.640625,\n                21.140625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "3f74e25ea3280e9f669e9585e7985be99966cccb0ba0f8a18b341bc06daaef0f",\n              "visual_tokens": 1960,\n              "lm_tokens": 2036\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_c00374385e",\n      "source_site": "PennPAL",\n      "scene_id": 5493550952,\n      "question": "In the last view of the wooden drawer just left of the white refrigerator before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_c00374385e_a_00000.png",\n              "droid_natural_c00374385e_a_00065.png",\n              "droid_natural_c00374385e_a_00131.png",\n              "droid_natural_c00374385e_a_00198.png",\n              "droid_natural_c00374385e_a_00263.png",\n              "droid_natural_c00374385e_a_00330.png"\n            ],\n            "annotated": [\n              "droid_natural_c00374385e_a_00000.png",\n              "droid_natural_c00374385e_a_00065.png",\n              "droid_natural_c00374385e_a_00131.png",\n              "droid_natural_c00374385e_a_00300.png",\n              "droid_natural_c00374385e_a_00315.png",\n              "droid_natural_c00374385e_a_00330.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.875,\n                21.203125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "bab9d2e209b203abcd11852847763fe4a701181ad94269c1c5cf96de19daa86c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.578125,\n                20.4375\n              ],\n              "prediction": "A",\n              "prompt_sha256": "c0afb3bec9c74ba24486933aa056f01683cbf514be67cb8f658bda48aabad3ed",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_c00374385e_b_00000.png",\n              "droid_natural_c00374385e_b_00044.png",\n              "droid_natural_c00374385e_b_00089.png",\n              "droid_natural_c00374385e_b_00133.png",\n              "droid_natural_c00374385e_b_00178.png",\n              "droid_natural_c00374385e_b_00223.png"\n            ],\n            "annotated": [\n              "droid_natural_c00374385e_b_00000.png",\n              "droid_natural_c00374385e_b_00044.png",\n              "droid_natural_c00374385e_b_00089.png",\n              "droid_natural_c00374385e_b_00194.png",\n              "droid_natural_c00374385e_b_00208.png",\n              "droid_natural_c00374385e_b_00223.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.640625,\n                20.328125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "bab9d2e209b203abcd11852847763fe4a701181ad94269c1c5cf96de19daa86c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.46875,\n                19.5625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "c0afb3bec9c74ba24486933aa056f01683cbf514be67cb8f658bda48aabad3ed",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_19bf4a67ee",\n      "source_site": "PennPAL",\n      "scene_id": 9160815698,\n      "question": "In the last view of the glass door of the blue countertop oven before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_19bf4a67ee_a_00000.png",\n              "droid_natural_19bf4a67ee_a_00036.png",\n              "droid_natural_19bf4a67ee_a_00073.png",\n              "droid_natural_19bf4a67ee_a_00110.png",\n              "droid_natural_19bf4a67ee_a_00147.png",\n              "droid_natural_19bf4a67ee_a_00184.png"\n            ],\n            "annotated": [\n              "droid_natural_19bf4a67ee_a_00000.png",\n              "droid_natural_19bf4a67ee_a_00036.png",\n              "droid_natural_19bf4a67ee_a_00073.png",\n              "droid_natural_19bf4a67ee_a_00162.png",\n              "droid_natural_19bf4a67ee_a_00173.png",\n              "droid_natural_19bf4a67ee_a_00184.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                21.65625,\n                22.0\n              ],\n              "prediction": "B",\n              "prompt_sha256": "8a4bc918d8c91248f181c1e83bb46c7fe3ca042e157891bdf51770148d302a5c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                21.9375,\n                21.71875\n              ],\n              "prediction": "A",\n              "prompt_sha256": "918d3b635c14d6ab09c22314e1279479ff0a0569f33cccbe7f3cd894266ee022",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_19bf4a67ee_b_00000.png",\n              "droid_natural_19bf4a67ee_b_00062.png",\n              "droid_natural_19bf4a67ee_b_00125.png",\n              "droid_natural_19bf4a67ee_b_00187.png",\n              "droid_natural_19bf4a67ee_b_00250.png",\n              "droid_natural_19bf4a67ee_b_00313.png"\n            ],\n            "annotated": [\n              "droid_natural_19bf4a67ee_b_00000.png",\n              "droid_natural_19bf4a67ee_b_00062.png",\n              "droid_natural_19bf4a67ee_b_00125.png",\n              "droid_natural_19bf4a67ee_b_00273.png",\n              "droid_natural_19bf4a67ee_b_00293.png",\n              "droid_natural_19bf4a67ee_b_00313.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.1875,\n                22.125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "8a4bc918d8c91248f181c1e83bb46c7fe3ca042e157891bdf51770148d302a5c",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.34375,\n                21.65625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "918d3b635c14d6ab09c22314e1279479ff0a0569f33cccbe7f3cd894266ee022",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        }\n      }\n    },\n    {\n      "pair_id": "droid_natural_10bd743a70",\n      "source_site": "TRI",\n      "scene_id": 8077138943,\n      "question": "In the last view of the glass-front cabinet door below the fruit basket before the scene changed, was it open or closed?",\n      "options": [\n        "Open",\n        "Closed"\n      ],\n      "recent_frames": [\n        "droid_shared_recent_00015.png",\n        "droid_shared_recent_00075.png",\n        "droid_shared_recent_00135.png",\n        "droid_shared_recent_00195.png"\n      ],\n      "branches": {\n        "a": {\n          "frames": {\n            "uniform": [\n              "droid_natural_10bd743a70_a_00000.png",\n              "droid_natural_10bd743a70_a_00077.png",\n              "droid_natural_10bd743a70_a_00154.png",\n              "droid_natural_10bd743a70_a_00232.png",\n              "droid_natural_10bd743a70_a_00309.png",\n              "droid_natural_10bd743a70_a_00387.png"\n            ],\n            "annotated": [\n              "droid_natural_10bd743a70_a_00000.png",\n              "droid_natural_10bd743a70_a_00077.png",\n              "droid_natural_10bd743a70_a_00154.png",\n              "droid_natural_10bd743a70_a_00358.png",\n              "droid_natural_10bd743a70_a_00372.png",\n              "droid_natural_10bd743a70_a_00387.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.09375,\n                22.296875\n              ],\n              "prediction": "B",\n              "prompt_sha256": "053a559af8ca2880af6bd638fc6eb052d30ea6171f1396eb47bf3db902bef3f8",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.453125,\n                21.703125\n              ],\n              "prediction": "A",\n              "prompt_sha256": "056c3d597b3f86e4243b9f1cf6814543a4e32e017aff3224c3c669f2a2705bec",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        },\n        "b": {\n          "frames": {\n            "uniform": [\n              "droid_natural_10bd743a70_b_00000.png",\n              "droid_natural_10bd743a70_b_00055.png",\n              "droid_natural_10bd743a70_b_00110.png",\n              "droid_natural_10bd743a70_b_00166.png",\n              "droid_natural_10bd743a70_b_00221.png",\n              "droid_natural_10bd743a70_b_00277.png"\n            ],\n            "annotated": [\n              "droid_natural_10bd743a70_b_00000.png",\n              "droid_natural_10bd743a70_b_00055.png",\n              "droid_natural_10bd743a70_b_00110.png",\n              "droid_natural_10bd743a70_b_00262.png",\n              "droid_natural_10bd743a70_b_00269.png",\n              "droid_natural_10bd743a70_b_00277.png"\n            ]\n          },\n          "archived_full": {\n            "canonical": {\n              "logits": [\n                22.171875,\n                22.15625\n              ],\n              "prediction": "A",\n              "prompt_sha256": "053a559af8ca2880af6bd638fc6eb052d30ea6171f1396eb47bf3db902bef3f8",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            },\n            "reversed": {\n              "logits": [\n                22.40625,\n                21.734375\n              ],\n              "prediction": "A",\n              "prompt_sha256": "056c3d597b3f86e4243b9f1cf6814543a4e32e017aff3224c3c669f2a2705bec",\n              "visual_tokens": 1960,\n              "lm_tokens": 2035\n            }\n          }\n        }\n      }\n    }\n  ],\n  "archived_full_zip_sha256": "e9eefa06ec8d1dcfed28668dd42fc0139cd7d85dc97aad791a23dbafdd9b26f0",\n  "archived_full_gpu": "NVIDIA GeForce RTX 4080"\n}\n', 'provenance/droid_terminal_runner.py': '"""Compare exact archived terminal frames under three context lengths."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom natural_history_runner import auto, base, choose, load_frames\n\nARTIFACT = "HERMES_droid_terminal_context_evidence.zip"\nFRAME_COUNTS = {"terminal_only": 3, "terminal_recent": 7, "full_annotated": 10}\nREPLAY_ATOL = 0.05\n\n\ndef compare(model, system, case, pixels, recent_features):\n    import torch\n\n    branches = {}\n    for side, branch in case["branches"].items():\n        histories = branch["frames"]\n        names = sorted(set(histories["uniform"]) | set(histories["annotated"]))\n        # Match the archived encoding order and reuse identical embeddings across arms.\n        bank = dict(zip(names, auto.encode_frames(model, np.stack([pixels[n] for n in names])), strict=True))\n        history = [bank[n] for n in histories["annotated"]]\n        terminal = history[-3:]\n        assert len(history) == 6 and len(recent_features) == 4\n        inputs = dict(terminal_only=terminal, terminal_recent=terminal+recent_features,\n                      full_annotated=history+recent_features)\n        token_count = terminal[0].shape[1]\n        views = {method: torch.cat(features, dim=1) for method, features in inputs.items()}\n        assert all(v.shape[1] == FRAME_COUNTS[m]*token_count for m, v in views.items())\n        rows, failures = {}, []\n        for order in base.ORDERS:\n            rows[order] = {m: choose(model, system, v, case["question"], case["options"], order)\n                           for m, v in views.items()}\n            replay, prior = rows[order]["full_annotated"], branch["archived_full"][order]\n            delta = max(abs(x-y) for x, y in zip(replay["logits"], prior["logits"], strict=True))\n            replay["replay_max_abs"] = delta\n            if delta > REPLAY_ATOL or any(replay[k] != prior[k] for k in\n                                        ("prediction", "lm_tokens", "visual_tokens", "prompt_sha256")):\n                failures.append(dict(order=order, replay=replay, archived=prior))\n        branches[side] = dict(answers=rows, terminal_frames=histories["annotated"][-3:], replay_failures=failures)\n    return dict(pair_id=case["pair_id"], scene_id=case["scene_id"], source_site=case["source_site"], branches=branches)\n\n\ndef summarize(blocks, targets):\n    methods = {m: dict(frames=frames, robust_pairs=[], correct_branch_orders=0, option_order_flips=0)\n               for m, frames in FRAME_COUNTS.items()}\n    scores = 0\n    for block in blocks:\n        pair = block["pair_id"]\n        for method, result in methods.items():\n            correct = 0\n            for side, branch in block["branches"].items():\n                labels = [branch["answers"][order][method]["prediction"] for order in base.ORDERS]\n                result["option_order_flips"] += len(set(labels)) != 1\n                correct += sum(label == targets[pair][side] for label in labels)\n                scores += len(labels)\n            result["correct_branch_orders"] += correct\n            if correct == 4:\n                result["robust_pairs"].append(pair)\n    added = {}\n    for before, after, stage in (("terminal_only", "terminal_recent", "added_recent"),\n                                 ("terminal_recent", "full_annotated", "added_earlier")):\n        left, right = (set(methods[m]["robust_pairs"]) for m in (before, after))\n        added[stage] = dict(gains=sorted(right-left), losses=sorted(left-right))\n    return dict(pairs=len(blocks), scores=scores, methods=methods, context_transitions=added)\n\n\ndef main():\n    import torch\n    from filelock import FileLock\n    from inference.llavaov_hermes import load_model\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "frames"):\n        parser.add_argument(f"--{name}", required=True, type=Path)\n    args = parser.parse_args()\n    work = args.work\n    with FileLock(str(work / "run.lock"), timeout=0):\n        export = work / "exports"\n        export.mkdir(exist_ok=True)\n        for name in ("summary.json", "pair_scores.json", "replay_gate.json", "run_config.json"):\n            (export / name).unlink(missing_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(work / "completion.json", dict(status="running"))\n        try:\n            spec = json.loads((work / "provenance/droid_terminal_cases.json").read_bytes())\n            assert len(spec["cases"]) == 7 and args.model.name == spec["model_revision"]\n            assert base.digest(args.frames) == spec["frame_zip_sha256"]\n            pixels = load_frames(args.frames, spec["frame_receipts"])\n            print("DROID_TERMINAL_INPUTS_VERIFIED", len(pixels), flush=True)\n            model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n            assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n            assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n            base.atomic(export / "run_config.json", dict(gpu=torch.cuda.get_device_name(), torch=torch.__version__,\n                        cuda=torch.version.cuda, model_revision=args.model.name, replay_atol=REPLAY_ATOL,\n                        archived_full_gpu=spec["archived_full_gpu"], archived_full_zip_sha256=spec["archived_full_zip_sha256"],\n                        frame_zip_sha256=spec["frame_zip_sha256"], frame_counts=FRAME_COUNTS))\n            blocks = []\n            with torch.inference_mode():\n                system = model.language_model.get_input_embeddings()(\n                    torch.as_tensor(model.init_prompt_ids, device=model.device)\n                )\n                names = spec["cases"][0]["recent_frames"]\n                assert len(names) == len(set(names)) == 4\n                shared = auto.encode_frames(model, np.stack([pixels[n] for n in names]))\n                for i, case in enumerate(spec["cases"], 1):\n                    assert case["recent_frames"] == names\n                    blocks.append(compare(model, system, case, pixels, shared))\n                    base.atomic(export / "pair_scores.json", blocks)\n                    print(f"DROID_TERMINAL_PAIR {i}/7 {case[\'pair_id\']}", flush=True)\n            failures = [dict(pair_id=block["pair_id"], branch=side, **failure) for block in blocks\n                        for side, branch in block["branches"].items() for failure in branch["replay_failures"]]\n            deltas = [row["full_annotated"]["replay_max_abs"] for block in blocks\n                      for branch in block["branches"].values() for row in branch["answers"].values()]\n            assert len(deltas) == 28\n            base.atomic(export / "replay_gate.json", dict(scores=len(deltas), max_abs=max(deltas), failures=failures))\n            if failures:\n                raise RuntimeError("Ten-frame baseline replay failed. See exports/replay_gate.json.")\n            targets = json.loads((work / "provenance/droid_natural_targets.json").read_bytes())\n            report = summarize(blocks, targets)\n            assert report["scores"] == 84\n            base.atomic(export / "summary.json", report)\n            base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n            print("DROID_TERMINAL_CONTEXT_COMPLETE", json.dumps(report), flush=True)\n        except BaseException as exc:\n            base.atomic(work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(work)\n\n\nif __name__ == "__main__":\n    main()\n'}
EXPECTED = {'pinned_source/inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'pinned_source/inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'pinned_source/inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'provenance/twinpast_base.py': '5f4e15c54168502763fb8ad2054e6c701e7440a09295b01240ef05e3865066f0', 'provenance/history_only_runner.py': '7614a0f8f72dd1e3e92e7b0a9a7e8ed232a1ab6778fac9a910fd48171c70f281', 'provenance/backbone_runner.py': '6a6f51a49f1ac1ce2f978f615b6a51a2b26404ca60147cf0b74f2c3b8f187ce4', 'provenance/automatic_runner.py': '6a2f33bb3907aea3a0b2b5aa9bdcd4bd6cb20c000f77247b3d11e1907378ecd3', 'provenance/temporal_runner.py': '5d2532993e68d2b378c395e75fb099ec7afaf32f54ca380a0c52541e84d01ee1', 'provenance/readout_runner.py': '7604f96c7ed0051adf5c6d066380c69019b550d8a06f4484365b944bd44b305b', 'provenance/droid_natural_manifest.json': '8422fcb786953ddb52ef7219b99cbe8bc21b78fdff4b6c00097fe6e2ebadb12e', 'provenance/droid_natural_targets.json': '778d545654a568e09836f58e89fba5190a35a4792fb856ba1a1354111c3f2f5e', 'provenance/natural_history_runner.py': 'fde16d2986ee757f7dfaa387a0aa7cbe73636006ce514a63a45f82e69a0a3779', 'provenance/droid_terminal_cases.json': 'fcbd31eeee7b619e8a8b3a76e6cb4257961cd06f3eb81b2fc72d5dcc569de2e3', 'provenance/droid_terminal_runner.py': '09761aa467c9e43bdaa506fb59de485224ac39d12ff87d2d30cc25342535b096'}

for relative, contents in SOURCES.items():
    raw = contents.encode('utf-8')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED[relative]
    if relative.endswith('.py'):
        ast.parse(contents, filename=relative)
    if not VALIDATE_ONLY:
        target = WORK / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.read_bytes() != raw:
            raise ValueError(f'Existing file differs at {target}. Preserve it and choose a fresh WORK directory.')
        if not target.is_file():
            target.write_bytes(raw)
print('EMBEDDED_SOURCE_AND_MANIFEST_PASS', BASE_REVISION)


In [ ]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""
if not VALIDATE_ONLY:
    for root in (REUSE, FALLBACK):
        candidate_python = root/'venv/bin/python'
        if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
            PYTHON = candidate_python
            print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)
            break
    PIP_ENV = package_environment(ENV)
    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                             'https://pypi.tuna.tsinghua.edu.cn/simple'], PIP_ENV)
            if attempt.returncode:
                logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                        'https://mirrors.aliyun.com/pypi/simple'], 'setup.log', PIP_ENV)
        UV = [sys.executable, '-m', 'uv']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)
        def install(packages, extra=()):
            install_packages(UV, PYTHON, packages, ENV, extra)
        wheel_name = 'torch-2.10.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl'
        wheel = next((p for root in (REUSE, FALLBACK, WORK)
                      if (p := root/'downloads'/wheel_name).is_file() and zipfile.is_zipfile(p)), None)
        if wheel is None:
            wheel = WORK/'downloads'/wheel_name
            wheel.parent.mkdir(parents=True, exist_ok=True)
            partial = wheel.with_suffix('.whl.part')
            if not zipfile.is_zipfile(partial):
                url = ('https://download-r2.pytorch.org/whl/cu128/'
                       'torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl')
                logged(['curl', '-fL', '-C', '-', '--connect-timeout', '20', '--speed-limit', '16384',
                        '--speed-time', '90', '--progress-bar', '-o', partial, url], 'torch_download.log', ENV)
            if not zipfile.is_zipfile(partial):
                raise ValueError(f'Incomplete PyTorch wheel: {partial}')
            partial.replace(wheel)
        install([str(wheel)])
        install(['numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
                 'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
                 'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
                 'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops', 'psutil', 'pandas'])
        # Download GitHub source with turbo, then install the local archive using
        # direct package networking for any isolated build dependencies.
        tf_archive = WORK/'downloads'/f'transformers-{TF_SHA}.zip'
        tf_archive.parent.mkdir(parents=True, exist_ok=True)
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path
dest = Path(sys.argv[2]); temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3: raise
        time.sleep(3 * (attempt + 1))
"""
            logged([sys.executable, '-c', fetch_tf,
                    f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}', tf_archive],
                   'source_download.log', ENV)
        install([str(tf_archive)], ['--no-deps'])
    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO/'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)


In [ ]:
# Standalone notebook cell. Uses WORK, ENV, MODEL_ID, MODEL_SHA and logged from setup.
import hashlib
import json
import re
import shutil
from pathlib import Path

SHARDS = (
    ("model-00001-of-00004.safetensors", 4911200360, "ad4cb27d2a3f6e987147619cf2508a5bbb1b6b4c2d5533501fcc4a287a90e5af"),
    ("model-00002-of-00004.safetensors", 4991497664, "cfaed9e4ba187283a936fd6df5d55148a6de5a8416f1279778b364232c05f39a"),
    ("model-00003-of-00004.safetensors", 4932752752, "86cb0bb7f7d57873b27f16fd66b54be71055e582a48ce26e30884bfb209dffd4"),
    ("model-00004-of-00004.safetensors", 1226266240, "a2d1e40e47e9b524efcbbcab68e97fe4e7aadfe9a51bc14b54cc2331c48ebff6"),
)
METADATA = (
    ("added_tokens.json", 122, "33e0fb93dadacb864bd2f2e8441e147daa2baceb67f94d3ef5283b495572cea0"),
    ("chat_template.json", 826, "2466d1704df30f0067f28d8e30e0190a1bf74e5b430942697af974d162a056bd"),
    ("config.json", 2528, "69277c2c9ba8a4f61d2e72f79fbe02e043b8c3af670858a606818527f971a0c2"),
    ("generation_config.json", 126, "89dc53229f50b59570b6852056dafeac8116c458f1a748bff491b6d4d24d3b51"),
    ("merges.txt", 1671853, "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"),
    ("model.safetensors.index.json", 77952, "e389b969d3b9b7120f136fbf6592cbbc1c07326157cae6f863fb64559261dae9"),
    ("preprocessor_config.json", 1732, "3644c108b9f0fa53e62ff422a9be6639642f0e64dab4a71f961c7911d4386384"),
    ("processor_config.json", 178, "04e9899e93f2a412c94e153cab4081457c9a44defb3b2c0b9df673d42c42cdd0"),
    ("special_tokens_map.json", 367, "f4f79e08d97f4d1c87f8d89264f525c8789da3b73b3bb55d1e12f692f41a7b1b"),
    ("tokenizer.json", 7028579, "3c0ce3213b50ff38d8aa1e91136a2d2cb142a3f569246170872e439cb2a29d15"),
    ("tokenizer_config.json", 1801, "b3151cdc153a44fa4e28abc8fc83567fc663aef7cd9dc041f5d66e617fe10b8e"),
    ("video_preprocessor_config.json", 621, "0ea9b672282b78353960b6069a521ed496a9f07c033e1e3362bff669234caa8d"),
    ("video_processor/preprocessor_config.json", 428, "1e71b2e75b90ddf696692529485b4a75fd54ecd4bbc03e2cc7be4af032875765"),
    ("vocab.json", 2776833, "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"),
)


def file_bytes(path):
    return path.stat().st_size if path.is_file() else 0


def verified(path, size, expected):
    if file_bytes(path) != size:
        return False
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest() == expected


def range_matches(headers, offset, end, total):
    replies = re.split(r"(?m)(?=^HTTP/)", headers.strip())
    final = replies[-1]
    return (re.match(r"HTTP/\S+\s+206\b", final) is not None
            and re.search(rf"(?im)^content-range:\s*bytes {offset}-{end}/{total}\s*$", final) is not None)


def fetch_pinned(name, size, expected, destination):
    if verified(destination, size, expected):
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".incomplete")
    direct_env = {k: v for k, v in ENV.items() if k.lower() not in {
        "http_proxy", "https_proxy", "all_proxy", "no_proxy"}}
    endpoints = (
        (f"https://modelscope.cn/models/{MODEL_ID}/resolve/master/{name}", direct_env),
        (f"https://hf-mirror.com/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
        (f"https://huggingface.co/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
    )
    offset = file_bytes(partial)
    if offset > size:
        raise RuntimeError(f"Oversized partial file: {partial} ({offset} bytes, expected {size})")
    chunk = partial.with_suffix(".range")
    headers = partial.with_suffix(".headers")
    while offset < size:
        end = min(offset + 64 * 1024**2, size) - 1
        length = end - offset + 1
        for url, env in endpoints:
            host = url.split("/")[2]
            print(f"MODEL_RANGE {name} bytes={offset}-{end}/{size} source={host}", flush=True)
            try:
                logged(["curl", "--fail", "--location", "--range", f"{offset}-{end}", "--connect-timeout", "15",
                        "--speed-limit", "16384", "--speed-time", "90", "--max-time", "600",
                        "--max-filesize", str(length), "--dump-header", headers, "--output", chunk,
                        "--progress-bar", url], "model_download.log", env)
            except RuntimeError:
                print(f"MODEL_ENDPOINT_RETRY {name} source={host}", flush=True)
                continue
            response = headers.read_text(encoding="iso-8859-1")
            # Some mirrors answer a request for an entire small file with a plain HTTP 200.
            whole_file = offset == 0 and length == size and verified(chunk, size, expected)
            if file_bytes(chunk) != length or not (whole_file or range_matches(response, offset, end, size)):
                print(f"MODEL_RANGE_REJECTED {name} source={host}", flush=True)
                continue
            with partial.open("ab") as output, chunk.open("rb") as incoming:
                if output.tell() != offset:
                    raise RuntimeError(f"Another downloader changed {partial}. Stop concurrent downloads.")
                try:
                    shutil.copyfileobj(incoming, output, length=1024**2)
                except BaseException:
                    output.flush()
                    output.truncate(offset)
                    raise
            offset += length
            print(f"MODEL_PROGRESS {name} {offset}/{size} bytes", flush=True)
            break
        else:
            raise RuntimeError(f"All endpoints failed for {name} at byte {offset}. Rerun to resume {partial}.")
    chunk.unlink(missing_ok=True)
    headers.unlink(missing_ok=True)
    if not verified(partial, size, expected):
        raise RuntimeError(f"SHA-256 mismatch in {partial}. The file was kept for inspection.")
    partial.replace(destination)


if not VALIDATE_ONLY:
    assert MODEL_ID == "llava-hf/llava-onevision-qwen2-7b-ov-hf"
    assert MODEL_SHA == "0d50680527681998e456c7b78950205bedd8a068"
    repo = Path(ENV["HF_HOME"]) / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    blobs = repo / "blobs"
    MODEL_PATH = repo / "snapshots" / MODEL_SHA
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    missing = sum(max(0, size - file_bytes(blobs / (sha + ".incomplete")))
                  for name, size, sha in SHARDS
                  if file_bytes(MODEL_PATH / name) != size and file_bytes(blobs / sha) != size)
    free = shutil.disk_usage(MODEL_PATH).free
    assert free >= missing + 2 * 1024**3, f"Need {(missing + 2 * 1024**3 - free) / 1024**3:.1f} GiB more free"
    print(f"MODEL_CACHE remaining_weight_download={missing / 1024**3:.2f} GiB", flush=True)
    for name, size, sha in SHARDS:
        path = MODEL_PATH / name
        if not verified(path, size, sha):
            blob = blobs / sha
            fetch_pinned(name, size, sha, blob)
            pending_link = path.with_name(path.name + ".verified-link")
            if pending_link.is_symlink():
                pending_link.unlink()
            pending_link.symlink_to(Path("../../blobs") / sha)
            pending_link.replace(path)
        print("MODEL_SHARD_VERIFIED", name, flush=True)
    for name, size, sha in METADATA:
        fetch_pinned(name, size, sha, MODEL_PATH / name)
    index = json.loads((MODEL_PATH / "model.safetensors.index.json").read_bytes())
    assert set(index["weight_map"].values()) == {name for name, _, _ in SHARDS}
    (WORK / "model_path.json").write_text(json.dumps(str(MODEL_PATH)), encoding="utf-8")
    print("MODEL_REVISION_PASS", MODEL_SHA, flush=True)


## 2. Replay the full history and check terminal-state recognition


In [ ]:
ARTIFACT = 'HERMES_droid_terminal_context_evidence.zip'

if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        source_revision=BASE_REVISION, model_revision=MODEL_SHA, transformers_revision=TF_SHA,
        embedded_sha256=EXPECTED, gpu=smi.stdout.strip()), indent=2), encoding='utf-8')
    env = {**ENV, 'PYTHONDONTWRITEBYTECODE': '1', 'HF_HUB_OFFLINE': '1'}
    logged([PYTHON, '-u', RUNNER, '--work', WORK, '--model', MODEL_PATH, '--frames', FRAME_PACKAGE],
           'droid_terminal_context.log', env, REPO)
else:
    print('VALIDATION_PASS: setup and embedded sources checked without GPU, network or file writes')


## 3. Download the evidence ZIP

After a runner error, run this cell manually to retrieve failure evidence.


In [ ]:

if not VALIDATE_ONLY:
    from IPython.display import FileLink, display
    artifact = WORK/ARTIFACT
    if artifact.is_file():
        link = Path.cwd()/ARTIFACT
        if artifact.resolve() != link.resolve():
            shutil.copyfile(artifact, link)
        print((WORK/'completion.json').read_text(encoding='utf-8'))
        display(FileLink(ARTIFACT))
    else:
        print('No runner ZIP yet. Inspect setup logs under', WORK/'logs')
